In [2]:
# 1. Install the Stable Stack
!pip install --no-cache-dir \
    transformers==4.46.0 \
    peft==0.13.0 \
    accelerate \
    trl \
    datasets

# 2. Remove potential conflicts
!pip uninstall -y bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 99.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 253.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 357.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import torch
from huggingface_hub import notebook_login

print("CUDA available:", torch.cuda.is_available())
print("GPUs found:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

# Login to access Google's Gemma model
notebook_login()

CUDA available: True
GPUs found: 2
Device Name: Tesla T4


In [4]:
import json
from datasets import load_dataset

# 1. Provide a small sample of our dataset (In reality, use 50+ examples)
raw_data =  [
  {
    "instruction": "Explain PNR Structure to a junior agent.",
    "input": "A trainee is struggling to remember the mandatory fields for a booking.",
    "output": "Think of a PNR as the 'Identity Document' of a trip. To make it official in the airline's eyes, you just need to remember the PRINT mnemonic: P for Phone, R for Received from, I for Itinerary, N for Name, and T for Ticketing. Without these five pillars, the central nervous system of the GDS won't allow the booking to live. It's like trying to mail a letter without an address; the system simply won't know where to send the data!"
  },
  {
    "instruction": "Explain how missing PNR elements break downstream reporting to a data engineer.",
    "input": "The mid-office system is showing 'Orphan Records' for certain bookings.",
    "output": "When a 'Received From' (R) element or a 'Phone' (P) field is missing or malformed in the PNR, the downstream parser often fails to attribute the transaction to a specific agent or office. This creates an 'orphan record' in your reporting layer. Essentially, the data pipeline loses its breadcrumbs. To fix this, we need to ensure the GDS-to-Hand-off API validates these five mandatory elements before the 'End of Transaction' command is allowed, otherwise, your General Ledger will never balance against the GDS activity."
  },
  {
    "instruction": "Explain why a PNR is important to a traveler.",
    "input": "A passenger asks why they need a 6-character code instead of just their name.",
    "output": "That 6-character code—your PNR—is like a digital suitcase that holds every single detail of your journey. While your name identifies you, the PNR identifies the specific 'handshake' between you, the airline, and the hotel. It ensures that when you show up at the gate, the airline's computer can instantly pull up your seat, your meal, and your connection. It's the golden key that keeps your travel plans synchronized across the globe!"
  },
  {
    "instruction": "Explain Amadeus Cryptic Commands to a junior agent.",
    "input": "The agent is used to a GUI and is intimidated by the 'black screen'.",
    "output": "Don't let the black screen scare you! Cryptic commands are just a shorthand language for talking directly to the airline's brain. Think of `AN` (Availability Neutral) as asking 'What's open?' and `SS` (Sell Segment) as saying 'I'll take that seat.' It’s much faster than clicking through menus—it’s like learning the secret codes to a video game that let you bypass the slow parts and get straight to the action."
  },
  {
    "instruction": "Explain how cryptic command logs assist in debugging to a data engineer.",
    "input": "A booking failed, but the UI logs are vague.",
    "output": "To find the root cause, we need to look at the 'Cryptic' level logs. When the UI sends a request, it translates your button click into a command like `NM1SMITH/JOHNMR`. If the GDS returns an error like 'FORMAT', it means the underlying EDIFACT string was malformed during the translation layer. By analyzing the raw command logs, we can see if the UI is failing to escape special characters before they hit the GDS terminal emulator."
  },
  {
    "instruction": "Explain the speed of cryptic commands to a traveler.",
    "input": "A traveler is watching an agent type rapidly on a text-only screen.",
    "output": "It looks like the Matrix, doesn't it? Your agent is using 'Cryptic Commands.' It's a specialized language that allows them to bypass the slow 'visual' internet and talk directly to the airline's inventory in milliseconds. It’s the difference between using a slow map app and having a direct radio line to the control tower—it ensures they get you the last available seat before anyone else can click 'buy' on a website."
  },
  {
    "instruction": "Explain Sabre TJR (Travel Journal Record) to a junior agent.",
    "input": "An agent's commands aren't working in a new office location.",
    "output": "The TJR is basically the 'DNA' of your specific office. It tells Sabre what you are allowed to do—like whether you can issue e-tickets or if you have permission to view certain fares. If your commands are failing, your 'DNA' might be missing a specific instruction. We’ll need to check the TJR settings to ensure your Pseudo City Code is properly configured for the tasks you're trying to perform."
  },
  {
    "instruction": "Explain how TJR settings affect API responses to a data engineer.",
    "input": "The API is returning 'NOT AUTHORIZED' for fare searches in one region only.",
    "output": "This isn't a code bug; it's a TJR configuration issue. The Travel Journal Record acts as a security gatekeeper at the PCC (Pseudo City Code) level. If the TJR for that specific office doesn't have the 'Negotiated Fares' flag enabled, the Sabre API will filter out those results before they even reach our middleware. We need to audit the TJR settings for that PCC to ensure the data flow isn't being throttled at the source."
  },
  {
    "instruction": "Explain TJR-driven personalization to a traveler.",
    "input": "A traveler wonders why their local agency can see better deals than a big website.",
    "output": "Every travel agency has a unique 'profile' in the system, known as a TJR. Think of it as a specialized lens. Your local agent's lens is tuned to find specific negotiated deals and local perks that a generic global website might overlook. It allows them to tailor the system's massive database specifically to the needs of travelers in your area."
  },
  {
    "instruction": "Explain GDS Queue Management to a junior agent.",
    "input": "The agent doesn't understand why they need to check 'Queue 1' every morning.",
    "output": "Imagine the GDS has a series of 'In-boxes' for you, called Queues. Queue 1 is usually for schedule changes. If an airline moves your flight by two hours, they drop a message in that box. If you don't check your boxes, your travelers show up to the airport for a flight that left an hour ago! It’s the system's way of tapping you on the shoulder to say, 'Hey, something changed—take a look!'"
  },
  {
    "instruction": "Explain Queue processing logic to a data engineer.",
    "input": "We need to automate the handling of schedule changes.",
    "output": "Automating queue processing is like building a mail-sorting robot. You'll need to poll the GDS for messages in specific Queue categories (like Q1 for schedule changes or Q97 for ticketing). The challenge is that these messages are often unstructured text. You'll need a regex parser to extract the PNR locator and the 'Old vs New' flight times so you can update the 'Itinerary' table in our database without manual intervention."
  },
  {
    "instruction": "Explain why 'being on a queue' is good for a traveler.",
    "input": "A traveler is on a waitlist for a sold-out flight.",
    "output": "Right now, your booking is sitting on a high-priority 'Queue' with the airline. Think of it as a digital line where the computer is constantly checking for a cancellation. The moment a seat opens up, the system 'notifies' our queue, and we can grab that seat for you instantly. It’s like having a dedicated scout watching the gates 24/7 just for you."
  },
  {
    "instruction": "Explain Passive Segments (AK/BK) to a junior agent.",
    "input": "An agent is booking a boutique hotel that isn't in the GDS.",
    "output": "When you book something outside the GDS, like a local tour or a boutique inn, the 'central brain' doesn't know about it. We use Passive Segments—codes like `AK`—to manually tell the PNR, 'Hey, I booked this elsewhere, but keep a record of it here.' It's like writing a note in the margin of a book so that when you print the itinerary, everything is in one place for the traveler."
  },
  {
    "instruction": "Explain how passive segments cause 'Double Counting' in data reporting to a data engineer.",
    "input": "The sales report shows twice the actual revenue for certain trips.",
    "output": "This is a classic 'Passive Segment' trap. The agent booked a hotel via an aggregator API (which hit our DB) and then added a passive segment in the GDS for the same hotel to put it on the itinerary. Your reporting logic is likely counting both the API record and the GDS passive segment as separate sales. You need to filter out segments with status codes like 'AK', 'BK', or 'MK' from your revenue calculations, as they are just informational 'ghost' records."
  },
  {
    "instruction": "Explain why an itinerary shows 'Confirmed - Outside System' to a traveler.",
    "input": "A traveler is confused by a note on their PDF itinerary.",
    "output": "That just means your agent went the extra mile to book a unique experience that isn't part of the standard airline network. We've manually added it to your master plan so you have one single timeline for your whole trip. It’s essentially us 'pinning' an external note into your digital travel diary so nothing gets lost!"
  },
  {
    "instruction": "Explain PCC (Pseudo City Codes) to a junior agent.",
    "input": "An agent is moving from the London branch to the New York branch.",
    "output": "Your PCC is like your office's 'Home Address' in the GDS world. In London, you lived at PCC '12AB'; in New York, you'll be at '34CD'. Every time you log in, the system checks your PCC to see which tickets you're allowed to print and which local deals you can access. It's the key that unlocks your specific branch's toolbox."
  },
  {
    "instruction": "Explain PCC-based data partitioning to a data engineer.",
    "input": "We need to ensure agents in France can't see the bookings of agents in Germany.",
    "output": "We handle this through PCC (Pseudo City Code) partitioning. In our database, every PNR record must be tagged with the PCC it originated from. Our API layer then applies a 'Where' clause to every query based on the user's authenticated PCC. This mimics the GDS's own security structure, ensuring that data 'silos' are maintained for privacy and compliance across different regions."
  },
  {
    "instruction": "Explain why an agency location matters to a traveler.",
    "input": "A traveler asks why their ticket was issued in a different city.",
    "output": "Sometimes we issue tickets through a specific 'Pseudo City' or branch because that location has a special agreement with the airline. Think of it like a store having a 'member-only' branch in another city—by routing the paperwork through there, we can often secure better rates or more flexible rules for your trip!"
  },
  {
    "instruction": "Explain NDC (New Distribution Capability) to a junior agent.",
    "input": "The agent is confused why some flights look like 'Web Deals' in the GDS.",
    "output": "NDC is like upgrading from a 1980s radio to a modern smartphone. Traditional GDS is text-heavy and limited. NDC allows airlines to send us 'Rich Content'—like photos of the seats, Wi-Fi packages, and personalized deals—straight from their own system. It’s the airline finally speaking the same modern language as the rest of the internet."
  },
  {
    "instruction": "Explain the architectural shift from EDIFACT to JSON for NDC to a data engineer.",
    "input": "The legacy parser is failing on new airline offers.",
    "output": "You're seeing the shift from EDIFACT to NDC's JSON-based XML standards. Legacy GDS uses a rigid, character-limited protocol (EDIFACT), but NDC uses a much more flexible, hierarchical JSON/XML structure. You'll need to build a new adapter layer that can handle these 'Offer' and 'Order' objects, which contain much more nested data (like ancillary attributes) than the old flat-file segments we're used to."
  },
  {
    "instruction": "Explain why 'Special Offers' are appearing now to a traveler.",
    "input": "A traveler sees a personalized 'bundle' offer during booking.",
    "output": "We're using a new technology called NDC that lets the airline talk directly to us in real-time. Instead of just a seat, they can now offer you a 'Travel Bundle'—like your seat, a checked bag, and lounge access—tailored specifically to your frequent flyer status. It's like the airline recognizing you at the door and offering you your 'usual' at a discount!"
  },
  {
    "instruction": "Explain Interline Agreements to a junior agent.",
    "input": "The agent is trying to book a trip using two different airlines on one ticket.",
    "output": "Interlining is like a 'Trust Treaty' between airlines. If Airline A and Airline B have an agreement, they've agreed to accept each other's passengers and baggage. It’s what allows you to check your bags in London and not see them again until you reach a tiny island in the Pacific, even if you switched planes twice. Without that agreement, you’d have to 're-check' at every stop!"
  },
  {
    "instruction": "Explain Interline validation logic to a data engineer.",
    "input": "The ticketing API is throwing 'NO INTERLINE' errors.",
    "output": "The system is performing a 'Ticketing Authority' check. Before we can issue a single E-ticket for multiple carriers, the GDS checks a 'Bilateral Table'. If Airline A doesn't have an interline agreement with Airline B, the system cannot generate a single payment settlement (BSP/ARC) for both. You need to catch this error in the 'Pricing' phase so the agent knows they'll have to issue two separate tickets instead of one."
  },
  {
    "instruction": "Explain why they can't 'connect' certain flights to a traveler.",
    "input": "A traveler wants to combine a super-cheap budget airline with a major carrier.",
    "output": "In the airline world, some companies don't have a 'handshake' agreement (we call this an Interline Agreement). It’s like two different train companies that use different sized tracks—they just can't share the same journey. If we booked them together, your bags wouldn't transfer, and you wouldn't be protected if one flight was late. We recommend sticking to 'partner' airlines to keep your trip seamless!"
  },
  {
    "instruction": "Explain Electronic Miscellaneous Documents (EMD) to a junior agent.",
    "input": "The agent needs to charge for a heavy bag after the ticket is issued.",
    "output": "An EMD is like a 'Digital Voucher' for anything that isn't the seat itself. Since the ticket is already 'closed' for the flight, we issue an EMD-A (Associated) for the bag. It’s a separate receipt that stays attached to your ticket so the gate agent knows you’ve already paid. Think of it as a 'digital sticky note' that proves you've paid for your extras."
  },
  {
    "instruction": "Explain EMD data reconciliation to a data engineer.",
    "input": "The finance team can't find where 'Seat Upgrade' revenue is coming from.",
    "output": "Ancillary revenue is often hidden because it's issued on EMDs rather than primary tickets. When you're pulling sales data, you need to query both 'Ticket' documents and 'EMD' documents. An EMD has its own 13-digit number, just like a ticket, but it's linked via an 'Association' field. If your ETL process only looks at the primary ticket price, you're missing all the high-margin ancillary revenue from bags, seats, and Wi-Fi."
  },
  {
    "instruction": "Explain why a separate receipt is sent for bags to a traveler.",
    "input": "A traveler receives two different confirmation numbers.",
    "output": "One number is for your seat (your ticket), and the other is for your 'extras' like your checked bag (that's the EMD). We keep them separate so that if you change your flight, your bag payment stays safe in its own digital 'wallet' until you’re ready to use it on your new flight. It’s just our way of keeping your payments organized!"
  },
  {
    "instruction": "Explain Global Indicators (AT/PA/EH) to a junior agent.",
    "input": "The fare for a flight to Sydney is different depending on if they fly via LA or Singapore.",
    "output": "Global Indicators tell the system which 'way' you're going around the world. 'AT' means you're crossing the Atlantic, 'PA' is the Pacific, and 'EH' is the Eastern Hemisphere (via Europe/Asia). The price changes because the 'rules of the road' and taxes are different for each ocean you cross. It’s like choosing a different highway—each one has its own toll price!"
  },
  {
    "instruction": "Explain Global Indicator logic in fare calculation to a data engineer.",
    "input": "The pricing engine is returning three different fares for the same city pair.",
    "output": "The pricing engine is evaluating 'Routing Surcharges' based on Global Indicators. For a JFK-SIN route, the 'AT' routing (via London) might have a higher fuel surcharge than the 'PA' routing (via Tokyo). When storing these fares, you must include the 'GI' code as a key attribute, or your UI will display three identical-looking flights with wildly different prices and no explanation why."
  },
  {
    "instruction": "Explain why the 'Long Way' might be cheaper to a traveler.",
    "input": "A traveler is confused why flying further costs less.",
    "output": "In the travel world, the 'Global Indicator' or the path you take determines the price. Sometimes taking the 'Atlantic' path (via Europe) is on a more popular route with more competition, making it cheaper than the shorter 'Pacific' path. It’s like taking a slightly longer bridge because the toll is significantly cheaper—it’s all about which digital 'highway' we use!"
  },
  {
    "instruction": "Explain Fare Basis Code Anatomy to a junior agent.",
    "input": "The agent sees a code like 'WH7LNR' and doesn't know what it means.",
    "output": "That code is a 'Story' about the ticket. The 'W' is the class (Economy), the 'H' means it's high season, and '7' might mean you had to book it 7 days in advance. 'NR' usually stands for Non-Refundable. Learning to read these is like learning to read a secret menu—it tells you exactly what the traveler can and cannot do without having to open the full 'Rules' book."
  },
  {
    "instruction": "Explain how to parse Fare Basis Codes for business intelligence to a data engineer.",
    "input": "The marketing team wants to know if we are selling more 'Non-Refundable' tickets.",
    "output": "You can extract a lot of 'hidden' data by parsing the Fare Basis string. Usually, the last two characters indicate the most important restrictions. We can build a regex to look for 'NR' (Non-Ref) or 'REF' (Refundable) within that string. This allows us to categorize our 'Inventory Mix' without needing a heavy API call to the 'Rules' database for every single booking."
  },
  {
    "instruction": "Explain what the 'Funny Code' on their receipt means to a traveler.",
    "input": "A traveler points to 'KLE00NR' on their invoice.",
    "output": "Think of that as the 'Serial Number' for your specific price. It’s a shorthand code that tells the airline exactly which 'sale' you bought, whether it’s a weekend-only deal or a special advance purchase. It’s basically the airline's way of making sure you get the exact service and flexibility you paid for!"
  },
  {
    "instruction": "Explain Tax Designators (US/ZP/YQ) to a junior agent.",
    "input": "The 'Taxes' section of the quote is huge, and the agent wants to explain it.",
    "output": "Taxes aren't just government fees. 'US' and 'ZP' are usually flight and airport taxes, but 'YQ' or 'YR' are actually 'Carrier Surcharges'—often for fuel. If a traveler asks why taxes are so high, it's often because the 'YQ' fee is bundled in there. It’s like a 'service fee' at a restaurant that gets added to the state tax on the bill."
  },
  {
    "instruction": "Explain how Tax Designators affect refund calculations to a data engineer.",
    "input": "The automated refund tool is returning the wrong amount for 'non-refundable' tickets.",
    "output": "When a ticket is non-refundable, the 'Base Fare' is lost, but many 'Government Taxes' (like US or ZP) are often still refundable by law. However, 'Carrier Surcharges' (YQ/YR) are usually non-refundable if the base fare is. Your refund logic needs to look at the 'Tax Designator' codes—refund the government ones, but keep the carrier ones. If you don't, you'll either over-refund the customer or face legal complaints."
  },
  {
    "instruction": "Explain why 'Taxes' are so high to a traveler.",
    "input": "A traveler is upset that a '$100 fare' has '$300 in taxes'.",
    "output": "It’s frustrating to see that jump! What the system calls 'Taxes' is actually a mix of airport fees, government security charges, and 'Carrier Surcharges' (like fuel costs). The airline keeps the base fare low to stay competitive, but the 'Taxes' section covers the actual cost of the fuel and the airport's permission to land. It's like a 'delivery fee' and 'service charge' on a pizza—it’s the cost of getting the product to you!"
  },
  {
    "instruction": "Explain MCT (Minimum Connect Time) to a junior agent.",
    "input": "The agent is trying to book a 45-minute connection in Heathrow.",
    "output": "Wait! Heathrow is huge. Every airport has an 'MCT'—the absolute shortest time the system thinks a human (and their bag) can make a connection. If the MCT is 60 minutes and you book 45, the system will let you do it, but it won't be a 'Legal' connection. If they miss the flight, the airline won't help them because you 'broke the rules' of the airport's timing."
  },
  {
    "instruction": "Fix My Itinerary: Flag an MCT violation.",
    "input": "Itinerary: JFK -> LHR (Arrival 09:00 AM). LHR -> CDG (Departure 09:35 AM).",
    "output": "I have to flag this! At Heathrow (LHR), the Minimum Connect Time for an international-to-international transfer is typically 60 to 90 minutes. A 35-minute gap is an 'MCT Violation'. Even if the first flight is on time, the traveler's bags will almost certainly not make the second plane, and the airline isn't obligated to re-book them for free. We need to move that second flight to at least 10:30 AM to be safe."
  },
  {
    "instruction": "Explain MCT data validation to a data engineer.",
    "input": "The search results are showing flights that travelers keep missing.",
    "output": "Our search algorithm isn't strictly enforcing MCT. We need to ingest the 'MCT Table' from IATA. This table specifies times based on the 'Arrival Terminal' vs. 'Departure Terminal'. A 'Terminal 2 to Terminal 5' connection needs more time than a 'Terminal 2 to Terminal 2' connection. We should filter out any itinerary where `DepartureTime - ArrivalTime < MCT` before the user even sees it."
  },
  {
    "instruction": "Explain Direct Connect vs. Aggregator to a junior agent.",
    "input": "The agent sees different prices for the same flight on two different screens.",
    "output": "One screen is 'Direct Connect'—talking straight to the airline like a private phone line. The other is an 'Aggregator'—like a middleman who gathers info from 50 airlines at once. The Direct Connect is often cheaper and faster, but the Aggregator lets you see everything in one place. It’s like buying a cake directly from the bakery vs. buying it from a grocery store that sells 10 different brands."
  },
  {
    "instruction": "Explain the latency difference between Direct Connect and Aggregators to a data engineer.",
    "input": "Users are complaining that 'Cheap' flights disappear when they click 'Book'.",
    "output": "This is a 'Caching' issue common with Aggregators. Aggregators often cache search results for 10-30 minutes to save on API costs. By the time the user clicks, the airline's 'Direct' inventory has changed. Direct Connects have lower latency and 'Live' inventory but are harder to maintain. We should implement a 'Pre-Book' check for Aggregator results to verify the price hasn't changed before we show the final payment screen."
  },
  {
    "instruction": "Explain why prices fluctuate between different sites to a traveler.",
    "input": "A traveler found a flight $20 cheaper on a random blog's link.",
    "output": "It’s all about how those sites 'connect' to the airline. Some sites use a direct 'hotline' to the airline, while others use a 'middleman' that might be looking at a saved version of the price from an hour ago. We always use the most direct connection possible to ensure that when you click 'buy', that seat is actually yours at the price you saw!"
  },
  {
    "instruction": "Explain OTA (Open Travel Alliance) XML Standards to a junior agent.",
    "input": "The agent is frustrated that every hotel booking screen looks different.",
    "output": "Actually, behind the scenes, they are mostly speaking the same language called 'OTA XML'. It’s a set of rules that ensures when we send a request for a 'King Room', every hotel understands exactly what we mean. It’s like a 'Universal Translator' for travel—it’s what allows our one system to talk to thousands of different hotels smoothly."
  },
  {
    "instruction": "Explain XML Schema validation for OTA standards to a data engineer.",
    "input": "We are getting 'Schema Validation' errors from a new hotel partner.",
    "output": "The partner is likely using a different version of the OTA (Open Travel Alliance) standard. Even though it's 'Standard', the tags for things like `Position` (lat/long) or `Amenity` codes can vary between the 2003 and 2014 versions. We need to check their WSDL file and ensure our XML generator is wrapping the `HotelResRequest` in the correct namespace version, or their server will keep rejecting our 'handshake'."
  },
  {
    "instruction": "Explain how 'Standards' help the traveler to a traveler.",
    "input": "A traveler asks how their booking gets to a small hotel in rural Italy.",
    "output": "It’s amazing, isn't it? We use a global 'standard' language that acts like a digital courier. Whether it's a giant Hilton or a tiny villa, our system sends the exact same 'coded message' that their computer is trained to understand. It ensures your reservation is waiting for you, no matter how far off the beaten path you go!"
  },
  {
    "instruction": "Explain Mid-Office/Back-Office Sync to a junior agent.",
    "input": "The agent forgot to 'Interface' a PNR and the accounting team is calling.",
    "output": "Think of the GDS as the 'Front Desk' and the Back-Office as the 'Accountant's Ledger'. The 'Sync' is the person who runs the paperwork from the desk to the ledger. If you don't 'Interface' the PNR, the accountant has no idea you sold a ticket! It’s like selling a coffee but forgetting to ring it up in the cash register—the money is there, but the books won't match."
  },
  {
    "instruction": "Explain PNR Hand-off (Hand-off files) to a data engineer.",
    "input": "The accounting system is missing the 'Tax Breakdown' for all tickets.",
    "output": "We need to check the 'I-File' or 'Interface Record' generated during the sync. The GDS creates a flat file (like an AIR or IUR file) containing all the PNR data. If the 'Tax Field' (usually the TX segment) is being truncated or skipped in your parser, the Back-Office (Agresso/TRAMS) will only see the total price, not the breakdown. We need to update the parser to recognize the specific sub-tags for 'YQ' and 'XT' taxes."
  },
  {
    "instruction": "Explain why an invoice takes a few minutes to arrive to a traveler.",
    "input": "A traveler is worried because they don't have a receipt immediately after booking.",
    "output": "Don't worry! Your booking is finished, but our system is just doing its 'final check.' It’s currently sending the details from the airline's system over to our accounting office to generate your official tax invoice. It’s like the waiter going back to the station to print your bill—it'll be in your inbox in just a moment!"
  },
  {
    "instruction": "Explain CRS (Computer Reservations System) to a junior agent.",
    "input": "The agent is confused by the difference between 'The GDS' and 'The Airline's System'.",
    "output": "The CRS is the airline's 'Private Warehouse'. They keep all their seats there. The GDS is like the 'Global Supermarket' that displays seats from 500 different warehouses. When you book a seat, the Supermarket (GDS) sends a message to the Warehouse (CRS) to take that seat off the shelf. They are two different systems 'talking' to each other to keep the inventory correct."
  },
  {
    "instruction": "Explain CRS-GDS synchronization latency to a data engineer.",
    "input": "We are getting 'Inventory Out of Sync' errors.",
    "output": "This usually happens when the CRS (the source of truth) hasn't updated the GDS (the distribution layer) yet. If an airline cancels a flight in their CRS, it can take anywhere from a few seconds to several minutes for that message to 'propagate' to the GDS. When building our 'Booking' flow, we should always perform a 'Sell' command to force a real-time check against the CRS before we take the user's payment."
  },
  {
    "instruction": "Explain the 'Heart' of the airline to a traveler.",
    "input": "A traveler asks where their data actually 'lives'.",
    "output": "Your data lives in the airline's 'Computer Reservations System'—think of it as the airline's heart and memory. It's the master vault where they keep every seat, every meal, and every passenger's name. Our systems just have a 'special key' to look inside that vault and make sure everything is perfect for your arrival!"
  },
  {
    "instruction": "Explain Ghost Segments to a junior agent.",
    "input": "The agent sees a flight in the PNR that has a status of 'HX' or 'NO'.",
    "output": "Those are 'Ghost Segments.' They are flights that the airline has cancelled or rejected, but they're still 'haunting' your PNR. If you don't delete them, the system gets confused and might prevent you from issuing a ticket. It’s like having an old, cancelled appointment still sitting on your calendar—it's not real anymore, so you need to clear it out to make room for the real ones!"
  },
  {
    "instruction": "Explain how ghost segments break GDS productivity tracking to a data engineer.",
    "input": "The agency is getting billed for 'Inactive Segments' by the GDS.",
    "output": "GDS providers charge 'churn' or 'inactive' fees for PNRs that contain 'HX' or 'UC' (Ghost) segments that aren't purged within 24 hours. We need to build a 'PNR Hygiene' script that runs every night, scans our active PNRs for these status codes, and automatically sends the 'cancel' command to the GDS. This will stop the 'data noise' and, more importantly, stop the unnecessary fees."
  },
  {
    "instruction": "Explain why a 'Flight Cancelled' message appears on a confirmed trip to a traveler.",
    "input": "A traveler sees an old, cancelled flight on their app itinerary.",
    "output": "I see that! It looks like a 'Ghost' of an old flight that was changed earlier. It's not your actual flight—your real one is confirmed right below it. Sometimes the system keeps a 'memory' of the change just in case we need to reference it. I'll clear that out for you now so your app looks nice and clean!"
  },
  {
    "instruction": "Explain Dual Phase Ticketing (TTL) to a junior agent.",
    "input": "An agent is confused why a booking disappeared after 24 hours.",
    "output": "Travel has two phases: 'Booking' and 'Ticketing'. Think of 'Booking' as putting an item in your online shopping cart—the store holds it for you, but only for a little while (that's the TTL, or Ticketing Time Limit). If you don't 'Ticket' it (the second phase, where you actually pay), the store puts the item back on the shelf. You always have to watch that clock!"
  },
  {
    "instruction": "Explain TTL (Ticketing Time Limit) automation to a data engineer.",
    "input": "We need to prevent 'Auto-Cancellations' for our corporate clients.",
    "output": "We need to monitor the `TKTL` (Ticketing Time Limit) field in the PNR. This is a timestamp provided by the airline. Our system should send an automated 'Reminder' to the agent 4 hours before the TTL expires. If we want to be more advanced, we can build a 'Queue' that flags any PNR where the TTL is within 2 hours but the 'Ticket Status' is still 'Open', ensuring we don't lose the inventory."
  },
  {
    "instruction": "Explain why a price 'expired' to a traveler.",
    "input": "A traveler is upset that they waited a day and the flight is gone.",
    "output": "I'm so sorry! Airlines have a 'Time Limit' on how long they'll hold a seat before they have to offer it to someone else. It’s like a 'reservation' at a busy restaurant—if you aren't there (or don't pay) by a certain time, they have to give the table to the next person in line. I'll try to find that same price for you again right now!"
  },
  {
    "instruction": "Explain VCN (Virtual Card Numbers) to a junior agent.",
    "input": "The agent is booking a hotel and the system generated a one-time credit card.",
    "output": "That's a VCN! Instead of using the agency's real credit card or your own, the system creates a 'Digital One-Time Card' just for this booking. It's much safer—it only works for that specific hotel, for that specific amount, and for those specific dates. It’s like a 'disposable' credit card that disappears once the job is done."
  },
  {
    "instruction": "Explain VCN reconciliation challenges to a data engineer.",
    "input": "The finance team can't match credit card statements to individual hotel bookings.",
    "output": "Since every VCN is a unique 16-digit number, you can't rely on the card number for reconciliation. You need to map the 'Virtual Card ID' to the 'PNR Locator' at the moment of generation. Ensure your database captures the VCN's 'Transaction ID'. When the bank feed comes in, you match the bank's transaction ID to our ID, which then links back to the PNR. Without this 'middle link', your reconciliation will be a manual nightmare."
  },
  {
    "instruction": "Explain how their payment is 'Secured' to a traveler.",
    "input": "A traveler is worried about their card info being sent to a small hotel.",
    "output": "We use 'Virtual Card' technology to protect you. Instead of sending your actual details, we generate a one-time-use secure code that only that hotel can use, and only for your specific stay. It’s like a high-tech shield that ensures your private financial information never even touches the hotel's computer!"
  },
  {
    "instruction": "Explain API Mapping (Room Types) to a junior agent.",
    "input": "The agent sees 'Deluxe Room' on one site and 'King Bed Superior' on another for the same hotel.",
    "output": "This is a 'Mapping' puzzle. Every hotel uses different names for their rooms. Our system tries to 'map' them all into categories we can understand, like 'Standard' or 'Premium'. Sometimes a 'Superior' at one hotel is actually a 'Deluxe' at another. It’s like one person saying 'Soda' and another saying 'Pop'—they mean the same thing, we just have to translate it!"
  },
  {
    "instruction": "Explain the difficulty of 'Room Type' normalization to a data engineer.",
    "input": "Our hotel search is showing 15 different types of 'Double Rooms' for one hotel.",
    "output": "We have a normalization problem. Supplier A calls it 'DBL RM', Supplier B calls it 'Double Standard', and the GDS calls it 'A2D'. We need to build a 'Mapping Table' that uses string matching and 'Attribute' extraction (like looking for 'King', 'Sea View', or 'Non-Smoking'). If we don't normalize these into a single 'Master Room Type', the user gets 'Option Fatigue' and our 'Lowest Price' logic fails because it can't compare apples to apples."
  },
  {
    "instruction": "Explain why the room name looks different on their confirmation to a traveler.",
    "input": "A traveler booked a 'Premium' room but the hotel email says 'Executive'.",
    "output": "Don't worry, you've got the right room! Hotels often have internal names (like 'Executive') that we translate into simpler terms (like 'Premium') on our website so it's easier to compare. It’s the same great room, just with a slightly fancier title in the hotel's own system!"
  },
  {
    "instruction": "Explain DIP (Direct Inward Processing) to a junior agent.",
    "input": "The hotel just told the agent they are sold out, but the GDS still shows one room.",
    "output": "That’s a 'DIP' delay. Most hotels send their updates to the GDS in real-time (that's DIP), but sometimes there's a tiny lag. The hotel's own desk always knows the truth first. It's like a store's website saying there's one shirt left in stock, but the person at the counter just sold it to the guy in front of you."
  },
  {
    "instruction": "Explain DIP 'Push' vs 'Pull' inventory to a data engineer.",
    "input": "We are getting too many 'Failed to Book' errors after a successful search.",
    "output": "We are relying on 'Push' inventory (DIP) that is stale. The GDS is waiting for the hotel to 'Push' updates. For high-demand periods, we should switch to a 'Pull' model—where we perform a `HotelDescriptiveInfo` or `HotelRatePlan` request at the moment of selection. This 'pulls' the live data directly from the hotel's PMS, reducing our 'Book-to-Fail' ratio significantly."
  },
  {
    "instruction": "Explain why a room 'disappeared' during booking to a traveler.",
    "input": "A traveler was halfway through booking when the system said 'No Longer Available'.",
    "output": "I am so sorry! Because thousands of people are looking at the same rooms at once, sometimes a 'live' update comes in right as you're clicking. It’s like a 'flash sale' where the very last item gets grabbed while it's in your hand. Let me find you an even better alternative right now!"
  },
  {
    "instruction": "Explain Booking Class Hierarchy (Y/J/F) to a junior agent.",
    "input": "The agent doesn't understand why 'Y' and 'Q' are both Economy but have different prices.",
    "output": "Think of Economy as a giant 'bucket'. Inside that bucket, there are smaller cups labeled Y, M, Q, and K. 'Y' is the most expensive cup—it’s flexible and always available. 'K' is the cheapest cup, but it's small and fills up fast. They all get the same seat on the plane, but you're paying for the 'rules' and the 'availability' of that specific cup."
  },
  {
    "instruction": "Explain how to use 'Class Mapping' for revenue analysis to a data engineer.",
    "input": "The sales team wants to know our 'Average Yield' per cabin.",
    "output": "You need to map the single-character 'Booking Class' to its 'Cabin' (First, Business, Premium Econ, Econ). We should build a 'Lookup Table' for each airline, as 'P' class might be 'First' on one airline but 'Premium Economy' on another. By tagging every ticket with its 'Cabin' based on the class, you can calculate the 'Yield' (Revenue per mile) and see which routes are our real money-makers."
  },
  {
    "instruction": "Explain why their friend paid less for the same seat to a traveler.",
    "input": "Two travelers are sitting together but one paid $200 less.",
    "output": "It’s all about when the 'buckets' were filled! Airlines sell seats in groups (we call them 'classes'). Your friend likely grabbed a seat from the 'Early Bird' bucket. Once that was full, the airline opened the next bucket, which costs a bit more. You both get the same great service, they just caught the 'opening bell' of the sale!"
  },
  {
    "instruction": "Explain GDS Scripts (Smart Flows) to a junior agent.",
    "input": "The agent is tired of typing the same 10 lines of text for every corporate booking.",
    "output": "You need a 'Smart Flow' script! It’s like a 'Macro' or a 'Shortcut' button. Instead of typing the company address, the loyalty number, and the billing code one by one, you just click the 'Corp Booking' button, and the script 'types' it all for you in a split second. It’s like having a personal assistant who handles all the boring paperwork so you can focus on the traveler."
  },
  {
    "instruction": "Explain Script version control to a data engineer.",
    "input": "Half the agency is using an old script that is putting the wrong codes in PNRs.",
    "output": "We need to move away from locally stored scripts. We should host our 'Smart Flows' on a central server and have the GDS terminal (Amadeus/Sabre) 'fetch' the latest version on startup. This ensures that when the billing codes change, we update it once in the cloud, and every agent in the world is instantly using the correct logic. No more data cleaning on the backend!"
  },
  {
    "instruction": "Explain why 'Special Requests' are so accurate now to a traveler.",
    "input": "A traveler is impressed that their agent never forgets their 'Nut Allergy' note.",
    "output": "We use specialized 'Smart Scripts' that act like a digital checklist. Every time we book for you, the system automatically checks your profile and 'stamps' those important notes directly onto your ticket. It ensures that the airline knows exactly what you need, every single time, without us ever missing a detail!"
  },
  {
    "instruction": "Explain BSP (Billing and Settlement Plan) to a junior agent.",
    "input": "The agent is confused about who they are actually paying when they issue a ticket.",
    "output": "BSP is like the 'Central Bank' of the airline world. Instead of you paying 100 different airlines individually, you send all your ticket money to the BSP once a week. They then 'sort' the money and send it to the right airlines. It’s a giant 'Clearing House' that keeps the whole industry's finances organized and safe."
  },
  {
    "instruction": "Explain BSP 'ADM' (Agency Debit Memo) data loops to a data engineer.",
    "input": "We are getting 'Fines' (ADMs) from airlines for pricing errors.",
    "output": "An ADM happens when the BSP detects a discrepancy between the 'Filed Fare' and the 'Ticketed Price'. To reduce this, we need to build a 'Post-Ticketing Audit' loop. Our system should compare the `FareConstruction` string in the GDS to the price we actually charged. If there's a mismatch, we should flag it *before* the BSP settlement cycle so the agent can void and correct it, saving us thousands in fines."
  },
  {
    "instruction": "Explain the 'Security' of their payment to a traveler.",
    "input": "A traveler asks if the airline already has their money.",
    "output": "We use a global system called BSP that acts as a secure 'escrow' for your flight. Your payment is held in a protected central account and only released to the airline once your ticket is fully verified and issued. It's the industry's way of making sure your money is handled with the highest level of international security!"
  },
  {
    "instruction": "Explain ARC (Airlines Reporting Corporation) to a junior agent.",
    "input": "The agent is moving from a UK agency to a US-based agency.",
    "output": "In the UK, you used BSP; in the US, you’ll use ARC. They do the exact same thing—they are the 'Financial Middleman' between you and the airlines. The main difference is just the specific reports and the schedule of when you 'settle' the bill. It’s like switching from one bank to another—the rules are slightly different, but the money still gets where it needs to go."
  },
  {
    "instruction": "Explain ARC vs BSP data format differences to a data engineer.",
    "input": "Our global sales dashboard is failing to import US sales data.",
    "output": "ARC (US) and BSP (Global) use different reporting formats. ARC uses a format called 'IAR' (Interactive Agent Reporting), while BSP uses 'RET' files. The field lengths for taxes and commission are different. You'll need to create two different mapping 'profiles' in our ETL pipeline to normalize these into a single 'Global Revenue' table. Don't forget that ARC data is usually in USD, while BSP can be in any local currency!"
  },
  {
    "instruction": "Explain why US bookings look different to a traveler.",
    "input": "A traveler sees an 'ARC' logo on their US-issued itinerary.",
    "output": "That 'ARC' logo is a badge of trust! It means your ticket was issued through the official 'Airlines Reporting Corporation,' which is the gold standard for travel finance in the United States. It’s your guarantee that your ticket is legitimate and backed by the major US airlines."
  },
  {
    "instruction": "Explain Look-to-Book Ratios to a junior agent.",
    "input": "The manager is asking agents to 'stop browsing' so much in the GDS.",
    "output": "Every time you 'search' in the GDS, it costs a tiny bit of money. The 'Look-to-Book' ratio is the balance between how many times we 'look' at flights vs. how many we actually 'book'. If we look 1,000 times for every 1 booking, the GDS might charge us a fee! It’s like a store that asks you to buy something if you’re going to spend all day trying on clothes."
  },
  {
    "instruction": "Explain the technical impact of high Look-to-Book on GDS infrastructure to a data engineer.",
    "input": "The GDS provider is threatening to throttle our API access.",
    "output": "Our 'Look-to-Book' ratio has spiked to 5000:1, which is putting too much 'Search Load' on their legacy mainframe. We need to implement a 'Caching Layer' for our most common searches (like LHR-JFK). By serving the 'Look' from our own cache and only hitting the GDS for the 'Book' or 'Final Price', we can drop our ratio and avoid the GDS surcharges and throttling."
  },
  {
    "instruction": "Explain why some sites are 'Faster' than others to a traveler.",
    "input": "A traveler wonders why a site they used felt 'laggy' when searching.",
    "output": "Some sites are 'heavy searchers'—they ask the airline for a new price every single second, which can slow things down. We try to be 'Smart Searchers' by keeping a quick-reference guide of the best prices, only reaching out to the airline's main computer when we're sure we've found your perfect match. It keeps our site fast and the prices accurate!"
  },
  {
    "instruction": "Explain HSP (Hotel Service Provider) Switches to a junior agent.",
    "input": "The agent is curious how the GDS 'knows' a hotel's availability.",
    "output": "Think of the 'Switch' (like Pegasus or DHISCO) as a giant 'Phone Exchange'. The GDS doesn't talk to every hotel; it talks to the 'Switch', and the Switch connects it to the hotel's own computer. It’s the bridge that allows a massive GDS to talk to a tiny hotel's front desk system instantly."
  },
  {
    "instruction": "Explain the technical 'Point of Failure' in the HSP Switch to a data engineer.",
    "input": "We are getting 'Connection Timed Out' only for certain hotel chains.",
    "output": "The bottleneck isn't our API or the GDS; it's likely the 'HSP Switch' (Pegasus/DHISCO). If the 'Switch' has a latency spike or if the hotel's PMS (Property Management System) isn't responding to the Switch's query within the SOAP timeout limit, the GDS returns a generic error. We should implement a 'Provider Health' dashboard to track which switches are failing so we can automatically reroute traffic to a different aggregator when a switch goes down."
  },
  {
    "instruction": "Explain the 'Digital Bridge' to a traveler.",
    "input": "A traveler is amazed that they can book a hotel in Tokyo from an office in London.",
    "output": "It’s all thanks to a 'Digital Bridge' that spans the world! When we click 'Book', our computer sends a message across a specialized network that connects directly to that hotel's front desk. It's like having a dedicated fiber-optic line straight to their lobby, ensuring your room is waiting for you when you land!"
  },
  {
    "instruction": "Explain TMC Portals (HR Integration) to a junior agent.",
    "input": "The agent is booking for a new traveler whose details are already in the system.",
    "output": "That's the power of the 'HR Feed'! Our system is linked directly to the client's company HR portal. When they hire someone new, the 'TMC Portal' automatically creates a travel profile with their name, passport, and even their favorite seat. It means you don't have to type a single thing—it’s all there waiting for you!"
  },
  {
    "instruction": "Explain HR-to-Profile data synchronization to a data engineer.",
    "input": "Employee names are showing up with typos in the travel profiles.",
    "output": "We need to audit the 'SFTP Sync' from the client's Workday or SAP system. The 'Source of Truth' is their HR database. If they have 'John (Project Mgr) Smith' as the name, our parser is grabbing the parentheses and putting them into the PNR, which the airline will reject. We need to add a 'Data Scrubbing' layer to our profile sync that strips out non-alpha characters and maps the fields correctly to the GDS `NM1` format."
  },
  {
    "instruction": "Explain why their 'Preferences' are already saved to a traveler.",
    "input": "A corporate traveler is surprised the agent knows they prefer an aisle seat.",
    "output": "We work closely with your company to make your travel as easy as possible! Our system is securely linked so that your preferences—like that aisle seat or your frequent flyer number—are automatically ready for us the moment we start your booking. It’s our way of making sure every trip feels customized just for you!"
  },
  {
    "instruction": "Explain Single Sign-On (SSO) for OBTs to a junior agent.",
    "input": "The client is complaining that they have to log in too many times.",
    "output": "We need to set up 'SSO' (Single Sign-On). It's like a 'Master Key'. Once they log into their company's main page, that 'Master Key' automatically unlocks our Travel Booking Tool (OBT) too. They don't need a second password—the system already knows who they are and that they're allowed to be there."
  },
  {
    "instruction": "Explain SAML/OIDC token hand-off for OBTs to a data engineer.",
    "input": "The SSO integration is failing with 'Invalid Token'.",
    "output": "The SAML assertion from the client's Identity Provider (IdP) is likely missing the 'User ID' attribute that our OBT requires. Check the XML claim—we need the `NameID` to match the `ExternalID` in our traveler profile database. If the 'Time Skew' between their server and our OBT server is more than 5 minutes, the token will also be rejected. Ensure both systems are synced to a reliable NTP server."
  },
  {
    "instruction": "Explain why they don't need a new password to a traveler.",
    "input": "A traveler is happy they didn't have to create a new account for the travel site.",
    "output": "Exactly! We’ve set up a 'Secure Handshake' with your company's network. Since you're already logged in at work, our system trusts that it's you and lets you right in. It’s one less password to remember and a much more secure way to keep your travel plans private!"
  },
  {
    "instruction": "Explain the Schengen 90/180 rule to a junior agent.",
    "input": "A traveler is planning multiple trips to Europe this year and is confused about how long they can stay.",
    "output": "Think of the Schengen zone as a single 'Digital House' with one front door. The 90/180 rule is like a rolling credit limit: your traveler has 90 'day-credits' to spend, but the bank only looks back at the last 180 days from today. If they stay too long, the 'house' locks them out. Always use a Schengen calculator before booking that third or fourth leg to ensure they haven't spent their credits too fast!"
  },
  {
    "instruction": "Explain the data challenges of tracking Schengen stays to a data engineer.",
    "input": "We need to build an automated alert for overstay risks in our booking app.",
    "output": "Tracking this is a multi-variant temporal problem. You can't just look at 'Total Days' in a year; you need a sliding window algorithm that evaluates `SUM(stay_duration)` where `exit_date - 180 <= entry_date`. The complexity arises with multi-entry visas and retroactive 'stay' calculations. Your logic must account for 'partial days' (entry and exit days count as full days) and cross-reference PNR data with actual 'Flown' status to avoid alerts based on cancelled trips."
  },
  {
    "instruction": "Reassure a traveler about their multi-trip Europe plan regarding Schengen rules.",
    "input": "The traveler is worried they'll be arrested for visiting France, then Italy, then Germany over six months.",
    "output": "Don't worry! Think of your Schengen visa as a 'Season Pass' to a giant park. As long as you don't spend more than 90 days total inside the park during any 180-day window, you can go in and out as much as you like. It's designed to let you explore freely; we'll just help you keep a quick tally of your days to make sure your 'pass' stays valid and active!"
  },
  {
    "instruction": "Explain IATA Resolution 830d to a junior agent.",
    "input": "The agent doesn't understand why they have to enter a mobile number for every passenger in a specific format.",
    "output": "IATA 830d is the 'Emergency Broadcast' rule. Airlines are now legally required to have a direct line to the passenger for IRROPS (irregular operations). We use the `SSR CTCE` (email) and `SSR CTCM` (mobile) codes. It’s like providing the airline with a walkie-talkie for each guest—if the flight is cancelled at 3 AM, the airline can reach them directly without waiting for our office to open."
  },
  {
    "instruction": "Explain how malformed contact data in PNRs affects automated messaging to a data engineer.",
    "input": "The airline says their automated flight alerts aren't reaching our customers.",
    "output": "The issue is likely our SSR formatting. Airlines parse `CTCM` fields using strict regex. If we send `CTCM+1-555-123`, but the airline's parser expects `CTCM1555123` (no dashes), the message fails silently. We need a 'Contact Normalization' layer in our API that strips special characters and ensures the country code is present, or the airline's 'notification engine' will treat the data as noise."
  },
  {
    "instruction": "Explain why personal contact info is needed to a traveler.",
    "input": "A traveler is hesitant to give their private cell phone number for a booking.",
    "output": "We completely respect your privacy! We only share this with the airline as a 'Safety Line.' If there’s a sudden gate change or a delay while you're on your way to the airport, the airline can text you immediately so you aren't left standing in the wrong terminal. It’s the fastest way to keep you moving smoothly if the schedule shifts!"
  },
  {
    "instruction": "Explain GDPR for Travel to a junior agent.",
    "input": "The agent is keeping a spreadsheet of traveler passport numbers on their desktop for 'speed'.",
    "output": "Stop right there! That spreadsheet is a 'Data Breach' waiting to happen. Under GDPR, passport numbers are 'High-Stakes Data.' We must keep them locked in the secure GDS profile, not on your desktop. Think of it like keeping a client's house key: you leave it in the office safe, you don't carry it around in your pocket where it could fall out!"
  },
  {
    "instruction": "Explain the 'Right to be Forgotten' logic to a data engineer.",
    "input": "A former client is requesting all their travel data be deleted.",
    "output": "This is a complex 'Purge' operation. You need to delete their PII (Personally Identifiable Information) from our Profiles DB, but we are legally required by tax law to keep the 'Financial Record' of the ticket for 7 years. You’ll need a 'Data Masking' strategy: redact the Name and Passport fields, but leave the Transaction ID and Amount. Essentially, keep the 'ghost' of the sale for the accountants, but erase the 'human' for the privacy team."
  },
  {
    "instruction": "Explain data protection to a traveler.",
    "input": "A traveler asks how we keep their passport and credit card info safe.",
    "output": "Your safety is our top priority, both in the air and on our servers! We use bank-level encryption, which means your sensitive info is 'scrambled' into a code that only authorized systems can read. We also follow strict international rules like GDPR to ensure that your data is only used to book your trip and is never shared with anyone who doesn't absolutely need it to get you to your destination!"
  },
  {
    "instruction": "Explain EC 261/2004 to a junior agent.",
    "input": "A traveler is calling from the airport because their flight to Paris was cancelled.",
    "output": "EC 261 is the 'Traveler's Bill of Rights' in Europe. If the airline is at fault (like a mechanical issue) and the delay is over 3 hours, the airline owes the passenger cash compensation, meals, and a hotel. It’s the airline’s 'Penalty Box' for being late. Make sure the traveler gets a written statement from the gate—it’s their ticket to getting paid later!"
  },
  {
    "instruction": "Explain automated EC 261 claim detection to a data engineer.",
    "input": "We want to build a feature that tells users they are eligible for money after a delay.",
    "output": "You need to join two data streams: the user's `FlownSegment` data and a 'Global Flight Status' feed (like OAG or FlightStats). The logic is: If `ArrivalActual - ArrivalScheduled > 180 mins` AND `Origin_or_Carrier_is_EU`, flag for EC 261. The 'Edge Case' is the 'Extraordinary Circumstances' (weather/strikes) which aren't in the API. You'll need to tag these as 'Potential Claims' rather than 'Guaranteed' to manage expectations."
  },
  {
    "instruction": "Explain flight delay rights to a traveler.",
    "input": "A traveler is frustrated because they’ve been waiting at the gate for 4 hours.",
    "output": "I know it's exhausting, but there is a silver lining! Because you're flying an EU airline, you are protected by a rule called EC 261. Since you've been delayed over 3 hours, the airline is actually required to provide you with meal vouchers now and potentially a cash payment of up to €600 later. Think of it as the airline 'buying back' the time you lost!"
  },
  {
    "instruction": "Explain DOT Part 382 to a junior agent.",
    "input": "An agent is booking a flight for a traveler who uses a motorized wheelchair.",
    "output": "DOT Part 382 is the US law that ensures 'The Sky is Open to Everyone.' You must never charge extra for mobility assistance, and you have to ensure the airline knows the 'Battery Type' for that wheelchair. It’s our job to make sure the airline is prepared to treat that traveler with dignity and provide the necessary equipment. It’s about making travel 'Barrier-Free'."
  },
  {
    "instruction": "Explain SSR code mapping for disability assistance to a data engineer.",
    "input": "Our booking tool needs to capture wheelchair requirements correctly.",
    "output": "You need to map user selections to specific IATA SSR (Special Service Request) codes. Don't just send 'Wheelchair.' You must distinguish between `WCHR` (can walk but needs help with distance), `WCHS` (can't do stairs), and `WCHC` (completely immobile). For motorized chairs, you also need to capture `WCBD` (dry cell) or `WCBW` (wet cell) batteries for hazmat compliance. If you send the wrong code, the passenger might get to the plane and realize the equipment they need isn't there."
  },
  {
    "instruction": "Reassure a traveler with a disability about their upcoming flight.",
    "input": "A traveler is nervous about how their wheelchair will be handled on a US flight.",
    "output": "You are in very good hands! Under US law (DOT Part 382), airlines are experts at assisting travelers with mobility needs. We will add a special 'digital tag' to your booking that tells the ground crew exactly what kind of assistance you need at every step. We'll also pre-register your wheelchair so they have the right equipment ready to load it safely. Our goal is to make your transition from the curb to the cabin completely seamless!"
  },
  {
    "instruction": "Explain APIS (Advanced Passenger Information System) to a junior agent.",
    "input": "An agent forgot to enter the passport info for a US-bound flight and the PNR won't ticket.",
    "output": "APIS is the 'Border Patrol's Guest List.' Governments like the US and UK require the airline to send them your passport details before the plane even takes off. If the list isn't complete, the 'Bouncer' (Customs) won't let the plane fly. That’s why the system is blocking the ticket—it's trying to prevent the airline from getting a massive fine for an incomplete guest list!"
  },
  {
    "instruction": "Explain APIS data transmission protocols to a data engineer.",
    "input": "We are getting 'Security Rejected' errors from the airline API for international bookings.",
    "output": "Check your `DOCS` and `DOCO` segment logic. APIS requires very specific data fields: Full Name (matching passport), Passport Number, Nationality, Expiry Date, and often the 'Address of First Night' in the destination country. If a single field—like the 'Country of Issuance'—is missing or uses a 3-letter code instead of a 2-letter code (or vice-versa, depending on the airline's API version), the entire security 'handshake' will fail."
  },
  {
    "instruction": "Explain security checks to a traveler.",
    "input": "A traveler is annoyed that they have to provide their passport details weeks before they fly.",
    "output": "I totally understand—it feels like a lot of paperwork! This is part of the 'Advanced Passenger Information' system. It’s essentially a way for the destination country to do a quick 'pre-check' so that when you land, your journey through customs is much faster. It’s like pre-registering for a conference so you can skip the long lines at the front desk and get straight to your vacation!"
  },
  {
    "instruction": "Explain IATA Resolution 753 to a junior agent.",
    "input": "A traveler is asking how they can track their bag on their phone.",
    "output": "IATA 753 is the 'No Bag Left Behind' rule. Airlines are now required to 'Check-In' your bag at four points: when you drop it, when it goes on the plane, when it switches planes, and when it arrives. It’s like a 'Digital Breadcrumb' trail for your suitcase. If an airline follows 753, they can tell you exactly which 'breadcrumb' your bag is at in real-time!"
  },
  {
    "instruction": "Explain baggage tracking API integration to a data engineer.",
    "input": "We want to show a 'Baggage Progress Bar' in our mobile app.",
    "output": "You’ll need to hook into the airline's 'Baggage Event' stream (often via SITA or a direct airline API). You are looking for 'Scan Events' at the `BPM` (Bag Proxy Message) level. Your UI needs to map these raw scans to user-friendly states: 'Bag Received', 'Loaded on Flight XXX', 'In Transit at Hub', and 'Arrived at Carousel'. Be prepared for 'Null' events where a handler misses a scan—your logic should 'assume' the last known good position."
  },
  {
    "instruction": "Explain bag tracking to a traveler.",
    "input": "A traveler is worried about their luggage getting lost on a connection.",
    "output": "You can breathe easy! Most major airlines now use a high-tech tracking system called Resolution 753. Think of it as a 'GPS' for your suitcase. It gets scanned every time it moves—from the check-in desk to the plane's cargo hold. If you download the airline's app, you can often see a little green checkmark the moment your bag is safely on board with you. It’s like 'Find My Phone,' but for your luggage!"
  },
  {
    "instruction": "Explain Cabotage Laws to a junior agent.",
    "input": "An agent is trying to book a British Airways flight for a traveler going from New York to Los Angeles.",
    "output": "You can't do that! That's a 'Cabotage' violation. Think of it like a 'Local Jobs' law: foreign airlines aren't allowed to pick up and drop off passengers between two cities in the same foreign country. BA can take someone from NY to London, but they can't 'compete' with US airlines for a domestic NY to LA trip. The system will block the 'Sell' command to prevent the airline from losing its flying license!"
  },
  {
    "instruction": "Explain Cabotage validation logic to a data engineer.",
    "input": "Our search engine is showing domestic routes operated by foreign carriers.",
    "output": "You need to implement a 'Cabotage Filter.' If `Origin_Country == Destination_Country` AND `Carrier_Home_Country != Origin_Country`, the flight is likely illegal for local sale. The exception is 'Beyond Rights' or 'Stopover Rights' for international tickets. For a pure domestic search, you must filter out any airline whose 'Nationality' doesn't match the country of travel, otherwise, you'll be sending users to book flights that will be rejected at the payment stage."
  },
  {
    "instruction": "Explain why a certain airline isn't available for a domestic flight to a traveler.",
    "input": "A traveler wants to fly a famous international airline for a short hop within the US.",
    "output": "I wish we could—those airlines are great! However, there's a global rule called 'Cabotage' that's a bit like a 'Home Team' rule. It means that for flights entirely within the US, only US-based airlines can fly those routes. It’s designed to support local aviation and jobs. But don't worry, we'll find you a great US carrier with a similar level of comfort for that short trip!"
  },
  {
    "instruction": "Explain Sixth Freedom Rights to a junior agent.",
    "input": "The agent is confused why Emirates is so popular for flights between London and Sydney.",
    "output": "That’s the 'Sixth Freedom' in action! It's when an airline carries passengers between two countries (UK and Australia) by stopping at its home hub (Dubai). It’s like a 'Hub-and-Spoke' superpower. They use their home base as a giant 'Switchboard' to connect the whole world. It’s perfectly legal and often gives travelers more options and better prices!"
  },
  {
    "instruction": "Explain routing logic for 6th Freedom carriers to a data engineer.",
    "input": "Our 'Cheapest Route' algorithm is ignoring hub-based international connections.",
    "output": "Your algorithm might be biased toward 'Direct' flights or 'Bilateral' partners. You need to enable 'Hub Search' for 6th Freedom carriers (like Qatar, Emirates, or Turkish). This means searching for `A -> HUB` and `HUB -> B` and combining them under a single 'Marketing Carrier' fare. These carriers often have highly optimized 'Minimum Connect Times' at their hubs specifically designed to win this 'Sixth Freedom' traffic."
  },
  {
    "instruction": "Explain why flying 'via a hub' is a good deal to a traveler.",
    "input": "A traveler asks why they have to stop in Doha to get to Africa.",
    "output": "It’s actually a very clever 'shortcut' used by some of the world's best airlines! By stopping at their home 'Super-Hub' in Doha, the airline can gather travelers from all over the world and connect them to their next destination all at once. It usually means you get a much more modern plane, amazing lounge options during your break, and often a better price than a direct flight!"
  },
  {
    "instruction": "Explain Health Passports (VeriFLY/IATA) to a junior agent.",
    "input": "The agent is manually checking COVID/Health docs and it’s taking forever.",
    "output": "Use the 'Health Passport' integrations! Think of them as a 'Pre-Cleared' badge. The traveler uploads their docs to an app like VeriFLY, the app verifies them, and then it sends a 'Green Light' signal to the airline's check-in system. It takes the legal liability off your shoulders and puts the traveler on the 'Fast Track' at the airport."
  },
  {
    "instruction": "Explain Health Credential API hand-off to a data engineer.",
    "input": "We need to show 'Document Status' in our check-in flow.",
    "output": "You need to query the 'Digital Health Folder' associated with the PNR. The API (like IATA Travel Pass) returns a boolean `ready_to_fly` and a list of 'Missing Requirements'. You shouldn't store the actual health documents (due to HIPAA/GDPR complexity); just store the 'Validation Token' and the status. This ensures that when the user hits 'Check-In', your system can pass that token to the DCS (Departure Control System) to bypass manual document checks."
  },
  {
    "instruction": "Explain the benefit of using a health app to a traveler.",
    "input": "A traveler is frustrated about having to download another app for their vaccine records.",
    "output": "I know, another app! But think of this one as your 'VIP Fast-Pass.' By uploading your documents now, a team of experts verifies them before you even leave for the airport. When you get to the desk, instead of a long paper check, the agent just sees a 'Verified' green light. It’s the difference between waiting in a long line and walking straight to the front!"
  },
  {
    "instruction": "Explain Duty of Care to a junior agent.",
    "input": "An agent doesn't understand why they need to run a 'Location Report' during a strike in Italy.",
    "output": "Duty of Care is the 'Guardian Rule' for business travel. Companies are legally responsible for the safety of their employees. If there’s a strike or an emergency, the company needs to know *exactly* who is in that city right now so they can get them out. Our system is the 'Tracker' that lets them fulfill that promise of safety. Every PNR is a 'Life Line' in these moments."
  },
  {
    "instruction": "Explain real-time traveler tracking to a data engineer.",
    "input": "We need to build a 'Crisis Dashboard' that shows traveler locations on a map.",
    "output": "You need to aggregate 'Current Location' data by merging PNR itineraries with 'Flight Status' updates. If a traveler is on flight BA123, their location isn't just their 'Destination'; until they land, their location is 'In-Flight'. Your query needs to look for travelers whose `ArrivalDate > CurrentTime` AND `DepartureDate < CurrentTime`. You also need to account for 'Overnight Stays' in hotels, pulling from the `HHL` segments in the GDS to ensure the map reflects where they are sleeping, not just where they landed."
  },
  {
    "instruction": "Explain 'Duty of Care' to a business traveler.",
    "input": "A traveler asks why their company's travel app is always asking for their location.",
    "output": "It’s all about your safety! Your company has a 'Duty of Care' promise, which means they want to be your 'Guardian Angel' while you're on the road. If there’s ever an emergency or a major travel disruption, they use that info to reach out to you immediately and offer help, or even arrange a quick way home. It’s their way of making sure you’re never truly on your own, no matter where you are!"
  },
  {
    "instruction": "Explain PCI-DSS Compliance to a junior agent.",
    "input": "The agent is writing down a client's credit card CVV code in a notebook.",
    "output": "Stop! Tear that page out and shred it immediately. That’s a huge 'PCI-DSS' violation. We are never allowed to write down or store the CVV code—not even on paper. Think of it like a 'Secret Handshake' that must be forgotten the moment it’s used. If we keep it, and our office is compromised, we could be responsible for the client's stolen money and face massive fines!"
  },
  {
    "instruction": "Explain credit card tokenization to a data engineer.",
    "input": "We need to store payment methods for 'One-Click' booking.",
    "output": "You must use 'Tokenization.' Never store the raw 16-digit PAN or the CVV in our database. When the user enters their card, send it directly to a PCI-certified vault (like Stripe or Adyen). They will return a 'Token'—a useless string like `tok_123`. You store that token. When it’s time to book, you send the token to the GDS or Airline API. This way, if our database is hacked, the hackers just get a list of useless 'tokens' and no actual credit card numbers."
  },
  {
    "instruction": "Explain payment security to a traveler.",
    "input": "A traveler is worried about their credit card being 'on file'.",
    "output": "We take your financial security incredibly seriously! We don't actually store your 'real' credit card number on our systems. Instead, we use 'Tokenization,' which replaces your card number with a unique, secure code that only works for your bookings with us. It’s like using a 'proxy' or a 'stand-in'—even if someone saw the code, they couldn't use it anywhere else. Your real card details stay locked in a high-security digital vault!"
  },
  {
    "instruction": "Explain IATA Resolution 024e to a junior agent.",
    "input": "The agent is confused by a new type of barcode on a baggage tag.",
    "output": "That's the 024e standard! It’s the new 'Universal Language' for bag tags. It allows different airlines and airports to read the same tag easily, even if they use different systems. It’s like every airport finally agreed to use the same 'Alphabet' so they can all read the 'Address' on your bag without getting lost."
  },
  {
    "instruction": "Explain baggage barcode standard parsing to a data engineer.",
    "input": "Our kiosk scanner is failing to read tags from partner airlines.",
    "output": "You need to update your parser to the IATA 024e XML/Binary standard. The 'License Plate' (the 10-digit bag number) is now often embedded in a 2D barcode (like a QR code) or an RFID chip, rather than just the old 1D stripes. Your scanner needs to extract the 'Issuer Code' and the 'Bag Serial' from the new data structure. Check the 'Tag Data Construct'—if you're just looking for a 1D string, you're missing 50% of the modern tracking data."
  },
  {
    "instruction": "Explain the new 'Smart Tags' to a traveler.",
    "input": "A traveler notices their bag tag looks different this time.",
    "output": "Good eye! That’s a new 'Smart Tag.' It uses a global standard that makes it much easier for airports all over the world to 'talk' to your bag. Whether you’re in New York or a small town in Greece, the machines there can now read your tag perfectly. It’s like your bag has its own 'International ID' that works everywhere, making sure it follows you to the right plane every time!"
  },
  {
    "instruction": "Explain ATOL Protection to a junior agent.",
    "input": "A UK traveler is asking if their 'Flight + Hotel' package is protected.",
    "output": "ATOL is the UK's 'Travel Safety Net.' If the travel company goes bust, the government ensures the traveler isn't stranded abroad and gets their money back. But it only applies to 'Packages' (like Flight + Hotel). If you book them separately, they might not be covered. Always issue that 'ATOL Certificate'—it’s the traveler's 'Guarantee Bond'!"
  },
  {
    "instruction": "Explain ATOL certificate generation logic to a data engineer.",
    "input": "We need to automate ATOL certificate delivery.",
    "output": "Your logic must identify 'Linked Travel Arrangements' or 'Packages'. If the database shows a `FlightID` AND a `HotelID` booked within 24 hours for the same traveler, trigger the ATOL API. You'll need to pass the 'Total Package Price' and 'Agency ID' to the CAA (Civil Aviation Authority) portal to receive a PDF certificate and a unique reference number. This must be stored in the 'Documents' table and emailed to the user instantly to comply with UK law."
  },
  {
    "instruction": "Explain financial protection to a UK traveler.",
    "input": "A traveler is worried about their holiday company going out of business.",
    "output": "You have absolutely nothing to worry about! Your trip is 'ATOL Protected.' Think of it as a government-backed 'Travel Insurance' that’s already built into your price. If anything ever happens to the travel provider, the ATOL system kicks in to make sure you can finish your holiday safely or get a full refund. It’s the ultimate peace-of-mind for your vacation!"
  },
  {
    "instruction": "Explain CORSIA to a junior agent.",
    "input": "A traveler is asking why there's a new 'Carbon Fee' on their corporate ticket.",
    "output": "CORSIA is the airline industry's 'Green Promise.' It’s a global plan to 'Offset' the carbon from flights. That fee is the traveler's contribution to environmental projects like planting forests or building wind farms. It’s how we make sure that as the world flies more, the planet doesn't pay the price. It's 'Carbon-Neutral' growth for the sky!"
  },
  {
    "instruction": "Explain carbon footprint calculation logic to a data engineer.",
    "input": "We need to show 'CO2 Emissions' for every flight in the search results.",
    "output": "You need a 'Carbon Calculator' API (like ICAO or Thrust Carbon). The math isn't just `Miles * Constant`. You need to account for 'Aircraft Type' (an old 747 vs a new A350), 'Load Factor' (how full the plane is), and 'Cabin Class' (a First Class seat 'costs' more CO2 because it takes up more space). Your search results should display an 'Estimated CO2' field, allowing users to filter by the 'Greenest' flight option."
  },
  {
    "instruction": "Explain 'Carbon Offsetting' to a traveler.",
    "input": "A traveler wants to know how they can fly more sustainably.",
    "output": "It’s wonderful that you’re thinking about the planet! We use a global system called CORSIA to help 'balance out' the impact of your flight. For every bit of carbon your journey produces, a contribution is made to environmental projects around the world. It’s like 'planting a tree' for every few miles you fly, helping to keep the skies blue for the next generation of travelers!"
  },
  {
    "instruction": "Explain Passport Validity Rules to a junior agent.",
    "input": "A traveler’s passport expires in 4 months, and they are flying to Thailand.",
    "output": "Stop! They might get turned away at the gate. Many countries have a '6-Month Rule'—your passport must be valid for 6 months *after* your return date. Think of it as a 'Safety Buffer.' Even if you're only staying for a week, the government wants to know you have enough 'time left' on your ID in case of an emergency. Always check the 'TIMATIC' database for the exact rules for each country!"
  },
  {
    "instruction": "Fix My Itinerary: Flag a Passport Validity risk.",
    "input": "Itinerary: London -> Bangkok (Travel Date: June 1st, Return Date: June 15th). Passport Expiry: August 20th.",
    "output": "I have to flag this immediately! Thailand strictly enforces a 6-month validity rule. While the traveler's passport is valid during the trip, it expires only 2 months after they return. The airline will likely deny boarding at Heathrow because the passport doesn't have the required 6-month 'buffer'. The traveler needs to renew their passport before this trip to avoid being stranded at the airport!"
  },
  {
    "instruction": "Explain passport validity validation to a data engineer.",
    "input": "We need an automated 'Passport Check' in our profile system.",
    "output": "You need to implement a 'Validity Logic' microservice. It should take `Passport_Expiry_Date` and compare it to `Itinerary_Return_Date`. The alert threshold should be `Return_Date + 185 days`. However, this is 'Destination Dependent'. You’ll need a lookup table (via TIMATIC API) because some countries only require 3 months or even just 'validity for the duration of stay'. Your system should default to the strictest rule (6 months) unless the specific destination API says otherwise."
  },
  {
    "instruction": "Explain Visa-on-Arrival (VoA) Logic to a junior agent.",
    "input": "A traveler is worried they don't have enough time to get a visa for their trip next week.",
    "output": "Check the 'VoA' status! For some countries, you don't need to go to an embassy beforehand. You just show up, pay a fee at a little kiosk, and get your stamp right there. It’s like a 'Tickets at the Door' event—as long as you have your passport and the fee ready, you're good to go! Just make sure they have a 'Passport Photo' and cash, as some kiosks are old-school."
  },
  {
    "instruction": "Explain VoA 'Proof of Intent' requirements to a data engineer.",
    "input": "The VoA application is failing because the user doesn't have 'Supporting Docs'.",
    "output": "VoA logic isn't just about the 'Stamp'. Most VoA countries require 'Proof of Onward Travel' and 'Proof of Accommodation'. Your system should check if the PNR contains a return flight and at least one hotel segment. If not, the 'VoA Eligibility' flag in your UI should show a warning: 'Visa-on-Arrival possible, but requires proof of return ticket.' Without this, the traveler might be denied boarding by the airline, as the airline is liable for their return if the visa is rejected."
  },
  {
    "instruction": "Explain the convenience of Visa-on-Arrival to a traveler.",
    "input": "A traveler is stressed about getting a visa for their last-minute trip.",
    "output": "Good news! For your destination, you can actually get your visa right when you land at the airport. It's called 'Visa-on-Arrival.' Think of it like 'Express Entry'—you just head to a special desk after you get off the plane, show your passport, and they'll take care of the rest right there. It saves you weeks of waiting at an embassy!"
  },
  {
    "instruction": "Explain Dangerous Goods Regulations (DGR) to a junior agent.",
    "input": "A traveler asks if they can pack their high-powered e-bike battery in their checked luggage.",
    "output": "Absolutely not! That’s a 'DGR' red alert. Large lithium batteries can catch fire in the cargo hold where no one can see them. They *must* be carried in the cabin or shipped via a special cargo service. Think of DGR as the 'Safety Shield' for the plane—it keeps 'Volatile Guests' like chemicals and batteries out of the dark cargo hold where they could cause trouble."
  },
  {
    "instruction": "Explain DGR 'Self-Declaration' logic to a data engineer.",
    "input": "We need to add a 'Safety Checklist' to the online check-in flow.",
    "output": "You need a 'Smart Hazmat Filter.' When a user checks in, present a visual grid of 'Forbidden Items' (batteries, aerosols, flammables). Your system must record a `timestamped_affirmation` that the user has read and complied. If they select 'Yes' to carrying specific items (like spare Li-ion batteries), your logic should trigger a pop-up with 'Carry-on Only' instructions and flag the PNR for 'Manual Review' at the bag drop counter."
  },
  {
    "instruction": "Explain why certain items are 'Forbidden' to a traveler.",
    "input": "A traveler is annoyed that they can't put their spare camera batteries in their suitcase.",
    "output": "I know it seems like a hassle, but it's for a very important safety reason! Lithium batteries are actually much safer in the cabin with you. If one were to overheat, the crew could deal with it instantly. Down in the cargo hold, it’s much harder to spot a problem. It’s one of those rules that keeps the whole plane safe and sound while you're in the air!"
  },
  {
    "instruction": "Explain Minority Consent (UMNR) to a junior agent.",
    "input": "An agent is booking a trip for a 14-year-old traveling alone to Mexico.",
    "output": "You need a 'Minority Consent' form. Many countries require a notarized letter from *both* parents giving permission for the child to travel alone. It’s a 'Safety Guard' against child abduction. If that kid shows up at the border without that 'Golden Ticket' (the letter), they could be held at the airport until a parent arrives. Always double-check the 'Entry Requirements' for minors!"
  },
  {
    "instruction": "Explain UMNR (Unaccompanied Minor) age-logic to a data engineer.",
    "input": "The booking engine is allowing 10-year-olds to book without supervision.",
    "output": "You need a 'Hard Block' on age. Calculate `Age = TravelDate - BirthDate`. If Age < 12, the 'Self-Service' booking must be disabled, and a 'Call to Book' message shown. For ages 12-17, the system must trigger a mandatory 'Minority Consent' checkbox and a field for 'Escort at Origin' and 'Guardian at Destination' contact details. These must be mapped to the `UMNR` SSR code in the GDS to ensure the airline's 'Chaperone Service' is activated."
  },
  {
    "instruction": "Explain why a child needs a special letter to a parent.",
    "input": "A parent is confused why their teenager needs a 'notarized letter' just to visit grandma.",
    "output": "It’s a global safety rule designed to protect children. Think of it as a 'Security Clearance' for your teen. Governments want to be 100% sure that both parents have agreed to the trip. Having that notarized letter in their backpack is like a 'VIP Pass' through customs—it proves everything is official and safe, so your teen can head straight to grandma's without any delays!"
  },
  {
    "instruction": "Explain Inadmissible Passengers (INAD) to a junior agent.",
    "input": "A traveler was sent back from the border because they didn't have a visa.",
    "output": "That's an 'INAD' situation, and it’s a nightmare for everyone. The airline gets a massive fine, and they have to fly the person back immediately. It’s like a 'Bouncer' at a club sending you home because you’re not on the list. Our job is to be the 'Pre-Bouncer'—we check the 'Guest List' (visa rules) before they even buy their ticket so they don't get rejected at the door."
  },
  {
    "instruction": "Explain 'INAD' liability tracking to a data engineer.",
    "input": "The finance team is seeing 'Fines' from airlines for 'Documentation Errors'.",
    "output": "We need to log every 'Visa Check' we perform. If an agent (or our automated system) clears a passenger for travel, store the `Visa_Validation_ID` and a screenshot of the rules at that timestamp. If a passenger is later declared 'INAD', we use this 'Audit Trail' to prove we performed due diligence. If the airline's fine is due to our system's error, we are liable. If the traveler lied about their documents, the 'Audit Trail' protects us."
  },
  {
    "instruction": "Reassure a traveler who is worried about being 'Denied Entry'.",
    "input": "A traveler is terrified they'll be 'sent back' from their vacation destination.",
    "output": "We are here to make sure that never happens! We use a professional system that cross-checks your passport and nationality against the destination's rules in real-time. Think of us as your 'Legal Scouts'—we go ahead and check all the digital 'border crossings' for you. If we say you're good to go, it's because all the 'Green Lights' are on in the official databases!"
  },
  {
    "instruction": "Explain OFAC Sanctions to a junior agent.",
    "input": "An agent is trying to book a hotel in Havana, Cuba for a US-based traveler.",
    "output": "Watch out! That’s an 'OFAC' minefield. US citizens can only travel to Cuba for specific 'Approved Reasons' (like 'Support for the Cuban People'). You can't just go for a beach holiday. Think of it as a 'Restricted Zone'—you need a 'Legal Reason' to cross the line. If we book a restricted hotel without the right paperwork, our agency could face huge government fines!"
  },
  {
    "instruction": "Explain 'Sanctioned Entity' filtering to a data engineer.",
    "input": "Our hotel API is returning properties that are on the US 'Prohibited' list.",
    "output": "You need to ingest the 'OFAC Specially Designated Nationals' (SDN) list. Your search engine must cross-reference hotel 'Parent Companies' and 'Street Addresses' against this list. If a match is found (e.g., a hotel owned by a sanctioned military group), you must 'Hard-Hide' it for users with a US Billing Address. Don't just show 'Sold Out'; the property should not exist in their search results to ensure 100% legal compliance."
  },
  {
    "instruction": "Explain travel restrictions to a traveler going to Cuba.",
    "input": "A US traveler wants to know if they can just 'go to the beach' in Cuba.",
    "output": "Cuba is a beautiful destination, but it has some 'Special Rules' for US travelers. Instead of a standard vacation, the US government asks that your trip fits into one of several 'Meaningful Categories'—like cultural exchange or supporting local businesses. Think of it as a 'Purpose-Driven' trip! We’ll help you choose the right category so your trip is both legal and incredibly rewarding!"
  },
  {
    "instruction": "Explain the Montreal Convention to a junior agent.",
    "input": "A traveler's suitcase was destroyed by the airline and they want a new one.",
    "output": "The Montreal Convention is the 'Global Insurance Treaty' for the sky. It sets the maximum amount an airline has to pay for lost or damaged bags (about $1,700). It’s like a 'Universal Guarantee'—it doesn't matter which airline you fly; if they break your stuff, the rules for how much they owe you are the same everywhere in the world."
  },
  {
    "instruction": "Explain Montreal Convention 'SDR' (Special Drawing Rights) calculation to a data engineer.",
    "input": "Our 'Baggage Claim' tool needs to show the maximum compensation in local currency.",
    "output": "The Montreal Convention uses 'SDRs' (Special Drawing Rights)—a 'World Currency' made of a basket of major currencies. The limit is currently 1,288 SDRs. You need an API that converts `SDR -> USD` or `SDR -> EUR` daily. Your claim tool should calculate: `MIN(User_Declared_Value, 1288 * Current_SDR_Rate)`. This ensures that the compensation offer you show is always legally accurate and up-to-date with global treaty changes."
  },
  {
    "instruction": "Explain baggage damage rights to a traveler.",
    "input": "A traveler is upset because the airline offered them 'only a fraction' of what their bag was worth.",
    "output": "I know it’s disappointing when your favorite things are damaged. There is an international agreement called the 'Montreal Convention' that all airlines follow. It sets a fair 'Safety Net' for everyone, ensuring a standard level of payment for luggage issues. While it has a maximum limit, it’s there to make sure you're treated fairly and consistently, no matter where you fly in the world!"
  },
  {
    "instruction": "Explain Bilateral Air Service Agreements to a junior agent.",
    "input": "The agent wonders why there are 20 flights a day to Paris but only 1 to a similar-sized city.",
    "output": "That's because of 'Bilateral Agreements.' It’s a 'Handshake' between two countries that decides exactly how many flights can go back and forth. It’s like a 'Trading Permit'—countries negotiate these for years. If the permit only allows 7 flights a week, that’s all we can book, no matter how many people want to go!"
  },
  {
    "instruction": "Explain 'Slot' and 'Frequency' constraints to a data engineer.",
    "input": "Our forecasting model is predicting 5 new flights on a route next year, but they haven't appeared.",
    "output": "Your model is ignoring 'Bilateral Constraints'. Even if demand is high, the airline can't add flights if the 'Treaty' limit has been reached. You need to add a 'Regulatory Ceiling' variable to your model. Check the ICAO database for the 'Frequencies' allowed between those two nations. If the treaty is capped at 14 flights a week, your forecast for '19 flights' is technically impossible until a new treaty is signed."
  },
  {
    "instruction": "Explain why flight options are limited to a traveler.",
    "input": "A traveler is frustrated that there's only one flight time available for their destination.",
    "output": "I hear you! Sometimes, the number of flights between two countries is decided by a 'Bilateral Agreement'—think of it as a 'Shared Calendar' between two nations. They agree on a specific number of 'slots' each day to keep things balanced and fair. While it means fewer times to choose from, it ensures that both countries' airlines have a fair chance to serve you!"
  },
  {
    "instruction": "Explain Digital Nomad Visas (Tax Residency) to a junior agent.",
    "input": "A traveler wants to stay in Portugal for 6 months and work from their laptop.",
    "output": "They need a 'Digital Nomad Visa!' You can't just go on a tourist visa if you're 'working,' even if it’s on a laptop. It’s like a 'Work-from-Anywhere' permit. It gives them the legal right to stay longer, but it also has 'Tax Hooks'—after a certain time, they might have to pay local taxes. Always advise them to check with a tax expert so they don't get a surprise bill from the government!"
  },
  {
    "instruction": "Explain 'Tax Residency' alerts to a data engineer.",
    "input": "We need to flag long-term stays that might trigger tax liabilities for our users.",
    "output": "This is a 'Duration Threshold' logic. Most countries trigger 'Tax Residency' after 183 days in a calendar year. Your system should track cumulative stays across multiple PNRs. If a user's total stay in a single country is approaching 150 days, send an automated 'Compliance Alert'. You'll need a 'Tax Treaty' lookup table because some countries have different thresholds for 'Digital Nomads' vs. 'General Travelers'."
  },
  {
    "instruction": "Explain the benefit of a Digital Nomad Visa to a traveler.",
    "input": "A traveler is excited about working from a beach in Bali for half a year.",
    "output": "That sounds like an absolute dream! To make sure your 'office at the beach' is 100% legal, you’ll want a 'Digital Nomad Visa.' Think of it as an 'Extended Stay Pass' that welcomes you to live and work there locally. It keeps everything official with immigration and can even give you some great local perks, like easier access to housing and community events!"
  },
  {
    "instruction": "Explain IATA EasyPay to a junior agent.",
    "input": "The agent is confused by a new payment option for issuing tickets.",
    "output": "IATA EasyPay is like a 'Pre-Paid Wallet' for our agency. Instead of waiting for a weekly bill, we put money in the 'wallet' and it's deducted the moment we issue the ticket. It’s much safer for the airlines because they get their money instantly, and it gives us a 'Secure Credit Line' so we never run out of ticketing power!"
  },
  {
    "instruction": "Explain 'EasyPay' wallet reconciliation to a data engineer.",
    "input": "The finance team can't balance the 'EasyPay' account.",
    "output": "EasyPay transactions are 'Real-Time Settled.' Unlike BSP (which is batched), every ticket issued via EasyPay has a corresponding `DEBIT_EVENT` in the IATA portal. You need to pull the 'Daily Transaction File' from IATA and match the `DocumentNumber` to our PNR. If a ticket is 'Voided,' the refund to the wallet is also instant. Your logic needs to handle 'Partial Voids' differently than credit card refunds to avoid double-counting the available balance."
  },
  {
    "instruction": "Explain the security of 'Pre-Paid' ticketing to a traveler.",
    "input": "A traveler asks if their payment goes directly to the airline.",
    "output": "Yes! We use a high-tech system called 'EasyPay' that works like a secure 'Instant Transfer.' The moment your ticket is confirmed, your payment is safely moved to the airline's account through a protected 'Digital Vault.' It’s one of the fastest and most secure ways to ensure your seat is locked in and your ticket is issued immediately!"
  },
  {
    "instruction": "Explain Yellow Fever Certification to a junior agent.",
    "input": "A traveler is flying from Brazil to South Africa and doesn't know about vaccines.",
    "output": "They must have their 'Yellow Card!' Many countries won't let you in—or even through the airport—if you’ve recently been in a 'Yellow Fever Zone' without proof of vaccination. Think of the Yellow Card as a 'Health Passport.' Without that little yellow book, the traveler might be stuck in quarantine or sent back on the next plane!"
  },
  {
    "instruction": "Explain vaccination-requirement logic to a data engineer.",
    "input": "We need to show 'Health Alerts' based on the user's travel history.",
    "output": "This is 'Chained Logic'. You don't just look at the 'Destination'; you have to look at the 'History'. If `Current_Trip_Origin` is a 'Yellow Fever Zone' AND `Destination` requires certification, trigger an alert. The complexity is 'Transit History': if a traveler spent 12 hours in a 'Yellow Fever Hub' (like Panama City) on their *previous* PNR within the last 10 days, the *current* destination might still require the certificate. You need to scan the user's 'Recent Flown Segments' for any matches against the WHO 'Endemic Countries' list."
  },
  {
    "instruction": "Explain the importance of the 'Yellow Card' to a traveler.",
    "input": "A traveler is annoyed that they have to carry a physical yellow booklet for their trip.",
    "output": "It feels a bit old-fashioned, doesn't it? But that 'Yellow Card' is actually a very important 'International Health Badge.' It’s a globally recognized proof that you're protected, and it's required by many countries to keep their local communities safe. Think of it as a 'Golden Key'—it’s the one thing that ensures you can breeze through health checks and get straight to your safari!"
  },
  {
    "instruction": "Explain ESTA/eTA Protocols to a junior agent.",
    "input": "A UK traveler thinks they don't need a visa for the US because they are a 'Visa-Waiver' citizen.",
    "output": "They still need an 'ESTA!' It’s not a full visa, but it’s a 'Pre-Screening.' Think of it as 'RSVP-ing' to the country. You're telling the US, 'Hey, I'm coming!' and they give you a 'Digital Thumbs-Up' back. If they don't have that thumbs-up at the airport, the airline won't even let them check in!"
  },
  {
    "instruction": "Explain ESTA 'Authorization Status' polling to a data engineer.",
    "input": "We need to check if our users have their US entry permits ready.",
    "output": "You need to integrate with the 'CBP ESTA' status API. The user provides their passport number, and the API returns `AUTHORIZATION_APPROVED`, `PENDING`, or `NOT_FOUND`. Your system should poll this status 72 hours before departure. If it’s `NOT_FOUND`, you must trigger an urgent email to the user with a direct link to the official application site. Be careful with 'Third Party' scam sites—your link must point to the `.gov` domain to protect your users."
  },
  {
    "instruction": "Explain why they need an 'ESTA' to a traveler.",
    "input": "A traveler is confused why they have to pay $21 if they don't need a visa.",
    "output": "It’s a bit of a 'Digital Handshake'! Even though you don't need a full visa, the US government asks all visitors to do a quick 'Pre-Registration' called an ESTA. It’s a simple way for them to say 'Welcome!' before you arrive. It’s valid for two years, so it's like having a 'Multi-Pass' that makes all your future US trips much faster and easier!"
  },
  {
    "instruction": "Explain the Package Travel Directive (PTD) to a junior agent.",
    "input": "An agent is building a 'Custom Holiday' with a flight, a car, and a hotel.",
    "output": "You've just built a 'Package' under the EU PTD rules! This means our agency is now legally the 'Organizer.' If the airline goes bust, *we* are responsible for getting them home. If the hotel is bad, *we* have to fix it. It’s like being the 'General Contractor' for their holiday—you're guaranteeing the whole thing, not just the individual pieces!"
  },
  {
    "instruction": "Explain 'Linked Travel Arrangement' classification to a data engineer.",
    "input": "We need to categorize our bookings for the legal department.",
    "output": "You need a 'Linkage Algorithm'. If a user purchases one travel service (e.g., a flight) and then buys a second service (e.g., a hotel) through a 'targeted link' or 'cross-sell' on our site within 24 hours, it becomes a 'Package' or a 'Linked Travel Arrangement' (LTA) under the PTD. You must tag these records in the database so the finance team can set aside the required 'Bonding' funds and the system can generate the mandatory 'Standard Information Form' for the user."
  },
  {
    "instruction": "Explain why 'Booking Together' is safer for a traveler.",
    "input": "A traveler asks if they should book their flight and hotel together or separately.",
    "output": "Booking them together is actually a great 'Safety Strategy!' When you combine them, you're protected by the 'Package Travel Directive.' It’s like having a 'Master Guarantee' for your whole trip. If anything goes wrong with one part, it’s our job to fix the whole thing for you. It’s the ultimate way to make sure your vacation is protected from start to finish!"
  },
  {
    "instruction": "Explain DPA (Data Processing Agreements) to a junior agent.",
    "input": "A large hotel chain wants us to sign a 'DPA' before they share their corporate rates.",
    "output": "A DPA is like a 'Data Pre-Nup.' It’s a legal contract that says, 'We both agree to keep the traveler's info safe.' It decides who is responsible if there's a leak. It’s the 'Digital Handshake' that allows us to share sensitive info like names and credit cards while keeping everyone legally protected. We can't move a single byte of data without it!"
  },
  {
    "instruction": "Explain DPA compliance audits to a data engineer.",
    "input": "The legal team wants to know where our 'Partner Data' is stored.",
    "output": "Under our DPAs (Data Processing Agreements), we must be able to provide a 'Data Map'. You need to document every 'Sub-Processor'—that means every API or Cloud Service where traveler data is sent. If we send PII to a hotel in Singapore, we need to ensure that the hotel’s DPA matches our own 'Standard Contractual Clauses'. Your database audit should be able to produce a report showing the 'Data Path' for any given traveler to ensure we are following the 'Privacy-by-Design' rules in our contracts."
  },
  {
    "instruction": "Explain why we have 'Privacy Contracts' to a traveler.",
    "input": "A traveler asks why we share their info with so many 'Partners'.",
    "output": "We only share your info to make your trip possible! To keep everything 100% safe, we have strict 'Privacy Contracts' with every hotel and airline we work with. Think of it as a 'Security Promise'—they’ve legally agreed to guard your data just as carefully as we do. It’s how we ensure that no matter where you are in the world, your private information is always protected by a high-tech shield!"
  },
  {
    "instruction": "Explain Modern Slavery Acts to a junior agent.",
    "input": "An agent is looking at a super-cheap hotel in a region known for labor issues.",
    "output": "We need to be careful! Many countries have 'Modern Slavery Acts' that require us to ensure our suppliers (like hotels and ground transport) treat their workers fairly. It’s not just about the price; it’s about 'Ethical Travel.' If a hotel is 'too cheap,' it might be because they aren't treating people right. We prioritize partners who have a 'Clean Labor' certificate to make sure your traveler stays in a place with good values!"
  },
  {
    "instruction": "Explain supplier 'Ethical Scoring' to a data engineer.",
    "input": "We want to add an 'Ethics Badge' to hotels in our search results.",
    "output": "You need to ingest data from 'Social Responsibility' databases (like EcoVadis or specialized NGO feeds). Your hotel table should have an `ethics_score` field. This score is based on the hotel's 'Modern Slavery Statement' and their labor audits. In your search algorithm, you can 'boost' hotels with a high ethics score or even 'Hide' properties that have been blacklisted for labor violations, ensuring our platform only supports 'Fair-Trade' travel."
  },
  {
    "instruction": "Explain 'Ethical Staying' to a traveler.",
    "input": "A traveler wants to stay in a hotel that 'gives back' to the local community.",
    "output": "I love that! We work hard to partner with hotels that follow 'Modern Slavery' and 'Fair Labor' standards. By choosing a hotel with an 'Ethical Badge,' you’re staying in a place that treats its staff with dignity and supports the local community. It’s a wonderful way to make sure your vacation isn't just a great experience for you, but a positive force for the people who live there too!"
  },
  {
    "instruction": "Explain Split Ticketing to a junior agent.",
    "input": "An agent wants to know why they shouldn't just book two separate one-way tickets to save a client $200.",
    "output": "Split ticketing is like taking two different buses instead of one through-ticket. While it saves money, you lose the 'Safety Net.' If the first flight is late, the second airline doesn't know the traveler exists and will mark them as a 'No-Show.' You’re essentially trading the airline's guarantee for a lower price. Always make sure the traveler knows they are responsible for their own 'connection'!"
  },
  {
    "instruction": "Explain the data tracking challenges of split tickets to a data engineer.",
    "input": "The reporting dashboard is showing two separate 'Trips' for what should be one journey.",
    "output": "Since split tickets generate two distinct PNRs (or two separate ticket numbers in one PNR), your analytics engine sees them as unrelated events. This breaks 'Journey Attribution.' You need to implement a 'Heuristic Linker' that looks for a `Destination_A` that matches a `Departure_B` within a 12-hour window for the same `Passenger_Hash`. Without this, your 'Cost per Trip' metrics will be wildly inaccurate."
  },
  {
    "instruction": "Explain the risks of separate tickets to a traveler.",
    "input": "A traveler found a cheaper way to fly by booking two different airlines separately.",
    "output": "It’s tempting to grab those savings! Just think of this as a 'Self-Connect' trip. Because the airlines aren't 'talking' to each other on one master ticket, they won't help you if a delay on the first flight causes you to miss the second. If you're comfortable with a long layover to act as a 'buffer,' it's a great deal—just remember to pack some extra patience just in case!"
  },
  {
    "instruction": "Explain Open-Jaw Itineraries to a junior agent.",
    "input": "A client wants to fly into Paris but return from Rome.",
    "output": "That's an 'Open-Jaw'—think of it as a triangle with a missing side. In the GDS, you'll need to use an `ARNK` (Arrival Unknown) segment to bridge the gap between Paris and Rome. It often prices out similarly to a standard round-trip because airlines see it as a single 'circle' journey. It's the best way to help a traveler see more of the world without backtracking!"
  },
  {
    "instruction": "Explain Open-Jaw pricing logic to a data engineer.",
    "input": "The pricing API is returning an error when trying to calculate a multi-city route.",
    "output": "The engine is likely failing on the 'Half Round Trip' (HRT) combination logic. For an Open-Jaw (e.g., JFK-CDG / FCO-JFK), the system must find two fares that are 'Combinable' in the fare rules (usually Category 10). If the 'End-on-End' restriction is triggered, it will default to two expensive one-way fares. Ensure your query allows for 'Multi-City' valuation rather than just 'Point-to-Point' to surface the correct combined price."
  },
  {
    "instruction": "Explain the convenience of an 'Open-Jaw' to a traveler.",
    "input": "A traveler is worried that flying home from a different city will be much more expensive.",
    "output": "Actually, it's a travel 'pro-tip!' Airlines often let you 'Open the Jaw' of your trip for a very similar price to a regular return flight. It’s like having a 'Flexible Return'—you can spend your time traveling overland from Paris to Rome and just fly home from there. It saves you the time and cost of circling back to your starting point!"
  },
  {
    "instruction": "Explain Hidden City Ticketing to a junior agent.",
    "input": "A traveler wants to book a flight to Miami with a stop in Charlotte, but they plan to just leave the airport in Charlotte.",
    "output": "That’s 'Hidden City Ticketing' or 'Skip-lagging,' and it’s a big 'Red Flag.' If the airline catches them, they will cancel the entire rest of the itinerary, including the return flight. Also, their bags will go to Miami without them! It’s like buying a movie ticket for a double feature but leaving after the first film—except the theater might ban you for life. We generally advise against it to protect the client's loyalty status."
  },
  {
    "instruction": "Explain 'Hidden City' detection logic to a data engineer.",
    "input": "We need to flag PNRs that look like 'Skip-lagging' before the airline sends a debit memo.",
    "output": "You need to build a 'Differential Price Checker.' Flag any PNR where `Price(A->B->C) < Price(A->B)`. This is the classic 'Hidden City' footprint. Additionally, look for 'One-Way' international bookings where the passenger has no checked bags. By flagging these to the revenue integrity team, you can prevent 'Short-Checking' of bags and protect the agency from future fines."
  },
  {
    "instruction": "Explain why they shouldn't 'Skip-lag' to a traveler.",
    "input": "A traveler saw a tip online about saving money by getting off at a layover.",
    "output": "It sounds like a clever hack, doesn't it? But it's quite risky. Airlines view your ticket as a 'Contract' to go to the final destination. If you hop off early, they usually cancel your return flight immediately, and your frequent flyer points could be frozen. Plus, your checked bags are heading to the final stop without you! It’s much safer to let us find you a legitimate deal to your actual destination."
  },
  {
    "instruction": "Explain Dynamic Pricing Algorithms to a junior agent.",
    "input": "A traveler is complaining that the price went up $50 while they were eating lunch.",
    "output": "That’s the 'Live Brain' of the airline at work. Dynamic pricing is like a digital auction that never stops. The system looks at how many people are searching, what competitors are charging, and even the weather. It’s not personal; it’s just the 'Demand Thermometer' rising. I always tell clients: if you see a price you like, grab it, because the 'brain' never sleeps!"
  },
  {
    "instruction": "Explain 'Price Volatility' data modeling to a data engineer.",
    "input": "We want to predict when a fare is 'At Its Lowest'.",
    "output": "You're modeling 'Dynamic Decay.' You need to ingest historical 'ATPCO' fare filings and real-time 'Availability' snapshots. The key features aren't just 'Days to Departure,' but also 'Velocity' (how fast seats are selling in a specific class). If the 'K' class is 80% full but 'L' class just opened up, the price will dip. Your model needs to track these 'Inventory Bucket' shifts to provide a 'Buy/Wait' recommendation."
  },
  {
    "instruction": "Explain why prices change so fast to a traveler.",
    "input": "A traveler feels like the airline is 'spying' on them because the price changed.",
    "output": "I promise it's not you! It's actually a global 'Digital Dance.' Think of it like the stock market for seats. As other people around the world book, the 'supply' of cheaper seats goes down, and the system automatically shifts to the next 'price bracket.' It’s like a popular concert—the closer it gets to the show, the more the remaining tickets are worth!"
  },
  {
    "instruction": "Explain Waitlist Clearance Logic to a junior agent.",
    "input": "A traveler with 'Gold Status' is on a waitlist for a business class upgrade.",
    "output": "The waitlist isn't 'First-Come, First-Served.' It’s a 'Priority Ladder.' Your Gold traveler is higher on the ladder than a Silver traveler, but both might be behind a 'Full Fare' passenger. The GDS uses a 'Priority Code' (like UPG1). Think of it as a VIP line at a club—status and the price of your original ticket are what get you past the velvet rope!"
  },
  {
    "instruction": "Explain waitlist status 'Polling' to a data engineer.",
    "input": "Our app needs to show the user's 'Waitlist Position'.",
    "output": "Airlines rarely expose the exact 'Position' via API for competitive reasons. You'll need to poll the PNR for 'Segment Status' changes (e.g., from `HL` to `HK`). However, you can 'estimate' the likelihood of clearance by checking the `LoadFactor` and the `AuthorizedLevel` of the cabin. If the cabin is `J4 C2 D0`, a waitlisted `D` class passenger is unlikely to clear. Your UI should manage expectations as 'Low', 'Medium', or 'High' probability."
  },
  {
    "instruction": "Explain how waitlists work to a traveler.",
    "input": "A traveler is frustrated that they haven't been 'confirmed' for their upgrade yet.",
    "output": "Waitlisting is like being on the 'On-Call' list for a premium table at a restaurant. The airline wants to see if someone will buy that seat at full price first. As we get closer to the flight, they start opening those seats to their most loyal guests. Since you have great status with them, you’re at the front of the line! We'll keep 'knocking on the door' for you until the moment you board."
  },
  {
    "instruction": "Explain Code-Sharing Operations to a junior agent.",
    "input": "A traveler is confused because they booked with Delta but the plane says 'Air France'.",
    "output": "That's a 'Code-Share'—think of it as a 'Team Effort.' Delta is the 'Marketing Carrier' (they sold the ticket), and Air France is the 'Operating Carrier' (they own the plane). The 'Marketing' carrier handles the money, but the 'Operating' carrier handles the actual flight and the bags. Always tell the traveler to look for the 'Operated by' note—that’s the desk they need to find at the airport!"
  },
  {
    "instruction": "Explain 'Marketing vs Operating' carrier data mapping to a data engineer.",
    "input": "The 'Baggage Allowance' logic is failing on code-share flights.",
    "output": "This is a common 'Attribute mismatch.' Under IATA Resolution 302, the baggage rules of the 'Most Significant Carrier' (usually the one crossing the ocean) or the 'Marketing Carrier' apply, but the 'Operating Carrier' enforces the physical weight. You need to store both the `Marketing_Carrier_Code` and the `Operating_Carrier_Code`. When querying the 'Baggage API,' ensure you are passing the 'Operating' code for technical specs and the 'Marketing' code for the allowance policy."
  },
  {
    "instruction": "Explain why their plane looks different to a traveler.",
    "input": "A traveler is worried they are at the wrong gate because the airline name is different.",
    "output": "You’re in exactly the right place! Think of it like a 'Partner Program.' Your favorite airline has teamed up with another world-class carrier to get you to your destination. You get all the benefits and miles of your original booking, but you get to experience the service of their partner today. It’s like booking a trip through a boutique agency but staying at a luxury resort they’ve hand-picked for you!"
  },
  {
    "instruction": "Explain Married Segment Logic to a junior agent.",
    "input": "An agent can see a seat from London to Dubai, and another from Dubai to Sydney, but can't book them together.",
    "output": "That's 'Married Segment' logic. The airline has 'linked' those two flights together as a special 'bundle.' They might be willing to sell the whole journey to Sydney, but they won't let you 'break the marriage' to just take the first leg. It’s like a 'Value Meal' at a fast-food place—the airline decided the bundle is worth more to them than the individual items. If you try to 'divorce' them, the system will reject the booking!"
  },
  {
    "instruction": "Explain Married Segment availability to a data engineer.",
    "input": "The 'Lowest Fare' search is returning higher prices than the sum of the individual legs.",
    "output": "Your search engine is hitting 'Married Segment Control.' Airlines use a different 'Inventory Bucket' for a connection (A->B->C) than for local legs (A->B). The availability `J4` on the long-haul might only be available if the feeder leg is also booked. You must ensure your API request uses the `Diagnostic_Married_Segment` flag. If you try to price them as separate 'Point-to-Point' requests, the GDS will return a 'No Availability' error or a much higher price."
  },
  {
    "instruction": "Explain why they can't 'just take one flight' to a traveler.",
    "input": "A traveler wants to book a long trip but just get off at the first stop because it's cheaper.",
    "output": "I know it seems counter-intuitive! Airlines look at your whole journey as a single 'linked' experience. They often offer a lower price for the long trip to stay competitive on that route. Think of it like a 'package deal'—the airline only offers that specific price if you take the whole package. If you miss one part, the 'digital link' breaks, and the rest of the trip is automatically canceled by their system."
  },
  {
    "instruction": "Explain Load Factor Analysis to a junior agent.",
    "input": "An agent is wondering why a flight is so expensive when the seat map looks half empty.",
    "output": "Never trust the seat map! Many people haven't picked seats yet, or the airline is 'holding' seats for a group. The 'Load Factor' is the real truth—it’s the percentage of actual tickets sold vs. seats on the plane. Even if the map looks empty, the 'Load Factor' might be 95%, which is why the price is so high. It’s like a 'sold out' concert where everyone is still in the lobby—the seats are taken, even if you can't see the people yet!"
  },
  {
    "instruction": "Explain 'Load Factor' vs 'Yield' to a data engineer.",
    "input": "Our revenue report shows high 'Occupancy' but low 'Profit' for a specific route.",
    "output": "You’re seeing the 'Load Factor' trap. A 99% Load Factor (full plane) doesn't mean profit if everyone bought 'K' class (cheap) seats. You need to calculate `RPM` (Revenue Per Mile) and `LF` (Load Factor) together. If the Load Factor is high but the Yield is low, the 'Inventory Nesting' logic was too aggressive. We need to adjust our dashboard to show 'Yield per Cabin' so the team can see where we're 'giving away' seats too cheaply."
  },
  {
    "instruction": "Explain why a 'Half-Empty' flight costs so much to a traveler.",
    "input": "A traveler is frustrated that the price is high even though there are 'plenty of seats' on the map.",
    "output": "It’s a common mystery! That seat map is just a 'rough draft.' Many business travelers or groups book their tickets but don't choose their seats until the very last minute. The airline's computer knows those seats are already 'spoken for,' even if they look empty on your screen. It’s like a restaurant that's fully 'booked' even if the tables haven't been sat yet!"
  },
  {
    "instruction": "Explain Fare Filing (ATPCO) to a junior agent.",
    "input": "An agent is curious where the 'Rules' for a ticket actually come from.",
    "output": "ATPCO is the 'World's Fare Library.' Airlines 'file' their prices and rules (like 'No Refunds') there. Think of it as a giant central 'Menu' that every GDS in the world reads. When you see a change in price, it’s because the airline just 'published' a new page in that library. It’s the 'Source of Truth' for everything we sell."
  },
  {
    "instruction": "Explain ATPCO 'Category' parsing to a data engineer.",
    "input": "We need to extract 'Refund Policies' from the fare rules automatically.",
    "output": "You need to parse the ATPCO 'Category' data. Specifically, `Cat 16` (Penalties) and `Cat 33` (Voluntary Refunds). These aren't just text; they are 'Data Tables.' Your parser needs to look for the 'Fare Component' and the 'Penalty Amount' fields. If you just do a keyword search for 'Refund,' you'll miss the nuance—like 'Refundable for a $200 fee.' You need a structured data extractor to turn those 'Cat' rules into a clean 'Refund: Yes/No' flag for the UI."
  },
  {
    "instruction": "Explain the 'Rule Book' to a traveler.",
    "input": "A traveler is asking why there are so many 'conditions' on their cheap ticket.",
    "output": "Every ticket comes with a 'Rule Book' (we call it a fare filing). It’s an international agreement that tells us exactly what we can and can't do—like whether you can change your date or get a refund. For those amazing lower prices, the 'Rule Book' is usually a bit stricter. Think of it as a 'Sale Final' sign at your favorite store—it’s how the airline keeps the price so low for you!"
  },
  {
    "instruction": "Explain Back-to-Back Ticketing to a junior agent.",
    "input": "A traveler wants to avoid a 'Saturday Night Stay' rule by booking two overlapping trips.",
    "output": "That’s 'Back-to-Back Ticketing,' and it’s a major 'Don't.' Airlines have robots that look for this. If they see you flying Out(Trip 1), Out(Trip 2), Return(Trip 2), Return(Trip 1), they’ll know you’re trying to 'trick' the Saturday stay rule. They can cancel your tickets or bill the agency for the difference. It’s like trying to use two different coupons for the same item—the system is trained to spot it!"
  },
  {
    "instruction": "Explain 'Back-to-Back' detection algorithms to a data engineer.",
    "input": "The revenue integrity team wants to block 'Rule Circumvention' bookings.",
    "output": "You need a 'Nested Itinerary' detector. Scan the database for `Passenger_Name` where `Departure_Trip_B` is between `Departure_Trip_A` and `Arrival_Trip_A`. If the destinations are the same (e.g., JFK-LHR-JFK), this is a high-probability 'Back-to-Back' violation. You should flag these for 'Manual Review' before the GDS sends a 'Debit Memo' (a fine) to our accounting department."
  },
  {
    "instruction": "Explain why they shouldn't 'Overlap' trips to a traveler.",
    "input": "A traveler found a 'clever way' to save money by booking two separate trips that overlap.",
    "output": "I see what you’re trying to do—it’s a very creative idea! However, I have to give you a 'heads up.' Airlines have very smart systems that look for 'Overlapping Trips.' If they catch it, they might see it as breaking their 'Saturday Stay' rules and could cancel your flights without warning. It’s much safer for us to find a 'Flexible' fare that gives you the savings without the risk of being stranded!"
  },
  {
    "instruction": "Explain Circle Trip Itineraries to a junior agent.",
    "input": "A traveler is going London -> Tokyo -> Singapore -> London.",
    "output": "That's a 'Circle Trip'—think of it as a world tour that ends back at home! In the GDS, you have to ensure the 'Fare Construction' is valid. The system will look at the 'Global Indicators' (like flying via the Pacific vs. the Atlantic). It’s like a 'Multi-Stop Pass'—as long as you keep moving in one general direction, the price is often much better than booking each flight separately."
  },
  {
    "instruction": "Explain 'Circle Trip' fare construction to a data engineer.",
    "input": "The pricing engine is struggling to find a single 'Base Fare' for a 4-stop trip.",
    "output": "For Circle Trips, the engine uses 'Round-the-World' (RTW) or 'Circle Pacific' logic. The key is the 'Higher Intermediate Point' (HIP) check. If any stop in the circle costs more than the final destination from the origin, that higher price applies to the whole trip. Your API needs to allow for 'Sequential Pricing' and check for 'Surcharges' at each stop. If you try to price it as 4 one-ways, the tax calculation will be significantly higher."
  },
  {
    "instruction": "Explain the 'Circle' journey to a traveler.",
    "input": "A traveler wants to visit three different countries in one go.",
    "output": "That’s what we call a 'Circle Trip,' and it’s one of the best ways to see the world! Instead of flying back and forth, we build a 'one-way loop' that takes you to all three countries and brings you home at the end. It’s often much cheaper than three separate trips, and it turns your travel time into part of the adventure. It’s like having a 'Grand Tour' ticket!"
  },
  {
    "instruction": "Explain Ticketing Time Limits (TTL) to a junior agent.",
    "input": "The agent is confused why a 'Confirmed' booking was cancelled at midnight.",
    "output": "A 'Confirmed' booking is just a 'Reservation'—it’s not a 'Ticket' yet. The TTL is the airline's 'Deadline.' If you don't 'Issue the Ticket' (which means the money moves) before that time, the airline's 'Auto-Purge' robot takes the seat back. Think of it like a 'hold' on a library book—if you don't pick it up by Tuesday, they give it to the next person on the list!"
  },
  {
    "instruction": "Explain TTL automation to a data engineer.",
    "input": "We need to ensure we never lose a booking because an agent forgot the deadline.",
    "output": "You need to ingest the `TKTL` (Ticketing Time Limit) field from the PNR. This is a `datetime` object. Build a 'TTL Monitor' service that sends an 'Urgent' notification to the agent 4 hours before expiration. For high-value corporate clients, you can even build an 'Auto-Ticket' feature that triggers a payment if a 'Pre-Authorization' is on file, ensuring the inventory is never lost to an 'Auto-Purge' event."
  },
  {
    "instruction": "Explain why a price 'expired' to a traveler.",
    "input": "A traveler is upset because the price they saw yesterday is gone today.",
    "output": "I am so sorry! Airlines have a 'Time Limit' on their best prices. Think of it like a 'Flash Sale'—they hold the seat for you for a little while, but if it isn't 'locked in' with a ticket, the system automatically puts it back on the market for other travelers. I’ll dive back into the system right now to find you the next best deal!"
  },
  {
    "instruction": "Explain Re-shopping Engines to a junior agent.",
    "input": "The agent is seeing a 'Better Price Alert' for a flight they booked last week.",
    "output": "That's our 'Re-shopping Engine' at work! It’s like a 'Bargain Hunter' that never stops. Even after you book, it keeps checking the GDS to see if a cheaper 'Bucket' opened up or if the airline launched a surprise sale. If the savings are more than the 'Change Fee,' we can re-book and save the client money. It’s our way of proving we’re always looking out for their wallet!"
  },
  {
    "instruction": "Explain Re-shopping 'Net Savings' logic to a data engineer.",
    "input": "The automation is suggesting re-bookings that actually cost us money.",
    "output": "Your logic is missing the 'Penalty' variable. The formula for a re-shop must be: `(Current_Fare - New_Fare) - (Airline_Change_Fee + Agency_Service_Fee) > Threshold`. You must also check the 'Fare Rules' to ensure the new fare is 'Combinable' with the existing itinerary. If you don't account for the 'Change Fee' (usually found in `Cat 16`), you’ll be suggesting 'savings' that are actually losses."
  },
  {
    "instruction": "Explain the 'Price Protection' to a traveler.",
    "input": "A traveler is worried that the price will drop after they buy their ticket.",
    "output": "Don't worry, we've got your back! We use a 'Re-shopping' system that keeps an eye on the price even after you've booked. If a significantly lower price pops up, we'll let you know immediately. It’s like having a 'Price Match Guarantee'—we’re always hunting for the best value, even when you’re already busy dreaming about your trip!"
  },
  {
    "instruction": "Explain Group Booking Blocks to a junior agent.",
    "input": "An agent is booking a destination wedding for 40 people.",
    "output": "Groups are different—it’s like renting out a whole section of a theater. You get a 'Block' of seats at a 'Fixed Price.' You don't need names yet, but you do need to pay a 'Deposit.' It protects you from the 'Dynamic Pricing' jumps, but you have a 'Utilization' rule: if you don't use 80% of the seats, you might lose your deposit. It’s all about balancing the 'Block'!"
  },
  {
    "instruction": "Explain 'Group Inventory' data sync to a data engineer.",
    "input": "The group booking names aren't syncing to the airline's 'DCS' (Departure Control System).",
    "output": "Group PNRs often use 'Name Slots' (e.g., `10SMITH/GROUP`) until the 'Final List' is provided. You need a 'Manifest Sync' trigger. 30 days before departure, your system must pull the names from our 'Group CRM' and push them into the GDS using the `NM1` format. If the 'DCS' doesn't receive individual names by the 'Deadline' (found in the group contract), the whole 'Block' will be flagged as 'Un-Ticketable'."
  },
  {
    "instruction": "Explain the benefit of a 'Group Block' to a traveler.",
    "input": "A traveler is organizing a family reunion and wants everyone to pay the same price.",
    "output": "That's the best part of a 'Group Block!' We lock in one fair price for everyone in your family, so no one feels left out if they book a little later. Think of it as 'Reserving a Row'—everyone gets to fly together, everyone pays the same, and you have plenty of time to gather all the names. It takes the stress out of organizing for a big crowd!"
  },
  {
    "instruction": "Explain Charters vs. Scheduled Flights to a junior agent.",
    "input": "The agent is booking a trip to a remote island and the flight doesn't have a 'Flight Number' like BA123.",
    "output": "That’s likely a 'Charter.' Scheduled flights are like 'Public Buses'—they run at the same time every day, no matter what. Charters are like 'Private Shuttles'—they only fly because a specific tour operator 'hired' the plane. The rules are much stricter (often 100% non-refundable), and the check-in is usually at a different desk. It’s a 'Private Contract' rather than a 'Public Service'."
  },
  {
    "instruction": "Explain 'Charter' GDS mapping to a data engineer.",
    "input": "Charter flights are showing up with 'Zero' availability in our search engine.",
    "output": "Charters don't use standard 'ATPCO' fare filings. They use 'Manual Pricing' or 'Direct Connect' via the tour operator's API. To show these in our search, you need to create a 'Virtual Carrier' or ingest the 'OAG Charter Feed'. When booking, your system shouldn't try to 'Auto-Price' via the GDS; instead, it must 'Force Price' based on the operator's static rate, or the GDS will return a 'No Fare Found' error."
  },
  {
    "instruction": "Explain the 'Charter' experience to a traveler.",
    "input": "A traveler is confused why their flight doesn't show up on the airport's main website yet.",
    "output": "Don't worry, that's normal for a 'Charter' flight! Think of it as a 'Specially Commissioned' journey just for your vacation group. These flights are often more direct and convenient for remote spots. They have their own dedicated team and schedule, which sometimes appears on the 'Public' boards a little later. It’s like having a 'Private Tour' instead of a public one!"
  },
  {
    "instruction": "Explain Overbooking Strategies to a junior agent.",
    "input": "A traveler was 'Denied Boarding' even though they had a confirmed ticket.",
    "output": "It’s a game of 'Musical Chairs' played with math. Airlines know that on average, 5% of people won't show up. So, they sell 105 seats for a 100-seat plane. Most of the time, the 'No-Shows' make it work out perfectly. But when everyone shows up, the music stops and someone is left without a chair. That’s when the 'Compensation' rules kick in!"
  },
  {
    "instruction": "Explain 'Denied Boarding' probability modeling to a data engineer.",
    "input": "We want to alert travelers if their flight is 'High Risk' for overbooking.",
    "output": "You need to look at the 'Authorized vs Physical' capacity. If a flight is `J4 Y10` but the seat map is full, the airline is 'Overbooked' on paper. Your model should factor in the 'Historical No-Show Rate' for that route/time. If the 'No-Show' rate is 2% but the overbook is 5%, flag the PNR as 'High Risk for IDB' (Involuntary Denied Boarding). We can then advise the traveler to check in early to secure their 'chair'."
  },
  {
    "instruction": "Explain why they were 'Bumped' to a traveler.",
    "input": "A traveler is very upset because they were told the flight is 'Full' despite their ticket.",
    "output": "I am so sorry! It is incredibly frustrating. Airlines use a 'Probability' system to keep their prices low—they assume a few people won't make it to the gate. Today, everyone made it! The good news is that you are now in a 'VIP Negotiation' position. The airline is required to offer you significant compensation and a new flight. I’ll stay on the line with you to make sure they get you the best possible 'thank you' for the inconvenience!"
  },
  {
    "instruction": "Explain Bid-to-Upgrade Systems to a junior agent.",
    "input": "A traveler received an email asking them to 'Bid' for a Business Class seat.",
    "output": "It’s an 'Upgrade Auction.' Instead of a fixed price, the airline says, 'What’s it worth to you?' The higher you bid, the better your chance. It’s a way for the airline to fill their premium seats at the last minute with people who really want them. Think of it as a 'Silent Auction' for a better view!"
  },
  {
    "instruction": "Explain 'Auction' status updates to a data engineer.",
    "input": "Our app needs to show if a user's 'Bid' was accepted.",
    "output": "You need to monitor the 'Document Status' via the airline's 'Ancillary API.' When a bid is accepted, the airline issues an `EMD-S` (Stand-alone Document) for the upgrade and updates the 'Class of Service' in the PNR (e.g., from `Y` to `J`). Your UI should trigger a 'Congratulations' notification the moment the `SegmentStatus` changes to `HK` in the higher cabin."
  },
  {
    "instruction": "Explain the 'Auction' to a traveler.",
    "input": "A traveler wants to know how much they should 'Bid' for an upgrade.",
    "output": "It’s like a 'Secret Auction!' There’s no perfect number, but I usually recommend looking at the 'Regular' price of Business Class and bidding about 40-50% of the difference. It’s a fun way to treat yourself to a little luxury for a price that *you* decide. Think of it as 'Naming Your Own Price' for a bit of extra comfort!"
  },
  {
    "instruction": "Explain Ancillary Revenue Optimization to a junior agent.",
    "input": "The agent is frustrated by how many 'Add-ons' there are (bags, seats, meals).",
    "output": "Airlines have 'Unbundled' their service. It’s like a 'Build-Your-Own' burger. Instead of everyone paying for a 'Combo Meal' (which includes a bag and a seat), you only pay for what you actually use. It keeps the 'Base Price' low for everyone, and it's our job to help the traveler choose the right 'Ingredients' for their trip."
  },
  {
    "instruction": "Explain 'Ancillary' data fragmentation to a data engineer.",
    "input": "We can't find the 'Checked Bag' info in the main PNR data.",
    "output": "Ancillaries aren't stored in the 'Itinerary' segments; they are in the `SSR` (Special Service Request) or `EMD` (Electronic Miscellaneous Document) section. You need to perform a 'Deep PNR' query. If you only look at the `AIR` segments, you'll miss the revenue from bags and seats. Your database needs a separate 'Ancillary' table that links to the `Ticket_Number` to show the 'Total Value' of the booking."
  },
  {
    "instruction": "Explain the 'Unbundled' price to a traveler.",
    "input": "A traveler is annoyed that they have to pay extra for a seat and a bag.",
    "output": "It’s definitely a change from the old days! Think of it as 'Personalized Pricing.' By stripping out those extras, the airline can offer you a much lower 'entry-level' price. If you’re a light traveler who doesn't need a bag, you save a lot! We’ll help you add exactly what you need—nothing more, nothing less—so you only pay for the parts of the flight that matter to you."
  },
  {
    "instruction": "Explain Stopover vs. Layover to a junior agent.",
    "input": "A traveler wants to stay in Dubai for 3 days on their way to Sydney.",
    "output": "A 'Layover' is just a 'short break' (usually under 24 hours) to catch your next plane. A 'Stopover' is when you 'Stay and Play' for more than 24 hours. The big difference is the price: Layovers are usually free, but a Stopover might change your 'Fare Basis' and add local taxes. It’s the difference between a 'Coffee Break' and a 'Mini-Vacation'!"
  },
  {
    "instruction": "Explain 'Stopover' duration logic to a data engineer.",
    "input": "The pricing engine is overcharging for long connections.",
    "output": "Your logic is likely treating a 'Stopover' as two 'One-Way' trips. You need to check the `Cat 8` (Stopovers) rule in the ATPCO filing. Many airlines allow one free stopover. Your pricing query must check if `Timestamp(Arrival_1) - Timestamp(Departure_2) > 24 hours`. If it is, you must apply the 'Stopover Surcharge' code rather than breaking the fare into two pieces, or the price will double."
  },
  {
    "instruction": "Explain the 'Two-for-One' trip to a traveler.",
    "input": "A traveler wants to know if they can 'visit' their connection city.",
    "output": "You absolutely can! It’s what we call a 'Stopover.' It’s like getting two vacations for the price of one. Instead of just hanging out in the airport, you can head out and explore the city for a few days before catching your next flight. Some airlines even offer free hotel stays for these! I’ll check the 'Stopover Rules' to see how we can turn your layover into a highlight of your trip!"
  },
  {
    "instruction": "Explain Surface Segments (ARNK) to a junior agent.",
    "input": "The traveler is flying to Paris, taking a train to Amsterdam, and flying home from there.",
    "output": "You need an `ARNK` (Arrival Unknown) segment! In the GDS, the computer sees a 'Gap' in the journey and gets worried. The `ARNK` tells the system: 'Don't panic, the traveler is taking a train/car for this part, but the trip is still one big circle.' It’s like putting a 'Placeholder' in the timeline so the 'Return' flight knows where to pick them up."
  },
  {
    "instruction": "Explain 'ARNK' segment parsing to a data engineer.",
    "input": "The 'Travel Map' in our app shows a straight line from Paris to Amsterdam that never happened.",
    "output": "Your mapping logic is 'Connecting the Dots' between flight segments. You need to detect `ARNK` (Surface) segments. When you see an ARNK, your UI should change the 'Line Type' to a dashed line or a different icon (like a train or car) and label it as 'Self-Travel.' This prevents the user from thinking they have a flight that doesn't exist and keeps the 'Journey Timeline' accurate."
  },
  {
    "instruction": "Explain the 'Travel Gap' to a traveler.",
    "input": "A traveler is confused by a line that says 'Arrival Unknown' on their itinerary.",
    "output": "That's just a 'Digital Bookmark!' Since you’re taking the train between those two cities, our system puts that note there to remind the airline that you haven't disappeared—you’re just exploring on the ground for a bit. It ensures your flight home stays 'linked' to your arrival. It’s our way of keeping your whole travel story in one place!"
  },
  {
    "instruction": "Explain Revenue Integrity to a junior agent.",
    "input": "The agent is getting emails from the airline about 'Churning' a booking.",
    "output": "Revenue Integrity is the airline's 'Data Police.' If you keep cancelling and re-booking the same seat to 'hold' it for a client, their robots will catch you—that's called 'Churning.' It costs the airline money and blocks other people from buying. They want 'Clean Data.' If the 'Police' catch us, they’ll send the agency a fine. We have to be 'Honest Bookers'!"
  },
  {
    "instruction": "Explain 'Duplicate Detection' logic to a data engineer.",
    "input": "Airlines are complaining that we have 'Ghost Bookings' in our system.",
    "output": "You need a 'Duplicate Resolver.' Scan for PNRs with the same `Name_Hash` AND `TravelDate` AND `Origin/Destination`. If two PNRs exist for the same person on the same day, the Revenue Integrity robot will flag it as 'Double-Booking' and might cancel both. Your system should automatically flag these 'Duplicates' for the agent to cancel the 'extra' one before the airline's 'Police' robot does it for us."
  },
  {
    "instruction": "Explain why their 'Hold' was cancelled to a traveler.",
    "input": "A traveler is upset because their 'provisional' booking disappeared after a few hours.",
    "output": "I am so sorry! Airlines are very strict about 'Holding' seats without a ticket. Their 'Integrity Systems' are constantly scanning for seats that aren't 'confirmed' and they sometimes reclaim them to give other travelers a chance to buy. It’s like a 'Waiting List' at a popular restaurant—if the table isn't fully reserved, they have to give it to the next person in line. Let's get that seat locked in for you now so it’s 100% yours!"
  },
  {
    "instruction": "Explain Point-of-Sale (POS) Logic to a junior agent.",
    "input": "The price for a flight is cheaper when the agent logs in to the 'UK Office' instead of the 'US Office'.",
    "output": "That's 'POS Logic.' Airlines offer different 'Sales' in different countries. Think of it like a global brand that has a 'Winter Sale' in London but not in New York. The GDS checks your 'Pseudo City Code' (your office address) to decide which 'Catalog' you're allowed to see. It’s all about where the 'Cash Register' is located!"
  },
  {
    "instruction": "Explain 'POS' currency and tax discrepancies to a data engineer.",
    "input": "Our global search is showing inconsistent prices for the same flight.",
    "output": "This is a 'Sourcing' issue. Price A is from a US-based PCC (in USD) and Price B is from a UK-based PCC (in GBP). The 'Base Fare' might be the same, but the 'Local Taxes' and the 'Exchange Rate' (ROE) will differ based on the 'Point of Sale.' You must always display the 'POS Country' alongside the price so the user understands why a 'cheap' UK fare might have a high 'Currency Conversion' fee."
  },
  {
    "instruction": "Explain why the price changed when they moved sites to a traveler.",
    "input": "A traveler found a cheaper price on a European website than on a US one.",
    "output": "It’s all about the 'Local Market!' Sometimes airlines run 'Special Sales' that are only available to people buying in a specific country. It’s like a local bakery having a 'neighborhood-only' discount. While we always try to find you the best global price, we’ll help you understand the 'Rules' of those local deals so you can decide if they're right for you!"
  },
  {
    "instruction": "Explain GDS Surcharges to a junior agent.",
    "input": "A traveler is asking why there's an 'Extra Fee' for booking through us instead of the airline directly.",
    "output": "Some airlines add a 'Legacy Fee' (a GDS Surcharge) because it costs them more to send their data through the GDS than through their own website. Think of it as a 'Convenience Fee' for having all your travel—hotels, cars, and multiple airlines—all on one single professional itinerary. It’s the price for the 'Big Picture' view!"
  },
  {
    "instruction": "Explain 'GDS Surcharge' data mapping to a data engineer.",
    "input": "The finance team can't find where the '$15 per segment' fee is coming from.",
    "output": "That’s the 'GDS Surcharge' (often coded as `YQ` or a separate `Service Fee` in the fare construction). Airlines use 'Cat 12' (Surcharges) to file these. You need to ensure your 'Tax Breakdown' logic identifies these specifically. If you just bundle them into 'Taxes,' the user will be confused. Label them as 'Distribution Surcharges' to align with the airline's own terminology."
  },
  {
    "instruction": "Explain the 'Service Fee' to a traveler.",
    "input": "A traveler is annoyed that our price is slightly higher than the airline's 'Direct' price.",
    "output": "I hear you—every dollar counts! That small difference is what the airline charges to 'broadcast' their info to professional systems like ours. Think of it as the 'Management Fee' for your trip. By booking through us, you get 'Whole Trip Protection'—if one airline is late, we handle the others for you. It’s the small price for having a 'Personal Travel Team' in your corner!"
  },
  {
    "instruction": "Explain Inventory Nesting to a junior agent.",
    "input": "The agent is curious why an airline would rather leave a seat empty than sell it for a lower price today.",
    "output": "That's 'Inventory Nesting.' The airline is 'Saving a Seat' for a high-paying business traveler who will book at the very last minute. Think of it like a 'VIP Booth'—they won't give it to a 'General Admission' guest even if the club is empty, because they know a VIP will show up later and pay ten times more. It’s a gamble on 'Value' over 'Volume'!"
  },
  {
    "instruction": "Explain 'Nesting' logic to a data engineer.",
    "input": "The 'Availability' API shows seats in 'Y' but zero in 'K' for an empty flight.",
    "output": "You're seeing the 'Nesting' hierarchy. Seats in higher buckets (Y) 'include' the seats in lower buckets (K). If the airline 'Closes' the K-bucket, they are 'Shielding' that inventory. Your availability engine should track these 'Bucket Closures' over time. If a K-bucket closes 30 days out, it’s a sign of high demand. Use this 'Bucket Velocity' to feed your 'Price Prediction' model—it’s a better signal than just 'Total Seats Remaining'."
  },
  {
    "instruction": "Explain why a 'Sale' ended early to a traveler.",
    "input": "A traveler is frustrated that the 'Economy Light' seats are gone even though the plane is half-empty.",
    "output": "It’s all part of the airline's 'Smart Planning.' They only set aside a small number of seats for those 'Super Sale' prices. Once those are gone, they 'protect' the remaining seats for people who might need to fly at the very last minute. It’s like a 'limited edition' release—once the first few are sold, the rest are kept in the vault for a different kind of traveler!"
  },
  {
    "instruction": "Explain Hotel Channel Management to a junior agent.",
    "input": "A hotel is telling the agent they have no rooms, but they are available on Booking.com.",
    "output": "That’s 'Channel Management.' A hotel has a 'Pile' of rooms, and they give small 'Stacks' to different sites. One stack is for the GDS, one is for Expedia, and one is for their own site. If our 'Stack' (the GDS) is empty, it doesn't mean the hotel is full—it just means that specific 'Pile' is gone. We might need to check a different 'Channel' to find that room for you!"
  },
  {
    "instruction": "Explain 'Channel Conflict' data sync to a data engineer.",
    "input": "We are showing different prices for the same hotel room across two different APIs.",
    "output": "You’re seeing 'Channel Discrepancy.' The hotel's 'Channel Manager' (e.g., Siteminder) hasn't synced the 'Yield' update across all providers yet. You need to implement a 'Consolidated View.' Query both the GDS and the 'Bedbank' APIs (like Hotelbeds) and de-duplicate the results based on the 'Room Code.' If the price difference is > 5%, prioritize the 'Direct Connect' rate as it’s the most likely to be 'Instantly Confirmed'."
  },
  {
    "instruction": "Explain why 'Sold Out' doesn't always mean sold out to a traveler.",
    "input": "A traveler found a room online after we told them the hotel was full.",
    "output": "I’m so glad you found that! Hotels are like big puzzles—they give different 'pieces' of their availability to different websites. Sometimes one site still has a 'piece' left after our professional system has run out. I’ll jump right on that and book it through that alternative 'channel' for you immediately. It’s all about leaving no stone unturned for your trip!"
  },
  {
    "instruction": "Explain Contracted/Negotiated Fares to a junior agent.",
    "input": "The agent is using a 'Private Code' (Cat 35) to get a special rate for a client.",
    "output": "That’s a 'Negotiated Fare.' It’s like a 'Secret Menu' that only our agency can see because we have a special contract with the airline. In the GDS, you have to enter a 'Corporate ID' or a 'Tour Code' to unlock it. It’s a 'Private Handshake' that gives our clients better prices or extra flexibility that the general public can't get!"
  },
  {
    "instruction": "Explain 'Private Fare' (Cat 35) data mapping to a data engineer.",
    "input": "Our API is failing to return 'Corporate Rates' for our biggest client.",
    "output": "You need to include the `Corporate_Account_Code` and `Auth_PCC` in your GDS 'Fare Search' request. Without the correct 'Account Code,' the GDS only returns 'Public' (Cat 25) fares. Ensure your system maps the client's `CompanyID` to their specific 'GDS Account Code' from the CRM. If the mapping is off, the 'Negotiated' fares will remain 'Hidden' in the API response."
  },
  {
    "instruction": "Explain the 'Exclusive Deal' to a traveler.",
    "input": "A traveler is happy that our price is better than what they saw on the airline's own site.",
    "output": "That’s one of the perks of working with us! Because of our long-standing partnership with the airline, they’ve given us an 'Exclusive Rate' that isn't available to the general public. Think of it as a 'Member-Only' discount that we’ve secured just for you. It’s our way of making sure you get the 'VIP' treatment every time you book!"
  },
  {
    "instruction": "Explain Bulk/Net Fares to a junior agent.",
    "input": "The ticket says 'IT' or 'BT' instead of a price (e.g., $500).",
    "output": "That's a 'Net Fare.' 'IT' stands for 'Inclusive Tour' and 'BT' for 'Bulk Ticket.' It means the price is a 'Secret' between us and the airline. You shouldn't tell the traveler the 'Net' price—you only tell them the 'Selling' price. It’s like a restaurant buying ingredients at a wholesale price; the customer only sees the price of the finished meal!"
  },
  {
    "instruction": "Explain 'Net Fare' revenue reconciliation to a data engineer.",
    "input": "The finance system is showing 'Zero' revenue for a $1,000 ticket.",
    "output": "Your parser is looking at the 'Base Fare' field on the ticket. For Net Fares (BT/IT), the ticket price is masked as 'Zero' in the GDS. You need to pull the 'Selling Price' from the 'Accounting Remarks' or the 'Mid-Office' record instead. If you don't use the 'Agent-Entered' price, your revenue reports will fail to capture the 'Mark-up' and the 'Total Transaction Value'."
  },
  {
    "instruction": "Explain why their receipt doesn't show a price to a traveler.",
    "input": "A traveler is confused why their official ticket says 'Bulk Fare' instead of the amount they paid.",
    "output": "Don't worry, your ticket is 100% official! You’ve secured what we call a 'Special Partnership Rate.' These are 'pre-purchased' bundles of seats that allow us to offer you a better deal. Because it's part of a 'wholesale' arrangement, the airline's standard receipt shows it as a 'Bulk' booking. Your official invoice from *us* is your proof of payment for the full amount!"
  },
  {
    "instruction": "Explain Fare Ladder Construction to a junior agent.",
    "input": "The agent is looking at a long string of codes like `LON BA NYC 250.00BA LON 250.00 NUC 500.00END`.",
    "output": "That's the 'Fare Ladder.' It’s the 'Blueprint' of the price. It shows exactly how the fare was built, piece by piece, across every city and every airline. The `NUC` is the 'Neutral Unit of Construction'—a global 'Math Currency' that lets the system add up different currencies without getting confused. Learning to read this is like learning to read an architect's drawing of your trip!"
  },
  {
    "instruction": "Explain 'NUC' conversion logic to a data engineer.",
    "input": "Our 'Price Breakdown' is showing incorrect totals for multi-currency trips.",
    "output": "You’re likely using a live market exchange rate instead of the 'IATA Rate of Exchange' (ROE). The Fare Ladder uses `NUCs` which must be multiplied by the official `ROE` from the day the ticket was issued. If you use a standard 'Forex' API, you'll be off by a few cents or dollars, causing the 'Settlement' to fail. You must ingest the 'IATA ROE' table daily to ensure your 'NUC -> Local Currency' math matches the GDS exactly."
  },
  {
    "instruction": "Explain the 'Price Breakdown' to a traveler.",
    "input": "A traveler is overwhelmed by the long list of fees and codes on their receipt.",
    "output": "I know it looks like a lot of 'Secret Code!' Think of that as the 'Itemized Receipt' for your journey. It shows the 'Base Price' for each flight, the specific 'Global Exchange' rates we used, and every local airport fee. It’s our way of being 100% transparent about where every penny of your travel investment is going—from the fuel in the plane to the security at the gate!"
  },
  {
    "instruction": "Explain Historical Fare Data to a junior agent.",
    "input": "The agent wants to know if they should tell the client to book now or wait.",
    "output": "Use the 'History'! It’s like looking at a 'Weather Forecast' for prices. By looking at what this flight cost on this same week last year, we can see if the current price is a 'Steal' or if it’s unusually high. It’s not a crystal ball, but it’s the best way to give the client an 'Expert Opinion' based on real facts, not just a guess."
  },
  {
    "instruction": "Explain 'Time-Series' fare analysis to a data engineer.",
    "input": "We want to build a 'Fare Trend' chart for our users.",
    "output": "You need to store 'Daily Price Snapshots' for specific city-pairs. Your database should track the `Lowest_Available_Fare` and the `Availability_Bucket` at the same time. By plotting this over a 365-day cycle, you can identify 'Seasonality' vs. 'Anomaly' (like a surprise sale). Your model should use 'Year-over-Year' (YoY) comparisons to account for holidays, rather than just 'Month-over-Month' which is too noisy."
  },
  {
    "instruction": "Explain 'Price Trends' to a traveler.",
    "input": "A traveler asks if the price will go down if they wait until Tuesday.",
    "output": "That’s the 'Million Dollar' question! We've looked at the 'Price History' for this route, and typically, prices start to climb as we get closer to your dates. While there are never any 'guarantees' in the world of airfare, the current price is very strong compared to last year's average. Think of it as 'locking in a win'—it’s a great deal right now, and it lets you start planning the fun parts of your trip today!"
  },
  {
    "instruction": "Explain Loyalty Program Revenue to a junior agent.",
    "input": "A traveler wants to pay for their flight using 'Points,' and the agent isn't sure how that affects the agency.",
    "output": "Points are a 'Digital Currency.' When a traveler uses them, the airline is essentially 'paying' for the seat using their own 'Bank.' For us, it’s a 'Loyalty Booking.' It’s great for the traveler's 'Wallet,' but we have to ensure we're following the 'Reward Rules'—which can be different from regular tickets. It’s like a store 'Gift Card'—it’s real value, just in a different form!"
  },
  {
    "instruction": "Explain 'Miles' valuation data to a data engineer.",
    "input": "We need to show the 'Value' of a points booking in our revenue report.",
    "output": "You need to implement a 'Cents-Per-Point' (CPP) calculation. The formula is: `(Cash_Price_of_Ticket - Taxes_Paid_on_Points_Booking) / Points_Used`. This allows you to 'Normalize' points revenue into a 'Dollar Value'. Store this 'Virtual Revenue' in your dashboard so the business can see the 'Total Volume' we are moving, even if the 'Cash' part of the transaction is low. It’s crucial for showing our 'Total Influence' on the airline's load factor."
  },
  {
    "instruction": "Explain the 'Value' of their miles to a traveler.",
    "input": "A traveler wants to know if they are 'Getting a good deal' by using their points today.",
    "output": "Using your points is like 'investing' your travel history! For this specific flight, your points are worth a lot—it’s like getting a 'high interest rate' on your miles. Since the cash price is high today, using your points is a 'Smart Move' that saves you real money. It’s the perfect time to 'cash in' on all those miles you’ve earned and enjoy a well-deserved win!"
  },
  {
    "instruction": "Explain the 'Food Return' problem to a junior agent.",
    "input": "A guest returned their dinner at the hotel, and the agent isn't sure how to handle the bill adjustment.",
    "output": "Think of the hotel bill as a 'Live Ledger.' When a guest returns a meal, you can't just delete it; you have to 'Void' or 'Post a Correction.' If you just delete the line, the system's memory gets a 'hiccup.' It’s like erasing a word in a pen-written diary—it leaves a mark. Always follow the official 'Void' process so the computer knows exactly why the money changed!"
  },
  {
    "instruction": "Explain how a meal refund breaks the POS-to-Finance pipeline to a data engineer.",
    "input": "The end-of-month revenue report shows a $50 discrepancy in F&B (Food and Beverage).",
    "output": "This is likely a 'Sync Timing' error. If a meal is refunded at the POS after the 'End of Day' (EOD) batch has already been sent to the General Ledger, the POS reflects the refund, but the Ledger still shows the original sale. You need to build a 'Late Adjustment' listener that catches transactions modified *after* their initial export timestamp and flags them as 'Retroactive Voids' to prevent reporting drift."
  },
  {
    "instruction": "Explain a restaurant bill correction to a traveler.",
    "input": "A traveler is worried because they saw a charge on their phone app for a meal they sent back.",
    "output": "Don't worry at all! Our systems work in 'waves.' You might see the initial charge first, but the 'Correction Wave' is right behind it. It’s like a digital eraser—the hotel has already processed the refund, and your bank will reconcile those two numbers shortly. You’ll only see the correct, final total on your statement!"
  },
  {
    "instruction": "Explain Room Move Reconciliation to a junior agent.",
    "input": "A guest was moved from Room 101 to 202, and their mini-bar charges are missing.",
    "output": "Moving a room is like moving houses mid-day. If the 'Folio' (the guest's bill) doesn't move with them in the system, the charges get 'Orphaned' in the old room. Always check that the 'Master Bill' has successfully 'shaken hands' with the new room number, or you'll be hunting for those mini-bar charges at checkout like lost keys!"
  },
  {
    "instruction": "Explain 'Orphaned Folio' logic to a data engineer.",
    "input": "We are losing revenue data when guests change rooms mid-stay.",
    "output": "The issue is that the `Room_ID` is used as a primary key for service charges instead of the `Guest_Stay_ID`. When a room move occurs in the PMS, the 'Linkage' between the transaction and the guest is broken if the system doesn't trigger a 'Folio Migration' event. You need to ensure that all downstream charges (Mini-bar, Spa, POS) are keyed to the `Reservation_UUID`, which remains constant regardless of the physical room number."
  },
  {
    "instruction": "Explain a room change to a traveler.",
    "input": "A traveler is moving to a better room and asks if they need to 'settle up' the first room's bill now.",
    "output": "Not at all! Think of your stay as one single 'digital tab.' Even though you're moving to a fresh new space, your bill follows you automatically like a helpful shadow. You can keep enjoying the hotel's amenities, and we'll have everything perfectly organized for you on one single summary when you’re ready to head home!"
  },
  {
    "instruction": "Explain Partial Refund Logic to a junior agent.",
    "input": "A traveler flew the outbound leg but wants a refund for the unused return leg.",
    "output": "Refunding a 'Half-Used' ticket is like trying to return half a sandwich—the price isn't just 50% of the total. You have to subtract the 'One-Way' price of the flight they *did* take from the total they paid. Often, a one-way is almost as expensive as a round-trip, so the 'Residual Value' might be small. It’s a math puzzle, so always use the 'Refund Quote' tool first!"
  },
  {
    "instruction": "Explain 'Residual Value' calculation for half-used tickets to a data engineer.",
    "input": "The automated refund tool is giving incorrect values for partially flown itineraries.",
    "output": "You can't use `Total_Price / 2`. You must perform a 'Historical Fare Recalculation.' The logic should be: `Refund_Amount = Total_Paid - (One_Way_Fare_at_Time_of_Booking + Cancellation_Penalty)`. You need to query the 'Historical ATPCO' table for the one-way fare that was valid on the original ticketing date. If your system defaults to 'current' one-way prices, you'll either over-refund or under-refund the user."
  },
  {
    "instruction": "Explain why a 'Half Refund' isn't 50% to a traveler.",
    "input": "A traveler is confused why they aren't getting half their money back for their unused return flight.",
    "output": "I know it seems like it should be a simple split! However, airlines price 'Round-Trips' like a special bundle deal. When you only use one half, the 'bundle' breaks, and the airline looks at the cost of a standalone 'One-Way' flight instead. It’s like buying a 'Buy One Get One' deal—if you return the second item, the first one reverts to its full, higher price. We’ll find every penny of value we can for you!"
  },
  {
    "instruction": "Explain Name Transposition to a junior agent.",
    "input": "An agent accidentally entered 'John/Smith' instead of 'Smith/John' in the PNR.",
    "output": "That's a 'Name Swap'! Think of the GDS as a very strict librarian—if the name doesn't match the passport 'Last/First' exactly, the computer says 'I don't know this person.' You usually can't just edit a name in a confirmed PNR. You often have to cancel and start over, or ask the airline for a 'Name Correction' waiver. Always double-check before you hit 'Enter'!"
  },
  {
    "instruction": "Explain the API rejection logic for swapped names to a data engineer.",
    "input": "The APIS (Security) submission is being rejected for a valid passenger.",
    "output": "The government 'Watchlist' API performs a strict string match against `Surname` and `GivenName`. If our UI transposes these fields, the check fails because 'John' is being checked against the 'Surname' list. You should implement a 'Passport OCR' (Optical Character Recognition) step in the UI that automatically maps the MRZ (Machine Readable Zone) fields to the correct PNR segments to eliminate human transposition errors."
  },
  {
    "instruction": "Explain why a 'Small Typo' in a name matters to a traveler.",
    "input": "A traveler is annoyed that they have to pay a fee to fix their first and last name being swapped.",
    "output": "It feels like a tiny detail, but to international security systems, it's like having a different 'Digital Fingerprint.' The computers at the border are very literal—they need the 'Key' (your name) to match the 'Lock' (your passport) perfectly. We're working with the airline to get this 'shaken out' so your journey through the airport is completely smooth and stress-free!"
  },
  {
    "instruction": "Explain Duplicate Segment Conflicts to a junior agent.",
    "input": "An agent booked two different flight options for the same traveler to 'show them both'.",
    "output": "Danger! That’s a 'Duplicate Segment Conflict.' The airline's 'Integrity Robot' will see two seats held for one person at the same time and might cancel *both* of them as a penalty. It’s like trying to sit in two chairs at once—the system knows it's impossible and clears the floor. Always pick one and 'sell' it before the robot catches you!"
  },
  {
    "instruction": "Explain 'Churn' detection algorithms to a data engineer.",
    "input": "We are getting high 'GDS Churn' fees in our monthly bill.",
    "output": "Airlines and GDS providers monitor 'Segments per PNR.' If your system is repeatedly booking and cancelling the same flight (Churning) to extend a TTL (Ticketing Time Limit), it triggers a fee. You need to implement a 'Hold Logic' that prevents the same `User_ID` from booking the same `Flight_ID` more than twice in a 24-hour window without ticketing."
  },
  {
    "instruction": "Explain why a 'Backup Option' was cancelled to a traveler.",
    "input": "A traveler is upset that the 'alternative' flight they were considering was cancelled by the airline.",
    "output": "The airline's system actually does that to protect you! Their 'Smart Robots' look out for multiple bookings for the same person. If they see two seats held for you at the same time, they worry it’s a mistake and clear them out to make sure someone else doesn't accidentally get charged. We’ll help you pick your absolute favorite option and lock it in so it’s 100% secure!"
  },
  {
    "instruction": "Explain an Unscheduled Aircraft Swap to a junior agent.",
    "input": "The airline changed from a Boeing 777 to a 787, and the traveler's seat 42A is gone.",
    "output": "That's an 'Equipment Swap'—the airline changed the 'Physical Map.' Since the new plane might have fewer rows or a different layout, the computer 'unseats' everyone and tries to put them back in similar spots. Think of it like a theater changing the chairs—your 'Row 10' might not exist anymore. You need to go in and manually 'Re-seat' your VIPs to make sure they're still happy!"
  },
  {
    "instruction": "Explain seat-map data drift during aircraft swaps to a data engineer.",
    "input": "The 'Selected Seat' in our database doesn't match what the airline shows after a schedule change.",
    "output": "This is 'Equipment Inconsistency.' When the `AircraftType` changes in the schedule feed, the `SeatMap` API must be re-queried immediately. Our database is likely holding a 'Stale' seat index. You need to build a trigger: if `New_Equipment_Code != Old_Equipment_Code`, invalidate the `Seat_Selection` record and send a webhook to the user to choose a new seat based on the new configuration."
  },
  {
    "instruction": "Explain why a 'Confirmed Seat' changed to a traveler.",
    "input": "A traveler is disappointed because their 'Extra Legroom' seat was moved.",
    "output": "I am so sorry about that! The airline had to switch to a different 'Model' of plane for your flight today—think of it like the airline bringing a different car to the pick-up. Since the inside is shaped a little differently, they’ve moved everyone to the closest match possible. I’ll dive into the new map right now and see if I can find you an even better 'Front Row' spot!"
  },
  {
    "instruction": "Explain Currency Fluctuations (ROE) to a junior agent.",
    "input": "An agent quoted a price in Euros this morning, but the system shows a different price this afternoon.",
    "output": "That's the 'ROE' (Rate of Exchange). Every day, IATA publishes a new 'official' rate for travel math. Even if the airline didn't change the price, the 'translation' into Euros did. It’s like the price of gold—it breathes. Always tell your client that quotes are 'subject to change' until the moment the ticket is issued and the rate is 'frozen'."
  },
  {
    "instruction": "Explain IATA ROE synchronization to a data engineer.",
    "input": "Our multi-currency pricing is off by a few cents compared to the GDS.",
    "output": "You are likely using a standard 'Forex' API. Airlines don't use market rates; they use the 'IATA 5-decimal ROE.' You must ingest the official IATA rate file (published daily) and use it for all `BaseFare -> LocalCurrency` conversions. If your math is off by even $0.01, the GDS will reject the 'Sell' command because the 'Calculated Total' won't match the 'Filed Fare'."
  },
  {
    "instruction": "Explain why a 'Saved Quote' changed price to a traveler.",
    "input": "A traveler is confused why the price for a flight in Japan changed even though they are paying in Dollars.",
    "output": "It’s a bit of 'International Math!' Because that flight is originally priced in Yen, our system has to 'translate' it into Dollars for you. Since global exchange rates shift slightly every day, the 'translation' changes too. It’s like the price of international stamps—the value of the 'connection' stays the same, but the cost to buy it in your local currency can shift a tiny bit with the world market!"
  },
  {
    "instruction": "Explain Ghost PNRs to a junior agent.",
    "input": "The airline says the passenger is 'Confirmed,' but the agent can't find the PNR in their GDS.",
    "output": "You’ve got a 'Ghost PNR.' This happens when the 'Bridge' between the airline and the GDS breaks. The airline 'thinks' they sent the data, but the GDS never caught it. It’s like a text message that says 'Sent' on one phone but never 'Arrived' on the other. You’ll need to call the airline's 'Help Desk' to 'Resync' or 'Push' the record back into your view."
  },
  {
    "instruction": "Explain 'Sync Latency' and PNR fragmentation to a data engineer.",
    "input": "We have 'Orphaned' bookings where we have a payment but no GDS record.",
    "output": "This is a 'Handshake Failure.' The payment was processed, but the `Commit` command to the GDS timed out. The airline's internal CRS might have created the record, but our system never received the 6-character locator. You need to implement a 'Reconciliation Worker' that queries the airline by `Flight/Name/Date` if a transaction is 'Pending' for more than 5 minutes to recover these 'Ghost' locators."
  },
  {
    "instruction": "Explain why a booking 'Disappeared' to a traveler.",
    "input": "A traveler is panicked because they can't see their trip on the agency's website anymore.",
    "output": "Don't panic! Your seat is safe. Sometimes the 'Digital Bridge' between the airline and our office takes a tiny 'coffee break.' While the data is refreshing, it might hide for a moment. I can see you’re 100% confirmed in the airline's master vault. I’ll give the system a quick 'nudge' so it shows up on your screen again in just a second!"
  },
  {
    "instruction": "Explain SSR vs. OSI to a junior agent.",
    "input": "The agent isn't sure whether to put a 'Meal Request' in SSR or OSI.",
    "output": "SSR (Special Service Request) is for things that need an 'Answer'—like a Vegan Meal or a Wheelchair. The airline *must* acknowledge it (status `KK`). OSI (Other Service Information) is just 'For Your Information'—like 'VIP Traveler' or 'Honey-mooners.' Think of SSR as a 'Request' that needs a 'Yes/No,' and OSI as a 'Sticky Note' that the crew just reads."
  },
  {
    "instruction": "Explain 'Reply Logic' for SSR codes to a data engineer.",
    "input": "The 'Special Requests' in our app always show as 'Pending'.",
    "output": "You are likely only looking at the `SSR` string and not the `Status_Code`. When we send an SSR, the airline sends back a status. `PN` means 'Pending,' but `KK` or `HK` means 'Confirmed.' Your UI needs to listen for these status updates. If you see `UN` or `NO`, the airline has rejected the request, and you must alert the user that their specific meal or service is unavailable."
  },
  {
    "instruction": "Explain 'Special Requests' to a traveler.",
    "input": "A traveler asks how the airline knows they need a 'Low Salt' meal.",
    "output": "We send a 'Digital Request' (called an SSR) directly to the airline's catering team. It’s like sending a direct note to the chef! The airline then sends us a 'thumbs up' to confirm they've got it on their list. We keep a close eye on that 'thumbs up' to make sure everything is perfectly prepared for you before you even step on the plane!"
  },
  {
    "instruction": "Explain 'Ticketing in the Void Window' to a junior agent.",
    "input": "An agent made a mistake and issued a ticket, but it's only been 10 minutes.",
    "output": "You're in the 'Grace Period'! Most tickets can be 'Voided' without a penalty if you do it on the *same day* before the airline's 'Settlement' starts. It’s like a 'Cancel' button on an email—if you're fast enough, you can take it back as if it never happened. Just don't wait until tomorrow, or the 'Void Window' will slam shut and it becomes a 'Refund' with fees!"
  },
  {
    "instruction": "Explain 'Void vs Refund' accounting logic to a data engineer.",
    "input": "The finance system is showing 'Duplicate' transactions for a voided ticket.",
    "output": "A 'Void' should result in a 'Net Zero' in the General Ledger. Unlike a refund (which is a new transaction), a void 'cancels' the original authorization. Your ETL (Extract, Transform, Load) process needs to check the `Ticket_Status` field. If status = `VOID`, you should suppress the original `SALE` record from the revenue report entirely, rather than showing a 'Sale' and a 'Refund,' as no money actually changed hands with the airline."
  },
  {
    "instruction": "Explain a 'Quick Fix' for a ticket error to a traveler.",
    "input": "A traveler realized they gave the wrong date 5 minutes after they paid.",
    "output": "You caught it just in time! Because we’re in the 'Instant Correction' window, I can 'Void' that mistake and issue the correct one immediately without any fees. It’s like having a 'Undo' button for your booking. I’ll have the correct ticket in your inbox in just a moment—it’s as if the first one never happened!"
  },
  {
    "instruction": "Explain Coupon Status Mismatches to a junior agent.",
    "input": "The traveler says they flew the flight, but the GDS still says 'Open' for that leg.",
    "output": "That's a 'Coupon Lag.' When the passenger boards, the airport 'flips a switch' to change the status to 'Used' or 'Flown.' If their computer was slow, the GDS still thinks the seat is 'Open' (available to fly). It’s like a library book that you returned, but the librarian hasn't scanned it yet. Don't try to refund it until the status settles, or the airline will think you're 'Double Dipping'!"
  },
  {
    "instruction": "Explain DCS-to-GDS status synchronization to a data engineer.",
    "input": "Our 'Flown Revenue' reports are inaccurate because of 'Open' status tickets.",
    "output": "You are seeing 'DCS Latency.' The Departure Control System (at the airport) usually batches 'Flown' updates to the GDS. You should add a 48-hour 'Cooling Period' to your revenue recognition logic. If a ticket is still `OPEN` 48 hours after the `Departure_Timestamp`, you should trigger an 'Inventory Audit' via the airline's API to verify the actual `Final_Status` before closing the books."
  },
  {
    "instruction": "Explain why a flight 'Still Looks Booked' to a traveler.",
    "input": "A traveler is worried because their app says 'Open' for a flight they already took.",
    "output": "No need to worry—that's just the system 'catching its breath!' Now that you've landed, the airport's computer is sending a 'mission accomplished' note to our system. It can sometimes take a day or two for that note to travel across the globe and update your app. You’ve done the flying; now the computers are just doing the paperwork!"
  },
  {
    "instruction": "Explain DCS Handover to a junior agent.",
    "input": "The agent is trying to change a seat 2 hours before the flight, but the GDS says 'Restricted'.",
    "output": "The airline has 'Taken Control' of the flight—this is the 'DCS Handover.' About 3 to 24 hours before take-off, the airline moves the data from the 'Global Supermarket' (GDS) to their 'Airport Control' (DCS). We can't touch the booking anymore; only the 'Gate Agent' at the airport has the keys now. It’s like a relay race—the GDS just handed the baton to the airport!"
  },
  {
    "instruction": "Explain 'Control' flags in PNR data to a data engineer.",
    "input": "Our API is returning 'NOT AUTHORIZED' for seat changes on flights departing today.",
    "output": "Check the `Control_Entity` flag in the PNR. If the value is `ARL` (Airline) instead of `AGT` (Agent), the DCS has taken control. Your UI should detect this flag and change the 'Edit' button to a 'View Only' mode, with a message advising the user to 'See Agent at Airport.' Trying to push changes during the 'Handover' period will always result in a `403 Forbidden` error."
  },
  {
    "instruction": "Explain why the agent 'Can't Help' right before a flight to a traveler.",
    "input": "A traveler is calling from the taxi to the airport wanting to change their seat.",
    "output": "I would love to help, but your flight has already been 'Handed Over' to the airport team! Think of it like a plane that's already on the runway—our office has 'signed off,' and the gate agents at the airport are now the ones in charge of the seating chart. The moment you walk into the terminal, just check in with the desk—they have the master list and can help you right there!"
  },
  {
    "instruction": "Explain Late-Show Reconciliation to a junior agent.",
    "input": "A traveler checked in online but never showed up at the gate.",
    "output": "That's a 'Late-Show' or 'No-Show' after check-in. The airline will mark the seat as 'Empty' at the very last second. Even though they checked in, the 'Ticket Coupon' will be changed to `NS` (No-Show). This usually means the ticket is 'Forfeited' (lost). It’s like checking in for a movie but never sitting in the seat—you can't ask for your money back after the film has ended!"
  },
  {
    "instruction": "Explain 'No-Show' data impact on 'Flown' revenue to a data engineer.",
    "input": "Our 'Passenger Counts' from the airline don't match our 'Ticket Sales'.",
    "output": "You need to distinguish between 'Issued' and 'Boarded'. A passenger who checks in (Status `CI`) but doesn't board (Final Status `NS`) represents 'Unearned Revenue' that the airline typically keeps. You should capture the `Final_Coupon_Status`. If it's `NS`, you should attribute that revenue to a 'Penalty/Forfeiture' bucket rather than 'Operational Flown Revenue,' as no actual service was rendered."
  },
  {
    "instruction": "Explain what happens if you 'Miss the Gate' to a traveler.",
    "input": "A traveler is upset because they checked in on their phone but were 'Left Behind' when they were late to the gate.",
    "output": "I am so sorry! I know how stressful that is. Because planes have a very strict 'Departure Slot,' they have to close the doors at a specific time, even for passengers who checked in online. Once that door closes, the system marks the seat as 'Empty' so the plane can balance its weight for take-off. We’ll jump in right now and see if we can 'rescue' any value from your ticket for a new flight!"
  },
  {
    "instruction": "Explain Interlining Baggage across LCCs to a junior agent.",
    "input": "An agent is booking a trip with a Low-Cost Carrier (LCC) connecting to a major airline.",
    "output": "Watch out for the 'Baggage Gap'! LCCs usually don't have 'Interline Agreements.' This means they won't 'talk' to the next airline. The traveler will have to pick up their bags at the 'Transfer' point, walk out, and 'Re-check' them. It’s not a 'Through-Journey' for the bags—it’s two separate trips. Make sure the traveler knows they need extra time for the 'Bag Run'!"
  },
  {
    "instruction": "Explain 'Baggage Continuity' validation to a data engineer.",
    "input": "Travelers are complaining that their bags didn't follow them on multi-carrier trips.",
    "output": "You need an 'Interline Validator.' When a user books a multi-carrier itinerary, your system should check the 'Bilateral Baggage Agreement' table. If Carrier A (e.g., Ryanair) and Carrier B (e.g., Lufthansa) have no agreement, you must trigger a UI warning: 'Manual Baggage Transfer Required.' Without this data-driven warning, users assume 'Through-Checking' and will miss their bags at the final destination."
  },
  {
    "instruction": "Explain 'Self-Transfer' of bags to a traveler.",
    "input": "A traveler is confused why they have to 'collect their bags' during a connection.",
    "output": "Since these two airlines are from 'different families' (they don't have a baggage agreement), they don't 'share' a luggage belt. Think of it like a 'self-service' connection—you’ll have a quick 're-union' with your bags at the halfway point, and then you’ll take them over to the next desk. We’ve given you a nice long layover to make sure this 'baggage hand-off' is easy and stress-free!"
  },
  {
    "instruction": "Explain Master Folio vs. Personal Folio to a junior agent.",
    "input": "A business traveler wants to pay for their room with the company card but their drinks with their own card.",
    "output": "You need to 'Split the Folio.' Move the 'Room and Tax' to the 'Master Folio' (for the company) and the 'Incidental' charges to the 'Personal Folio'. It’s like having two separate 'shopping carts' at the same store. If you mix them up, the company will see those poolside cocktails, and the traveler will be stuck explaining them to their boss!"
  },
  {
    "instruction": "Explain Folio 'Partitioning' logic to a data engineer.",
    "input": "Our expense integration is pulling 'Personal' charges into 'Corporate' reports.",
    "output": "The PMS integration is likely sending a single `Total_Amount` for the stay. You need to parse the 'Line Items' using `Charge_Category` codes. Filter for `RM` (Room) and `TX` (Tax) for the corporate feed, and categorize `FB` (Food/Bev) or `OT` (Other) as 'Incidental'. If your API doesn't support 'Split Folio' identifiers, you’ll need to build a 'Rules Engine' that masks specific categories based on the client's expense policy."
  },
  {
    "instruction": "Explain the 'Two Bills' system to a traveler.",
    "input": "A business traveler is worried about their personal expenses showing up on their company receipt.",
    "output": "We’ve got you covered! We’ve set up 'Two Digital Pockets' for your stay. Your room and travel costs go into the 'Company Pocket,' and any personal treats you enjoy go into your own 'Private Pocket.' When you check out, you’ll get two separate, clean receipts—one for your office and one just for you. It keeps your business and pleasure perfectly separate!"
  },
  {
    "instruction": "Explain Rounding Errors in Tax Calc to a junior agent.",
    "input": "The agent's total is $500.01, but the client's credit card was only authorized for $500.00.",
    "output": "That's a 'Penny Error.' When the system adds up multiple taxes (US, ZP, XF), it sometimes rounds up. If the 'Authorization' and the 'Final Price' are off by even one cent, the whole 'Ticketing Handshake' will fail. You’ll need to 'Re-Authorize' for the exact, rounded amount. It’s like a vending machine that won't give you the snack because you're one penny short—the computer is that literal!"
  },
  {
    "instruction": "Explain 'Floating Point' rounding issues in tax engines to a data engineer.",
    "input": "We are getting 'Price Mismatch' errors from the GDS for international tickets.",
    "output": "You are likely experiencing 'Cumulative Rounding' errors. If you round each tax component *before* summing them, you’ll get a different result than if you sum them and *then* round the total. GDS providers usually expect 'Component-Level Rounding'. You must match the 'IATA Tax Precision' for each specific country code (e.g., some round to the nearest whole unit, some to two decimals) to ensure your `Total_Tax` string matches their expectation exactly."
  },
  {
    "instruction": "Explain a 'One Cent' delay to a traveler.",
    "input": "A traveler is annoyed that they have to 're-approve' their payment for a one-cent difference.",
    "output": "I know it seems silly! Because international travel involves so many different 'mini-taxes' from around the world, the system's 'calculator' sometimes lands on a slightly different penny at the very last second. To keep everything 100% legal and official, the system just needs a quick 'thumbs up' from you for that new total. It’s just our way of making sure your ticket is perfectly balanced and ready to fly!"
  },
  {
    "instruction": "Explain Multi-Currency Settlement to a junior agent.",
    "input": "The agent is booking in London (GBP) but the flight is a domestic flight in Japan (JPY).",
    "output": "This is 'Cross-Border Settlement.' The airline wants JPY, but we are paying in GBP. The GDS will use a 'Conversion Hub' (the ROE) to bridge the gap. Be careful—if the traveler cancels, they might get back a different amount of GBP than they paid because the 'Exchange Bridge' moved in the meantime. It’s like buying a souvenir abroad and returning it later—the 'value' is the same, but the 'cash' might be different!"
  },
  {
    "instruction": "Explain 'Settlement Currency' drift to a data engineer.",
    "input": "Our 'Profit Margin' reports are showing 'Losses' due to exchange rates.",
    "output": "You are seeing 'Settlement Drift.' We sell in `Currency_A` but the BSP settles in `Currency_B`. You need to capture the 'FX Rate at Sale' AND the 'FX Rate at Settlement'. Your profit calculation should be: `(Sale_Amount * Rate_A) - (Settlement_Amount * Rate_B)`. If you don't account for the 'Spread' between these two dates, your margin data will be 'noise' rather than 'fact'."
  },
  {
    "instruction": "Explain why a refund might be 'Slightly Different' to a traveler.",
    "input": "A traveler is upset because they received $10 less on their refund than they expected due to currency changes.",
    "output": "I totally understand that frustration! Because your original flight was priced in a different currency, the 'Exchange Bridge' shifted a tiny bit between the day you booked and the day of the refund. It’s like returning an item you bought while on vacation—you get back the full 'local' price, but when it’s translated back into your home currency, the world market might have shifted a few dollars. It’s the one part of global travel that’s always in motion!"
  },
  {
    "instruction": "Explain No-Rec (No Record) Passengers to a junior agent.",
    "input": "A traveler is at the desk, they have a ticket, but the agent says they aren't on the 'Flight List'.",
    "output": "That's a 'No-Rec'! It’s a 'Data Disconnect.' The 'Ticket' exists, but it never 'Talked' to the 'Flight List' (the DCS). It’s like having a ticket to a concert but your name isn't on the 'Will Call' list. You need to 'Force' a resync by entering the 'Ticket Number' manually into the DCS to 'prove' the passenger is allowed on board. It’s a manual 'Digital Handshake'!"
  },
  {
    "instruction": "Explain 'E-Ticket to PNR' linkage failures to a data engineer.",
    "input": "Airlines are reporting 'Security Gaps' where passengers have tickets but no APIS data.",
    "output": "This is a 'Linkage Failure.' The `TKT` segment exists in the GDS, but the `TKNE` (Ticket Number Exchange) message never reached the airline's internal host. You need to build a 'Linkage Audit': 24 hours before flight, query the airline's 'Native PNR'. If the `TKT` field is empty despite being 'Issued' in our DB, you must re-transmit the `TKNE` command to ensure the passenger appears on the airport's 'Manifest'."
  },
  {
    "instruction": "Explain a 'Desk Delay' to a traveler.",
    "input": "A traveler is panicked because the check-in agent 'Can't find them' in the system.",
    "output": "Stay calm—this is just a 'Digital Handshake' delay! You have your official ticket number right there, which is your 'Golden Key.' Sometimes the airport's list just needs a quick 'manual update' to see your reservation. Show them your ticket number, and they’ll be able to pull you right into the flight list. It’s like showing your ID at a party when you’re not on the list—once they see it, you’re in!"
  },
  {
    "instruction": "Explain Infant-in-Lap Logic to a junior agent.",
    "input": "A traveler is flying with a baby, and the baby turns 2 years old during the trip.",
    "output": "This is a 'Birthday Trap'! For the outbound flight, the baby is an 'Infant' (free/cheap on lap). But for the return, they are now a 'Child' and *must* have their own seat by law. The GDS will reject a 'Lap' status for the return leg. You have to book the whole trip as a 'Child' or book two separate one-ways. If you ignore it, the airline will 'Stop' them at the gate on the way home!"
  },
  {
    "instruction": "Explain age-validation logic for multi-leg trips to a data engineer.",
    "input": "Our booking engine is allowing 'Infant' fares for children who turn 2 mid-trip.",
    "output": "Your age validation is only checking the `Departure_Date`. You must implement a 'Whole-Trip Age Check.' The logic should be: `IF (Return_Date - Birth_Date) >= 2 years THEN Passenger_Type = CHILD`. If you only check the start date, the return leg will fail at the 'Pricing' or 'Ticketing' stage, or worse, during 'Airport Check-In,' leading to massive last-minute fare differences."
  },
  {
    "instruction": "Explain a 'Birthday Upgrade' to a traveler.",
    "input": "A parent is surprised they have to buy a full seat for their toddler's return flight.",
    "output": "It’s a big milestone! Because your little one turns two during your adventure, they officially 'graduate' to having their very own seat for the flight home. Safety rules ask that every 'big kid' (age 2 and up) has their own space and seatbelt. Think of it as their very first 'grown-up' travel perk—more room for them to wiggle and a much more comfortable ride for you too!"
  },
  {
    "instruction": "Explain Group Name List Failures to a junior agent.",
    "input": "The 'Group Leader' cancelled, and now all 20 people in the group are 'Gone' from the GDS.",
    "output": "That's a 'Master PNR Collapse.' In a group booking, everyone is 'Tied' to the leader. If you cancel the leader without 'Splitting' the others first, the system thinks the 'whole party' is over. You must always 'Divide' or 'Split' the people you want to keep before you touch the 'Cancel' button on anyone else. It’s like a 'Digital Chain'—don't break the first link!"
  },
  {
    "instruction": "Explain 'PNR Splitting' logic and data integrity to a data engineer.",
    "input": "We are losing 'Sub-PNRs' in our database when a group reservation is modified.",
    "output": "When a GDS 'Split' occurs (Command `SP`), the original PNR locator remains, but a *new* 6-character locator is created for the split-off passengers. Your system must listen for the `Split_Event` and immediately map the new locator to the relevant `Passenger_IDs`. If you only track the original 'Master' locator, the split passengers will 'disappear' from our active tracking and reporting."
  },
  {
    "instruction": "Explain why a 'Group Change' took a few minutes to a traveler.",
    "input": "A traveler in a group is worried because their friends are confirmed but they are 'Waiting'.",
    "output": "Don't worry! Because you're traveling as a 'Team,' the system just needs a moment to 'un-clip' your reservation from the main group so we can make your specific change. It’s like getting a 'separate check' at a big dinner—it takes a second for the waiter to organize, but it ensures your own plans are perfectly settled and independent!"
  },
  {
    "instruction": "Explain Waitlist Segment Churn to a junior agent.",
    "input": "The agent is 're-requesting' a waitlist every hour to 'help it clear'.",
    "output": "Stop! That’s 'Segment Churn.' Every time you re-request, you're actually 'Resetting your spot' in line. The airline's robot sees this as 'Gaming the System' and might block you entirely. It’s like getting out of line to ask the clerk 'How much longer?'—you just end up at the very back of the line. Just 'Hold' and let the priority ladder do its work!"
  },
  {
    "instruction": "Explain technical load of 'Waitlist Polling' to a data engineer.",
    "input": "The GDS is charging us 'Inquiry Fees' for excessive PNR refreshes.",
    "output": "Our 'Waitlist Monitor' is polling too aggressively. You should implement 'Exponential Backoff' for waitlist status checks. Instead of polling every hour, poll at `T+1`, `T+4`, `T+12`, and then once every 24 hours. The probability of a status change (from `HL` to `HK`) increases as the `DepartureDate` approaches, so you should only increase frequency in the final 72 hours to save on API costs and avoid 'Churn' flags."
  },
  {
    "instruction": "Explain 'Patience' on a waitlist to a traveler.",
    "input": "A traveler is anxious and wants the agent to 'Do something' to speed up their upgrade.",
    "output": "I know the wait is the hardest part! Right now, your 'spot' is perfectly secured in the airline's priority line. The best thing we can do is stay exactly where we are—every time we 'check' too officially, it's like tapping the airline on the shoulder, and they prefer us to just wait our turn. We’re at the 'front of the pack' for your status level, so let's let the system work its magic!"
  },
  {
    "instruction": "Explain Inflight Wi-Fi Tokenization to a junior agent.",
    "input": "A corporate traveler wants to know why they can't 'Pre-pay' for Wi-Fi on all their flights.",
    "output": "Inflight Wi-Fi is like a 'Local Shop' on the plane. Most airlines use third-party providers (like Gogo or Panasonic). Our 'Central System' doesn't always have a 'key' to their 'Cash Register'. Often, the traveler has to buy it 'on the fly' (literally!). We can sometimes buy 'Wi-Fi Tokens' beforehand, but they are like 'Coupons'—you have to enter the code once you're in the air."
  },
  {
    "instruction": "Explain Wi-Fi 'Third-Party' API integration to a data engineer.",
    "input": "We want to bundle 'Wi-Fi' into our 'All-Inclusive' corporate rates.",
    "output": "This is a 'Service Fragmentation' challenge. Most Wi-Fi providers don't sync with the GDS `EMD` system. You’ll need to integrate with the provider's own 'Voucher API' to purchase a 'Token'. Your database must store this token and your mobile app must 'Push' it to the user's lock screen upon `Take-off_Event`. If the provider's API is down, you must have a 'Refund Trigger' if the `Inflight_Login` never occurs, to prevent 'Ghost Charges'."
  },
  {
    "instruction": "Explain 'Buying Internet' in the sky to a traveler.",
    "input": "A traveler wants to know if their Wi-Fi is 'included' since they are in Business Class.",
    "output": "It’s a great question! For your specific flight, the Wi-Fi is provided by a 'Partner' in the sky. If it’s not already part of your seat, you can easily grab a 'Pass' the moment you reach 10,000 feet. It’s like a 'digital toll booth' for the clouds—you just log in with your seat number, and you’re connected! I’ll check to see if your airline has a 'Pre-paid Voucher' we can grab for you now!"
  },
  {
    "instruction": "Explain Hotel 'Walk' Logistics to a junior agent.",
    "input": "The hotel is overbooked and is 'Walking' the guest to a different hotel.",
    "output": "A 'Walk' is the hotel version of a 'Bump.' The hotel *must* pay for the new room and the taxi to get there. In our system, the 'Booking' is still at the first hotel, but the 'Guest' is at the second. You need to add an 'OSI' note to the PNR so we know where they *actually* slept, or the emergency 'Duty of Care' tracker will look for them in the wrong place!"
  },
  {
    "instruction": "Explain 'Hotel Relocation' data handling to a data engineer.",
    "input": "Our 'Traveler Tracker' map is showing guests at the 'Original' hotel instead of the 'Relocated' one.",
    "output": "You need to build a listener for 'Hotel Status' updates (specifically the `RELOCATED` or `WALKED` status). When a hotel sends a 'Relocation' notice via the GDS or Switch, your system should automatically create a 'Shadow Segment' in the database for the new property. This ensures the 'Active Location' on the map and the 'Duty of Care' alerts are pinned to the *actual* physical address of the traveler."
  },
  {
    "instruction": "Explain being 'Walked' to a traveler.",
    "input": "A traveler is very upset because their hotel says they don't have a room for them.",
    "output": "I am so sorry! This is what the industry calls a 'Hotel Walk.' Think of it as an 'Instant Upgrade' elsewhere. The hotel is taking full responsibility—they are moving you to a nearby property (often a nicer one!) and covering the full cost of your stay and your travel there today. I’ll stay on the line until you’re comfortably settled in your new 'VIP' room. We’ll make sure you’re treated like a total priority!"
  },
  {
    "instruction": "Explain City Code Overlap to a junior agent.",
    "input": "An agent booked a flight to 'London' (LON) but the traveler wanted 'Heathrow' (LHR).",
    "output": "Watch your codes! `LON` is a 'City Code' (it includes Heathrow, Gatwick, City, Stansted). `LHR` is a 'Specific Airport.' If you book `LON`, the computer picks the cheapest one—which might be Gatwick when they need Heathrow for a connection! It’s like asking a taxi to take you to 'New York'—you might end up in Brooklyn when you needed Manhattan!"
  },
  {
    "instruction": "Explain 'Multi-Airport' search normalization to a data engineer.",
    "input": "Our 'Shortest Flight' filter is suggesting routes with 4-hour 'Cross-Town' transfers.",
    "output": "Your algorithm is treating `LON` as a single point. You need to implement a 'Transfer Penalty' for cross-airport connections. If `Arrival_Airport != Departure_Airport` (e.g., LHR to LGW), your logic must add a 'Ground Transfer Time' (e.g., +180 mins) and a 'Transfer Cost' to the total. Without this 'Spatial Awareness,' your search results will be 'technically possible' but 'humanly impossible'."
  },
  {
    "instruction": "Explain why there are 'Multiple Airports' to a traveler.",
    "input": "A traveler is confused why their flight says 'LON' but their friend's says 'LHR'.",
    "output": "It’s a bit of 'City Shorthand!' `LON` is the master code for all the big airports serving London. Think of it as the 'Front Door' to the city. Your specific ticket will show which 'Room' (like Heathrow or Gatwick) you’re actually arriving in. I’ll make sure your flight lands exactly where you need to be for your easy transfer into the heart of the city!"
  },
  {
    "instruction": "Explain Cross-Border Fare Arbitrage to a junior agent.",
    "input": "A traveler wants to book a flight starting in 'Mexico' to save money on a trip to Europe.",
    "output": "That's 'Fare Arbitrage.' Airlines price flights based on where they *start*. Sometimes starting in a different country is cheaper, even if you add a flight to get there! But be careful—if you don't take that first flight from Mexico, the airline will cancel the 'cheap' trip to Europe. It’s like a 'Coupon' that only works if you enter through the 'Front Door' of the store!"
  },
  {
    "instruction": "Explain 'Point-of-Origin' fraud detection to a data engineer.",
    "input": "We are getting 'Security Alerts' from airlines for 'Suspicious Routing'.",
    "output": "Airlines use 'Origin Spoofing' detectors. If a user with a 'US IP' and 'US Credit Card' is consistently booking 'Nested' trips starting in 'Lower-Yield' markets (like Egypt or Mexico), the Revenue Integrity robot flags it as 'Arbitrage.' You should implement a 'Market Consistency' check: if `Origin_Country != Billing_Country`, add a 'Manual Audit' flag to ensure the traveler actually intends to travel to the origin, rather than just 'gaming' the pricing engine."
  },
  {
    "instruction": "Explain why the 'Starting Point' matters to a traveler.",
    "input": "A traveler asks why a flight from London to New York is more expensive than one from Dublin to New York with a stop in London.",
    "output": "It’s a quirk of the 'Global Travel Market!' Airlines set their prices based on where the journey begins to stay competitive in that specific city. Think of it like a 'Local Sale'—the airline in Dublin is having a big 'Grand Opening' sale to attract travelers, even if the flight stops in London along the way. It’s a great way to save, as long as you're happy to start your adventure in Dublin!"
  },
  {
    "instruction": "Explain API Latency and Caching to a junior agent.",
    "input": "The price the agent saw 30 seconds ago is gone, and now it's $100 higher.",
    "output": "You just hit a 'Cache Wall.' To stay fast, websites often look at a 'Saved Snapshot' of prices from a few minutes ago. When you click 'Book,' the system 'Calls' the airline for the *real* live price. It’s like a 'Sale' sign that someone forgot to take down—it looks like a deal until you get to the register. Always tell the client: 'It’s a snapshot, let's lock it in now!'"
  },
  {
    "instruction": "Explain 'Stale Data' thresholds to a data engineer.",
    "input": "Users are complaining that our 'Search Prices' don't match our 'Booking Prices'.",
    "output": "Your 'Cache TTL' (Time-To-Live) is too long. For 'High-Volatility' routes (like LCCs), you should reduce cache TTL to < 2 minutes. Additionally, you should implement a 'Pre-Flight Check': when a user moves from 'Search' to 'Details,' trigger a background 'Live Re-Price' API call. If the price difference is > 1%, update the UI immediately *before* they enter their credit card to maintain trust and transparency."
  },
  {
    "instruction": "Explain why the 'Price Jumped' to a traveler.",
    "input": "A traveler is upset because the '$400' flight they saw turned into '$500' when they clicked 'Buy'.",
    "output": "I am so sorry! That is incredibly frustrating. Think of it like a 'Flash Sale' at a very busy store—sometimes the very last 'discount' seat is grabbed by another traveler just as we’re reaching for it. Our 'Search' shows a quick preview, but when we go to 'Check Out,' the airline gives us the absolute latest, live second-by-second price. Let me see if I can find another 'hidden deal' to get that price back down for you!"
  },
  {
    "instruction": "Explain OTA Scraper Detection to a junior agent.",
    "input": "The agent's 'Quick Search' tool is blocked, and they are seeing a 'Verify You are Human' screen.",
    "output": "The airline's 'Security Robot' thinks you're a 'Bot'! If we search too fast or use 'unauthorized' tools to scrape prices, the airline blocks our 'Digital Address' (IP). They want to protect their servers from 'Data Thieves.' Always use the official GDS or our approved portal—it’s like having an 'Official ID Badge' that lets the security robot know you’re one of the 'Good Guys'!"
  },
  {
    "instruction": "Explain 'Rate Limiting' and bot detection to a data engineer.",
    "input": "Our 'Aggregator' API is getting `429 Too Many Requests` errors from our partners.",
    "output": "We are hitting 'Scraper Detection' thresholds. You need to implement 'Request Throttling' and 'User-Agent' randomization. More importantly, check if we are sending 'Headless' browser headers. Suppliers look for 'Fingerprints' (like missing CSS or JS execution). To avoid blocks, we should use a 'Circuit Breaker' pattern: if one IP gets a 429, pause all requests from that cluster and switch to a 'Back-off' strategy to prove we are a 'Friendly' high-volume partner, not a malicious scraper."
  },
  {
    "instruction": "Explain 'Security Checks' on the website to a traveler.",
    "input": "A traveler is annoyed that they have to click 'I am not a robot' three times.",
    "output": "I know it’s a bit of a hurdle! It’s actually a 'Security Shield' we use to protect you and the airlines. It stops 'Bad Robots' from swamping the system and grabbing all the best deals before real people like you can get them. It’s like having a 'Bouncer' at the door to make sure only real travelers get into the 'VIP Sale'! Thank you for helping us keep the system fair for everyone!"
  },
  {
    "instruction": "Explain NDC Servicing Limitations to a junior agent.",
    "input": "An agent booked a flight via NDC but can't change the date using standard GDS commands.",
    "output": "That's an 'NDC Wall.' Because NDC is the airline's 'Private Language,' our standard GDS tools can't always 'translate' a change. It’s like having a 'Limited Edition' item—you can't just return it at any store; you have to go back to the original 'Brand Boutique'. We have to use a special 'NDC Portal' or call the airline directly to touch that booking."
  },
  {
    "instruction": "Explain 'Servicing' gaps in the NDC JSON structure to a data engineer.",
    "input": "Our 'Change Flight' API is failing for all NDC-sourced bookings.",
    "output": "The 'OrderChange' and 'OrderReshop' methods are not yet standardized across all NDC carriers. Some require a `Change_Reason` code, while others only support 'Full Cancellation and Re-book.' You need to build a 'Servicing Capability' map: if `Source == NDC` AND `Carrier == BA`, use the `Post-Booking_API_v2`. If carrier-level servicing isn't supported via API, you must route the agent to a 'Manual Support' workflow to avoid corrupting the `Order_Object` in the airline's CRS."
  },
  {
    "instruction": "Explain why an 'Online Change' isn't working to a traveler.",
    "input": "A traveler is frustrated that they can't change their 'NDC' flight through the standard app.",
    "output": "I hear you! Because you’ve got a 'Special Direct' ticket (we call it NDC), it has its own 'Exclusive Lane' for changes. It’s a bit like having a VIP backstage pass—it gives you a great deal, but you have to go through a 'Special Desk' to make any edits. I’ll jump into that 'Private Lane' for you right now and get your dates changed manually—I’ve got the 'Secret Key' to get it done!"
  },
  {
    "instruction": "Explain Broken Itinerary Logic to a junior agent.",
    "input": "The outbound flight was cancelled, and now the GDS says the return flight is 'Inactive'.",
    "output": "This is 'Itinerary Collapse.' When the 'Start' of a trip fails, the system assumes the 'End' is also cancelled. It’s like a 'Domino Effect'—the first one fell, so the system knocked the rest down to save the seats for someone else. You have to 'Rescue' the return flight by telling the system: 'Wait! The traveler still wants to go, just on a different start!' You have to manually 'Protect' those segments."
  },
  {
    "instruction": "Explain 'Sequence Enforcement' logic in CRS to a data engineer.",
    "input": "Our app is showing 'Invalid' for return flights after an outbound cancellation.",
    "output": "This is 'Out-of-Sequence' logic. Most CRSs (airline host systems) will 'Auto-Cancel' subsequent segments if the first coupon isn't 'Lifted' (flown). You need to build an 'Irrops Listener.' If a `Flight_Cancellation_Event` occurs, your system must check for 'Associated Segments' in the PNR. You should automatically send a 'Keep Segments' (status `GK`) request to the airline to prevent the 'Auto-Purge' robot from wiping the rest of the traveler's journey."
  },
  {
    "instruction": "Explain why their 'Return Flight' is missing to a traveler.",
    "input": "A traveler is panicked because their return flight disappeared from their app after their flight to the city was cancelled.",
    "output": "I am so sorry for the shock! It’s just the airline's 'Auto-Clean' system being a bit too helpful. Because the first flight was changed, the system 'paused' the rest of your trip to wait for your new plans. Don't worry—your return seat is still 'reserved' in the master vault. I’m going in right now to 're-activate' it so it shows back up on your phone. Your way home is safe!"
  },
  {
    "instruction": "Explain Passport Change mid-journey to a junior agent.",
    "input": "A traveler lost their passport while abroad and has an 'Emergency Travel Doc' now.",
    "output": "This is a 'Security Update.' The old passport info in the PNR is now 'Dead.' You must update the `DOCS` and `APIS` segments with the new 'Emergency' document number. If the 'Key' (the doc number) doesn't match the 'ID' they show at the airport, they’ll be pulled aside for questioning. It’s like changing your house key—the locks at the gate need to know about the new one!"
  },
  {
    "instruction": "Explain 'APIS Resubmission' logic to a data engineer.",
    "input": "The 'Security Check' failed at the airport for a passenger with a new passport.",
    "output": "The DCS is still holding the 'Original' APIS submission. When a document is updated in the PNR mid-journey, you must trigger a 'Force Resubmit' to the government APIS gateway. This is often done via the `SSR DOCS` command with a 'Modify' flag. If you just 'Add' the new info without 'Deleting' the old, the system gets a 'Conflict' error and the passenger is flagged as 'High Risk' due to inconsistent ID data."
  },
  {
    "instruction": "Explain an 'Emergency Passport' update to a traveler.",
    "input": "A traveler is worried that their new 'Emergency Document' won't work for their flight home.",
    "output": "You've done the hard part getting that new document! Now, I’m doing the 'Digital Hard Part' for you. I’m going into the airline's security vault and 'Updating the Guest List' with your new number. It’s like giving the bouncer a 'heads up' that you’re coming with a different ID tonight. When you get to the airport, the system will be ready and waiting for you—no questions asked, just a 'welcome back'!"
  },
  {
    "instruction": "Explain Irrops Re-accommodation to a junior agent.",
    "input": "A major storm has grounded 50 flights, and the agent is overwhelmed by the re-booking list.",
    "output": "Think of Irrops (Irregular Operations) as 'Digital Triage.' When the system crashes due to weather, we can't help everyone at once. We prioritize the 'High-Risk' patients first—families with kids, unaccompanied minors, and those with tight connections. Use the GDS 'Re-accommodation' tool to look for 'Protection' segments. It’s like being a lighthouse keeper in a storm; your job is to guide the most vulnerable ships to harbor first!"
  },
  {
    "instruction": "Explain the data sync between DCS and PNR during mass cancellations to a data engineer.",
    "input": "The PNRs are showing 'Confirmed' but the airline's app says 'Cancelled'.",
    "output": "This is a 'Handshake Lag' during high-volume events. The Departure Control System (DCS) at the airport has cancelled the flight, but the message hasn't 'propagated' back to the GDS yet. You need to implement a 'High-Frequency Poller' for PNRs in active 'Irrops' status. If the DCS status = `FLNK` (Flown) or `CNCL` (Cancelled), but the GDS segment is still `HK`, your system must force a 'Status Update' to prevent the user from seeing stale, 'happy' data during a crisis."
  },
  {
    "instruction": "Comfort a traveler during a mass flight cancellation.",
    "input": "A traveler is stuck at the airport and feels like the airline 'doesn't care'.",
    "output": "I hear how frustrating this is, and I promise you’re not just a number on a screen. Right now, the airline's 'Rescue Teams' are working 24/7 to find a new path for you. Think of this as a temporary 'pause' while we find you a safer, smoother route. I’m personally monitoring the 'Seat Availability' every few minutes to grab the first opening that appears. We’re going to get you home!"
  },
  {
    "instruction": "Explain the 'Broken Connection' Bridge to a junior agent.",
    "input": "A traveler missed their flight because the security line was 2 hours long.",
    "output": "This is a 'Passenger-Fault' connection break. Since the airline didn't cause the delay, they aren't legally required to re-book for free. However, we use the 'Flat Tire Rule'—a supportive 'handshake' where we ask the airline for a 'favored' rate to get them on the next flight. It’s about building a bridge out of a bad situation even when the 'contract' doesn't demand it."
  },
  {
    "instruction": "Explain 'Time-to-Gate' predictive alerts to a data engineer.",
    "input": "We want to warn users if they are at risk of missing their flight due to airport congestion.",
    "output": "You need to merge 'Flight Schedule' data with 'Live TSA Wait Time' APIs and 'Airport Spatial' data. Calculate the `Buffer = (Boarding_Time - Current_Time) - (Security_Wait + Walk_Time_to_Gate)`. If `Buffer < 15 mins`, trigger an 'Urgent Action' alert. The 'Last Mile' data is the hardest—you’ll need a lookup table for 'Average Walk Time' per terminal to ensure the prediction is realistic for a traveler with bags."
  },
  {
    "instruction": "Explain why they have to pay for a new flight after missing a connection to a traveler.",
    "input": "A traveler is upset that they have to pay a 'Change Fee' because they were late to the airport.",
    "output": "I know it’s a tough break—security lines can be so unpredictable! Because the airline kept your seat ready and waiting until the very last second, the system treats it as a 'missed appointment.' However, I’ve reached out to their team and used our partnership to 'bridge the gap.' They’ve agreed to waive the highest part of the fee, so we can get you on the next flight for just a small 'service update' cost. Let's get you back on track!"
  },
  {
    "instruction": "Explain Denied Boarding Compensation (DBC) to a junior agent.",
    "input": "A flight is overbooked, and the airline is looking for 'Volunteers'.",
    "output": "DBC is like a 'Passenger Auction.' The airline is willing to 'buy' the seat back from the traveler. Your job is to help the traveler negotiate the best deal—vouchers, cash, or a first-class seat on the next flight. It’s a 'Win-Win': the airline avoids a fine, and the traveler gets a 'Travel Bonus.' Just make sure they get the offer in writing before they leave the gate!"
  },
  {
    "instruction": "Explain DBC 'Liability Tracking' to a data engineer.",
    "input": "The accounting team needs to track 'Compensation Vouchers' issued during Irrops.",
    "output": "Vouchers are often issued as 'EMD-S' (Stand-alone) documents. Your system must capture the `EMD_Number` and the `Reason_Code` (e.g., `VDB` for Voluntary Denied Boarding). Many vouchers have a 'Use-by' date. You should build a 'Voucher Vault' in the user's profile that tracks the balance and expiry. If the data isn't captured at the moment of 'issuance' in the DCS, it becomes a 'dark liability' that finance can't account for."
  },
  {
    "instruction": "Explain the benefit of 'Volunteering' to a traveler.",
    "input": "A traveler is asked to give up their seat for a $500 voucher and a flight 4 hours later.",
    "output": "This is what we call a 'Travel Jackpot!' If you aren't in a rush, the airline is essentially offering to 'pay' for your next vacation just for 4 hours of your time. You’ll still get to your destination today, but you’ll have an extra $500 in your pocket for your next adventure. It’s like the airline is saying 'thank you' with a huge gift—I’d say it’s a fantastic win for you!"
  },
  {
    "instruction": "Explain Ramadan Etiquette in the Middle East to a junior agent.",
    "input": "An agent is booking a trip to Dubai during Ramadan.",
    "output": "Ramadan is a 'Holy Pause' in the region. During daylight hours, many restaurants will be closed or hidden behind screens, and the 'vibe' is much quieter. You should advise your traveler to avoid eating or drinking in public out of respect. It’s like being a guest in someone’s home while they are resting—you just speak a little softer and wait for the sunset 'Iftar' celebration to see the city come alive!"
  },
  {
    "instruction": "Explain 'Holiday Calendar' logic for regional amenities to a data engineer.",
    "input": "Our app is showing 'Open' for hotel restaurants that are actually closed for Ramadan.",
    "output": "You need a 'Locale-Aware Operating Hours' microservice. Standard 'Static' hours are insufficient. You need to ingest a 'Religious/Cultural Calendar' and apply 'Seasonal Overrides' to the `OpeningHours` field in your database. For Middle Eastern cities during Ramadan, your API should automatically tag restaurant results with a 'Limited Service' warning and adjust the 'Closing Time' to reflect the late-night Iftar shifts."
  },
  {
    "instruction": "Give cultural advice to a traveler visiting the Middle East.",
    "input": "A traveler is worried they'll 'do something wrong' during their visit to Doha.",
    "output": "It’s wonderful that you’re being so thoughtful! The most important thing to remember is that the local culture prizes 'Modesty and Respect.' Think of it as 'Dressing for a special occasion'—keeping shoulders and knees covered is a great way to show you value their traditions. During the day, it's a quiet time of reflection, but after sunset, the hospitality is legendary! Just follow the local lead, and you’ll have a deeply rewarding experience."
  },
  {
    "instruction": "Explain East Asian Etiquette to a junior agent.",
    "input": "A business group is traveling to Tokyo for a high-level meeting.",
    "output": "In Japan, 'The Group is One.' Advise them on the 'Meishi' (Business Card) exchange—it’s like a 'Mini-Ceremony.' You give and receive cards with both hands and never just slide them into a pocket. Also, seating is about 'Seniority.' The 'power seat' is the one furthest from the door. Think of it as a 'Dance of Respect' where every move shows you value your host's time and status."
  },
  {
    "instruction": "Explain 'Seniority-Based' seating algorithms to a data engineer.",
    "input": "Our corporate tool needs to auto-assign seats based on company hierarchy.",
    "output": "You need to map the 'Job Level' from the HR feed to the `Seat_Map` priority. In many cultures, the 'CEO' must be in the most 'Auspicious' seat (often 1A or the front-right of a car). Your algorithm should perform a 'Hierarchy Sort' on the passenger list before calling the `Seat_Assign` API. If a junior staffer is accidentally placed in a 'Superior' seat than their manager, it can cause real-world social friction for the client."
  },
  {
    "instruction": "Help a business traveler prepare for a meeting in China.",
    "input": "A traveler is nervous about their first big meeting in Beijing.",
    "output": "You’re going to do great! One quick 'pro-tip': in China, 'Punctuality' is the ultimate sign of respect—it’s better to be 10 minutes early than 1 minute late. Also, during dinner, let the host take the first bite. Think of it as a 'Welcome Ceremony' where your host leads the way. By showing this simple respect for their 'lead,' you’ll build a massive amount of 'Guanxi' (trust) before you even start talking business!"
  },
  {
    "instruction": "Explain High-Touch VIP Management to a junior agent.",
    "input": "A celebrity is flying through Heathrow and needs maximum privacy.",
    "output": "For VIPs, we use 'The Invisible Hand.' We arrange 'Greeters' who meet them at the plane door and 'Tarmac Transfers' so they never even enter the main terminal. It’s like being a 'Ghost Guide'—your job is to make sure they move from Point A to Point B without the world ever seeing them. Every second of their journey is 'Pre-cleared' so they never have to stop."
  },
  {
    "instruction": "Explain VIP 'Security Flag' data handling to a data engineer.",
    "input": "We need to ensure VIP PII (names/passports) isn't visible to all support staff.",
    "output": "You need 'Attribute-Based Access Control' (ABAC). Tag specific PNRs with a `VIP_STATUS` flag. For these records, the UI must 'Mask' the Name and Contact segments unless the agent has a high-security clearance. Additionally, you should implement an 'Audit Log' that records every single 'View' of a VIP PNR. If a celebrity's flight details leak, you need a 'Data Trail' to identify exactly which system or user accessed that specific record."
  },
  {
    "instruction": "Explain 'Privacy Protection' to a high-profile traveler.",
    "input": "A VIP traveler is concerned about 'Paparazzi' at the airport.",
    "output": "Your privacy is our 'Primary Mission.' We have arranged a 'Special Operations' team at the airport who will meet you directly on the tarmac. You’ll be whisked away in a private vehicle to a secure, 'members-only' suite where your customs and security checks will be handled in total peace. To the rest of the world, you’ll be a 'Digital Ghost'—you’ll be halfway to your hotel before anyone even knows you’ve landed!"
  },
  {
    "instruction": "Explain Crisis Management (Natural Disasters) to a junior agent.",
    "input": "A volcano has erupted in Iceland, and all flights across the Atlantic are grounded.",
    "output": "This is a 'Black Swan' event. Our priority is 'Passenger Accountability.' We use our 'Tracker' to find every single person in the 'Zone of Impact.' It’s like a 'Digital Roll Call'—we don't care about re-booking yet; we just need to confirm everyone is safe and has a roof over their head. Once the 'Roll Call' is done, we start the slow process of 'Evacuation' by priority."
  },
  {
    "instruction": "Explain 'Crisis Mapping' data aggregation to a data engineer.",
    "input": "The crisis team needs to see all travelers within 500 miles of an earthquake epicenter.",
    "output": "You need a 'Geospatial Query' on your active PNR database. Map the `Hotel_Address` or `Airport_Code` to Lat/Long coordinates. Use a 'Haversine Formula' to find all travelers within the radius. The challenge is 'Stale Data': if a traveler checked out of their hotel early, your map is wrong. You must cross-reference with 'DCS Flown' status to ensure your 'Crisis List' only includes people who are actually physically still in the zone."
  },
  {
    "instruction": "Calm a traveler during a natural disaster disruption.",
    "input": "A traveler is terrified because their flight was cancelled due to a nearby earthquake.",
    "output": "I am right here with you, and you are our #1 priority. Think of us as your 'Base Camp'—we are currently tracking your exact location and have already notified the local 'Safety Teams' that you’re there. Right now, the best thing to do is stay in your current safe spot. We are working on a 'Special Rescue' plan to get you to the nearest open airport the moment it’s safe. You are not alone!"
  },
  {
    "instruction": "Explain De-escalation Techniques to a junior agent.",
    "input": "An 'Angry Traveler' is shouting at the desk because their bag is lost.",
    "output": "Remember: 'They aren't mad at you; they are mad at the situation.' Use the 'L.A.S.T.' method: Listen, Apologize, Solve, and Thank. Let them 'vent' their frustration—it’s like a boiling pot that needs to let off steam. Once they’re calm, you become the 'Hero' who finds their bag. Your calm voice is the 'Anchor' that stops them from drifting into a rage."
  },
  {
    "instruction": "Explain 'Sentiment Analysis' for support logs to a data engineer.",
    "input": "We want to identify 'High-Risk' interactions in our chat logs automatically.",
    "output": "You should implement a 'Natural Language Processing' (NLP) model to score the 'Sentiment' of every customer message. Look for 'Volatility Keywords' (e.g., 'Useless', 'Legal', 'Shouting'). If the sentiment score drops below a certain threshold, the system should automatically 'Escalate' the chat to a senior 'Resolution Specialist' and flag the PNR with a 'Sensitive Handling' note to warn future agents of the traveler's current state."
  },
  {
    "instruction": "Calm an angry traveler whose bag was lost.",
    "input": "A traveler is red-faced and yelling that 'The airline stole my clothes!'",
    "output": "I hear how incredibly frustrating this is—I would be just as upset if my personal things were missing. Please know that I am now your 'Personal Detective.' I have already opened a 'Global Trace' on your bag, and I won't stop until we have its exact location. While we wait for the update, let's get you a 'Comfort Kit' and a credit for some immediate essentials. We’re going to find it for you!"
  },
  {
    "instruction": "Explain Disability Sensitivity to a junior agent.",
    "input": "An agent is booking for a traveler who is 'Person with Reduced Mobility' (PRM).",
    "output": "Words matter. We use 'Person-First' language (e.g., 'a person who uses a wheelchair' rather than 'wheelchair-bound'). Think of yourself as an 'Accessibility Scout'—you’re checking for 'Barriers' before the traveler gets there. Your job is to make the world's 'Infrastructure' feel invisible so they can focus on the 'Inspiration' of their trip. Every 'SSR' code you enter is a 'Digital Ramp' you’re building for them."
  },
  {
    "instruction": "Explain 'Accessibility' data fields in the hotel API to a data engineer.",
    "input": "Our hotel search doesn't show if a room has a 'Roll-in Shower'.",
    "output": "You need to map 'OTA Room Amenity Codes' specifically for accessibility. Codes like `80` (Wheelchair accessible) or `122` (Roll-in shower) are often buried in a sub-array. Your UI should have a 'verified' accessibility filter. If a hotel 'claims' to be accessible but doesn't provide these specific codes, you should flag it as 'Unverified' to prevent a traveler from arriving and finding a 'Physical Barrier' that ruins their stay."
  },
  {
    "instruction": "Reassure a traveler with accessibility needs.",
    "input": "A traveler is worried that the hotel 'won't really' be accessible when they arrive.",
    "output": "I completely understand that concern—you need to be sure! I have personally called the hotel's 'Front Office Manager' and had them walk into the specific room to verify the 'Roll-in Shower' and the doorway width. We’ve also added a 'Must-Comply' note to your booking. Think of me as your 'Advance Team'—I’ve cleared the path so you can just roll right in and start your vacation!"
  },
  {
    "instruction": "Explain Dietary Requirement Accuracy to a junior agent.",
    "input": "The agent forgot to confirm the 'Kosher' meal for a long-haul flight.",
    "output": "A 'Meal Error' isn't just a 'Wrong Order'—it can be a 'Religious or Health Crisis.' If a traveler has a severe allergy or a strict religious diet, missing that meal means they don't eat for 12 hours. It’s like forgetting to pack the fuel for the passenger! Always double-check the `HK` (Confirmed) status on `VGML` (Vegan) or `KSML` (Kosher) requests—never assume it’s 'automatic'."
  },
  {
    "instruction": "Explain 'Dietary Profile' sync to a data engineer.",
    "input": "The 'Meal Preference' in the user profile isn't showing up in the PNR.",
    "output": "This is a 'Mapping Failure.' Your profile system might use 'Vegan,' but the GDS needs the 4-letter code `VGML`. You need a 'Translation Layer' that maps user-friendly text to IATA-standard SSR codes. Furthermore, you should build a 'Confirmation Loop': 24 hours before flight, if the PNR status for the requested SSR is not `HK`, trigger a 'Priority Alert' to the fulfillment team to manually contact the airline."
  },
  {
    "instruction": "Explain a 'Meal Confirmation' to a traveler.",
    "input": "A traveler with a severe nut allergy is nervous about their flight meal.",
    "output": "Your safety is the 'Main Course' for us! We have sent a 'High-Priority Alert' to the airline's catering team with your specific requirements. I’ve already received their 'Digital Handshake' (a confirmation code) that your safe meal is being prepared in a dedicated kitchen. When you board, just give the flight attendant a quick 'hello' and mention your name—they’ll have your meal ready and waiting just for you!"
  },
  {
    "instruction": "Explain Medical Clearance (MEDIF) to a junior agent.",
    "input": "A traveler is flying two weeks after a major heart surgery.",
    "output": "This is a 'MEDIF' (Medical Information Form) situation. The airline's 'Medical Doctor' needs to sign off that the passenger is 'Fit to Fly.' Think of it as a 'Health Passport' for the sky. If you don't get this paperwork done, the captain can 'Refuse Carriage' at the gate. We’re not doctors, but we are the 'Administrators' who make sure the doctors have talked to each other before take-off."
  },
  {
    "instruction": "Explain 'Medical Document' encryption to a data engineer.",
    "input": "We need to store the 'MEDIF' forms for our travelers.",
    "output": "This is 'Highly Sensitive Health Data' (HIPAA/GDPR). You must never store these forms in the general 'Documents' folder. They must be placed in an 'Isolated Vault' with 'Single-Use Access Tokens'. After the flight is completed, the form should be automatically 'Purged' or 'Deep-Masked.' Your system should only track the 'Status' (e.g., `CLEARED_BY_AIRLINE`), never the 'Diagnosis' or the actual medical details."
  },
  {
    "instruction": "Explain the 'Fit to Fly' check to a traveler.",
    "input": "A traveler is annoyed that they have to get a 'doctor's note' just to go on vacation after a minor surgery.",
    "output": "I know it feels like an extra hurdle! But think of this as your 'Professional Sky-Clearance.' Because the air pressure at 30,000 feet is a bit different than on the ground, the airline just wants to make sure your body is 100% ready for the adventure. It’s their way of being your 'Sky Guard'—once your doctor gives the 'thumbs up,' you can fly with total peace of mind!"
  },
  {
    "instruction": "Explain Tipping Culture Nuances to a junior agent.",
    "input": "A traveler is going to Japan and wants to know how much to tip the hotel staff.",
    "output": "In Japan, 'Service is an Honor'—tipping can actually be seen as an insult. It’s like trying to 'pay' a friend for a gift they gave you. In the USA, it’s a 'Service Requirement' (20%). Your job is to be the 'Cultural Translator.' Advise the traveler on the 'Local Rule Book' so they don't accidentally cause 'Social Friction' during their stay."
  },
  {
    "instruction": "Explain 'Tipping Advice' data mapping to a data engineer.",
    "input": "We want to show 'Tipping Guidelines' for every city in our app.",
    "output": "You need a 'Destination Content' API (like Lonely Planet or a custom CMS). Your database needs a `Country_Metadata` table with fields for `Tipping_Expected (Boolean)` and `Average_Percent (Int)`. In your 'Itinerary View,' use a 'Contextual Trigger': if the traveler is in 'New York,' show a 'Dining Tip' pop-up. If they are in 'Tokyo,' show a 'Tipping Etiquette' note that explains why *not* to leave extra cash."
  },
  {
    "instruction": "Give tipping advice to a traveler going to France.",
    "input": "A traveler is confused about 'Service Compris' on a French restaurant bill.",
    "output": "French dining is a 'Classy Affair!' When you see 'Service Compris,' it means the service 'is already included' in the price of your meal. You don't need to add a big 20% tip like you would at home. If the service was truly wonderful, it's a lovely gesture to leave a 'Petit Geste'—just a few extra Euros on the table. Think of it as a 'small thank you' rather than a 'payment'!"
  },
  {
    "instruction": "Explain Pet-in-Cabin (PETC) Sensitivity to a junior agent.",
    "input": "A traveler is booking their small dog to fly with them in the cabin.",
    "output": "PETC is a 'Space Race.' Most planes only allow 2-3 pets in the cabin total. You must request the `PETC` SSR immediately and wait for the `KK` (Confirmed). It’s like booking a 'First-Come, First-Served' parking spot. Also, be the 'Comfort Consultant'—advise the traveler on 'Breathable Carriers' and 'Quiet Time' before the flight so the pet is a 'Happy Traveler' too."
  },
  {
    "instruction": "Explain 'Pet Inventory' validation to a data engineer.",
    "input": "Our booking tool allows unlimited 'Pet' requests, but the airline keeps rejecting them.",
    "output": "You are ignoring 'Inventory Caps.' Airlines don't just check 'Weight'; they check 'Total Count.' You need to query the `Cabin_Pet_Inventory` API before allowing the user to add a `PETC` SSR. If the count is `MAX`, your UI should show 'Pet Space Full' and offer a different flight. If the API doesn't exist, you must flag all 'Pet Bookings' for 'Manual Verification' before the user pays for their own seat."
  },
  {
    "instruction": "Explain 'Flying with a Pet' to a traveler.",
    "input": "A traveler is nervous about their cat 'crying' on the plane.",
    "output": "It’s totally normal to be a bit 'cat-anxious!' Think of your cat's carrier as their 'Private Sky-Cave.' If you put a t-shirt that smells like home inside, they’ll feel safe and snug. Most pets actually find the 'White Noise' of the plane very soothing and sleep the whole way. I’ve already secured your cat's 'Boarding Pass'—they’re all set for their first big 'purr-ney'!"
  },
  {
    "instruction": "Explain the Service Recovery Paradox to a junior agent.",
    "input": "The hotel messed up a guest's room, and the agent is worried the guest will never come back.",
    "output": "This is your 'Moment of Power!' The Paradox says that if you fix a mistake *amazingly* well, the guest will be *more* loyal than if nothing had gone wrong at all. It’s like 'Kintsugi'—the Japanese art of fixing broken pottery with gold. The crack is still there, but the piece is now more beautiful and valuable. Use a 'Recovery Gift' (like a free dinner) to 'Gold-Fill' the mistake!"
  },
  {
    "instruction": "Explain 'Recovery Attribution' in CRM to a data engineer.",
    "input": "We want to track if 'Service Recovery' actually helps retain customers.",
    "output": "You need to tag 'Post-Irrops' bookings. Create a `Recovery_Event` flag in the CRM. Link the 'Cost of Recovery' (vouchers/upgrades) to the `User_ID`. Then, perform a 'Retention Analysis': compare the 'Lifetime Value' (LTV) of users who experienced a 'Resolved Crisis' vs. those who had 'Perfect' trips. If the 'Resolved Crisis' group has a higher 'Return Rate,' you've proven the Service Recovery Paradox with data."
  },
  {
    "instruction": "Fix a hotel mistake for a traveler.",
    "input": "A traveler is calling because the hotel 'lost their reservation' and they are standing in the lobby.",
    "output": "I am so sorry for this 'hiccup'—I am taking charge of this right now! I’ve already called the manager on duty. Not only are they 'finding' your room, but I’ve also secured an 'Upgrade' to a Junior Suite and a complimentary breakfast for your whole stay as a 'thank you' for your patience. We’re turning this little 'rough start' into a 'VIP win' for you!"
  },
  {
    "instruction": "Explain Sustainable Travel Advisory to a junior agent.",
    "input": "A traveler wants to 'Reduce their Footprint' on a trip to Europe.",
    "output": "You are the 'Green Guide.' Suggest the 'Rail-over-Air' switch for short hops (like London to Paris). It’s not just about the planet; it’s about 'Slow Travel'—seeing the countryside instead of just the clouds. You’re helping them trade 'Speed' for 'Sustainability' and a much more 'Scenic' story to tell when they get home."
  },
  {
    "instruction": "Explain 'Carbon vs Time' trade-off logic to a data engineer.",
    "input": "We want to show 'Green Alternatives' in our flight search results.",
    "output": "You need to integrate a 'Modal Shift' engine. For any flight search < 500km, your API should trigger a parallel 'Rail Search.' Your UI should display the 'Carbon Savings' (e.g., -90% CO2) alongside the 'Time Difference' (e.g., +2 hours). This 'Comparative Data' allows the user to make an informed 'Value Choice' based on their personal 'Green vs. Fast' priorities."
  },
  {
    "instruction": "Explain 'Eco-Friendly' options to a traveler.",
    "input": "A traveler wants to know if taking the train is 'worth it'.",
    "output": "It’s more than worth it—it’s a 'Scenic Upgrade!' By taking the high-speed train, you’re not only reducing your carbon footprint by nearly 90%, but you’re also skipping the airport lines and arriving right in the heart of the city. Think of it as 'The Most Relaxing 2 Hours in Europe'—you get a great view, a comfortable seat, and the planet gets a nice big 'thank you' from you too!"
  },
  {
    "instruction": "Explain Bleisure (Business + Leisure) to a junior agent.",
    "input": "A business traveler wants to stay an extra 3 days after their meeting ends.",
    "output": "This is 'The Great Transition.' The first 3 days are 'Company Money,' and the last 3 days are 'Personal Money.' You have to 'Split the Bill' cleanly. Think of it as a 'Cost-Sharing' adventure—the company pays for the 'Arrival' and 'Departure' flights, and the traveler pays for the 'Extra' hotel nights. It’s the ultimate 'Perk' of modern business travel!"
  },
  {
    "instruction": "Explain 'Policy-Splitting' for Bleisure bookings to a data engineer.",
    "input": "Our system is struggling to categorize 'Mixed-Purpose' trips for tax reasons.",
    "output": "You need a 'Date-Range Attribution' logic. Your database should flag individual `Hotel_Nights` as either `CORP` or `SELF`. For the flights, the system must check the 'Corporate Travel Policy': if the 'Business Purpose' is > 50% of the trip duration, the flight remains a 'Business Expense.' If not, the system should trigger a 'Partial Reimbursement' calculation to ensure the company only pays for the 'Fair Share' of the airfare."
  },
  {
    "instruction": "Explain 'Extending a Trip' to a business traveler.",
    "input": "A traveler wants to know if they can bring their spouse for the weekend after their conference.",
    "output": "You absolutely should—it’s the 'Bleisure' way! Since your company is already covering your flights, you’re essentially getting a 'Free Ride' to a great weekend destination. We’ll keep your 'Official Business' receipt perfectly separate so your boss is happy, and then we’ll handle all the 'Fun Weekend' details for you and your spouse. It’s the perfect 'Mini-Vacation' built right into your work week!"
  },
  {
    "instruction": "Explain Lounge Access Rights to a junior agent.",
    "input": "A traveler is confused why they can't get into the lounge even though they have a 'Gold' card.",
    "output": "Lounge rules are a 'Logic Maze.' It depends on the 'Alliance' (e.g., Star Alliance), the 'Class of Service,' and the 'Operating Carrier.' Sometimes a 'Partner' flight doesn't count. You are the 'Gatekeeper Guide'—you check the 'Lounge Table' and tell them exactly which 'Secret Door' their card actually opens today. It’s like having a 'VIP Key' that only works on certain locks!"
  },
  {
    "instruction": "Explain 'Lounge Entitlement' API logic to a data engineer.",
    "input": "Our app is giving wrong advice about lounge access.",
    "output": "Your logic is too 'Binary.' Lounge access is a 'Multi-Factor' calculation: `(Carrier_Alliance) + (Loyalty_Tier) + (Cabin_Class) + (Itinerary_Type)`. For example, a 'Gold' member might get access on 'International' but not 'Domestic' routes. You need to ingest the 'Lounge Access Matrix' from the alliances and run a 'Simulated Check' against the PNR before telling the user 'Welcome In.' If in doubt, add a 'Check with Desk' disclaimer."
  },
  {
    "instruction": "Explain 'Lounge Perks' to a traveler.",
    "input": "A traveler asks if the lounge is 'really worth the walk' to the other terminal.",
    "output": "Oh, it’s a 'Sanctuary' in the storm! Think of the terminal as a busy street and the lounge as a 'Quiet Garden.' You’ll find free gourmet snacks, a peaceful place to charge your phone, and even a shower to freshen up. Since your 'Gold' status has earned you this 'VIP Pass,' I highly recommend taking that 5-minute walk—it’ll turn your wait into a 'Pre-flight Spa' experience!"
  },
  {
    "instruction": "Explain Baggage Claim Disputes to a junior agent.",
    "input": "A traveler says their bag was 'Scratched' and wants a full replacement.",
    "output": "You are the 'Expectation Manager.' Airlines don't cover 'Normal Wear and Tear' (like scratches or small dents)—it’s like a car getting a tiny stone chip on the highway. They only cover 'Functional Damage' (like a broken wheel or a giant hole). Be supportive but firm: 'The bag protected your clothes, which was its job!' Help them understand what is 'Claimable' vs. 'Character' for a well-traveled bag."
  },
  {
    "instruction": "Explain 'Baggage Damage' classification in the CRM to a data engineer.",
    "input": "We are seeing a high volume of 'Unsuccessful' baggage claims.",
    "output": "You need a 'Damage Categorization' drop-down in the support portal. Use categories: `COSMETIC` (Scratch/Scuff), `STRUCTURAL` (Wheel/Handle), `CRITICAL` (Puncture/Total Loss). If the agent selects `COSMETIC`, the system should automatically trigger a 'Policy Education' template for the user, explaining why the airline will likely reject the claim. This reduces 'False Hope' and prevents the user from wasting time on a doomed claim."
  },
  {
    "instruction": "Handle a 'Damaged Bag' complaint for a traveler.",
    "input": "A traveler is upset because their new suitcase has a 'scuff mark' after its first flight.",
    "output": "I know that 'first scuff' on a new bag hurts! Think of it as your bag's 'first badge of honor'—it’s proof of a successful adventure. While the airline's 'Safety Rules' usually only cover things that stop the bag from working (like a broken wheel), I’ve checked your 'Travel Insurance' and they might have a small 'New Gear' benefit for you. Let's see if we can get a 'Care Kit' to buff that out and keep your bag looking great!"
  },
  {
    "instruction": "Explain Jet Lag Mitigation Advice to a junior agent.",
    "input": "A traveler is flying from LA to London and is worried about being a 'Zombie' for their meeting.",
    "output": "You are the 'Sleep Scientist.' Advise them on the 'East-ward Shift'—stay awake until the local bedtime, no matter what! Tell them to 'Chase the Sun' during the day and use 'Deep Darkness' at night. It’s like 'Re-setting the Internal Clock'—if they can push through that first day, they’ll win the whole trip. Your advice is their 'Bio-Hack' for a successful journey!"
  },
  {
    "instruction": "Explain 'Timezone-Aware' itinerary push notifications to a data engineer.",
    "input": "We want our app to help travelers 'Beat Jet Lag'.",
    "output": "You need a 'Circadian Logic' engine. When a trip crosses > 3 timezones, trigger a 'Jet Lag Protocol'. 24 hours before flight, send a 'Hydration Reminder.' Upon landing, check the 'Local Time' at the destination. If it's 'Daylight,' send a notification: 'Go for a walk in the sun!' If it's 'Night,' send: 'Time to dim the lights.' Your app is acting as the traveler's 'External Biological Clock' to speed up their adaptation."
  },
  {
    "instruction": "Give jet lag advice to a traveler.",
    "input": "A traveler wants to know the 'secret' to landing and feeling great.",
    "output": "The secret is 'The Sun is Your Friend!' The moment you land, try to spend at least 30 minutes in natural sunlight—it tells your brain exactly what time it is. Also, drink twice as much water as you think you need. Think of it as 'Re-charging your internal battery'—a little sunshine and a lot of water will have you feeling like a 'Local' in no time!"
  },
  {
    "instruction": "Explain Safety Briefing Compliance to a junior agent.",
    "input": "A traveler was 'Offloaded' because they refused to turn off their laptop during take-off.",
    "output": "Safety is a 'Zero-Tolerance' zone. When the crew says 'Laptops Off,' it’s not a 'Suggestion'—it’s a 'Legal Command.' If a traveler argues, the captain sees it as a 'Security Threat.' Our job is to be the 'Safety Advocate' before they fly—explain that 'The Crew is the Law' in the sky. It’s about being a 'Team Player' for the safety of the whole plane."
  },
  {
    "instruction": "Explain 'Safety Incident' tagging in the CRM to a data engineer.",
    "input": "We need to identify 'Difficult Passengers' for future bookings.",
    "output": "This is 'Risk Profiling.' If an airline sends a 'Passenger Incident Report,' you must tag the `User_Profile` with a `SAFETY_COMPLIANCE_ISSUE` flag. Your 'Self-Service' booking tool should trigger a 'Mandatory Acknowledgment' of the airline's 'Code of Conduct' for any future bookings by this user. It’s about protecting the traveler from themselves and ensuring our agency isn't seen as a 'Risk' by the airlines."
  },
  {
    "instruction": "Explain why 'The Rules Matter' to a stubborn traveler.",
    "input": "A traveler is complaining that the flight attendants were 'rude' for making them stow their bag.",
    "output": "I know it can feel like a lot of small rules! But think of the crew as the 'Guardians of the Sky.' Their #1 job is to make sure that in an emergency, there’s nothing blocking your path to safety. When they ask you to stow your bag, they’re actually 'Clearing the Lane' for you. It’s their way of keeping you and everyone else 100% safe—they’re looking out for you, every step of the way!"
  },
  {
    "instruction": "Explain Bereavement Fares to a junior agent.",
    "input": "A traveler needs to fly tonight because a family member has passed away.",
    "output": "This is a 'Mission of Mercy.' Some airlines offer 'Bereavement Fares'—they aren't always 'Cheaper,' but they are 'Ultra-Flexible.' You can change or cancel them without the usual fees. You’ll need a 'Death Certificate' or the 'Funeral Home' details. You aren't just an agent today; you’re a 'Compassionate Guide' helping someone get where they need to be during their hardest hour."
  },
  {
    "instruction": "Explain 'Bereavement Waiver' data fields to a data engineer.",
    "input": "The 'Refund Fee' was automatically charged on a bereavement ticket.",
    "output": "Your 'Auto-Refund' logic is too rigid. You need a 'Waiver Code' field in the transaction table. If the agent enters a `BEREAVEMENT_WAIVER` code (authenticated by the airline's `OSI` note), the system must override the 'Cat 16' penalty math and set `CancellationFee = 0`. You must also ensure the 'Required Docs' (PDF) are linked to this record for IATA audit purposes, or the airline will send a debit memo later."
  },
  {
    "instruction": "Express sympathy and help a traveler with an emergency trip.",
    "input": "A traveler is crying on the phone because they need to fly home for a funeral.",
    "output": "I am so deeply sorry for your loss. Please take a deep breath—I am taking over all the logistics for you right now. I’ve found a 'Flexible Mercy' fare that lets you change your return date whenever you’re ready, with no fees. I’ll handle all the 'paperwork' with the airline so you can focus on being with your family. We’re going to get you home on the very next flight."
  },
  {
    "instruction": "Explain Force Majeure to a junior agent.",
    "input": "A traveler is demanding a refund because their flight was cancelled due to a 'General Strike' in France.",
    "output": "Force Majeure is an 'Act of God' (or a major human event like a strike) that is 'Outside the Airline's Control.' In these cases, the airline doesn't owe 'Delay Compensation' (cash), but they *do* still owe a 'New Flight' or a 'Refund.' It’s like a 'Game Reset'—no one is at fault, so we just try to get everyone back to 'Start' as fairly as possible."
  },
  {
    "instruction": "Explain 'Force Majeure' trigger logic to a data engineer.",
    "input": "Our 'Auto-Compensation' tool is flagging strikes as 'Payable' events.",
    "output": "You need an 'Exemption Filter.' Integrate a 'Global Incident Feed' (like GDACS). If a flight is cancelled and there is a 'Verified Force Majeure' event (Strike, Volcanic Ash, War) at the origin or destination on that date, the system must toggle the `EC261_ELIGIBLE` flag to `FALSE`. This prevents our system from making 'False Promises' to users about cash compensation that the law doesn't support."
  },
  {
    "instruction": "Explain why a 'Strike' means no extra cash to a traveler.",
    "input": "A traveler is upset that the airline won't pay them $600 for a delay caused by a strike.",
    "output": "I hear your frustration—it’s so tough when your plans are disrupted! Because a strike is considered 'Outside the Airline's Control' (what we call 'Force Majeure'), the law doesn't require them to pay an extra 'penalty' fee. Think of it as a 'Global Weather' event—the airline is just as stuck as you are! However, they *are* still 100% responsible for getting you a new flight or your money back, and I’m making sure they do that right now!"
  },
  {
    "instruction": "Explain Language Barriers in Irrops to a junior agent.",
    "input": "A traveler is stuck in a small airport in China where no one speaks English.",
    "output": "You are their 'Digital Translator.' Use 'Visual PNRs'—send them a screenshot of their new flight with the 'Gate' and 'Time' clearly circled. Tell them to 'Show the Screen' to any official. It’s like a 'Universal Passport'—a flight number is a flight number in any language. Your clear, visual data is the 'Bridge' that gets them through the communication gap."
  },
  {
    "instruction": "Explain 'Multi-lingual Itinerary' generation to a data engineer.",
    "input": "Our travelers in foreign countries can't communicate with local gate agents.",
    "output": "You need 'Local-Language PDF' generation. Your itinerary service should detect the `Destination_Country_Language`. For travelers, provide a 'Help Card' in the local language (e.g., Mandarin) that says: 'I am a passenger on flight XXX. Please help me find my gate.' This 'Contextual Translation' should be pushed to the mobile app's 'Offline Mode' so it works even if the traveler has no data."
  },
  {
    "instruction": "Help a traveler who can't speak the local language.",
    "input": "A traveler is panicked because they can't understand the 'Cancellation' announcements in a foreign airport.",
    "output": "Don't worry—I am your 'Voice' on the ground! I’ve just sent a special 'Traveler Card' to your phone. It has all your new flight details written in the local language. Just show that screen to any staff member, and they’ll know exactly where to point you. I’m also calling the airline's local 'English Desk' to make sure they’re looking out for you at the gate. You’ve got this!"
  },
  {
    "instruction": "Explain Over-Tourism Sensitivity to a junior agent.",
    "input": "A traveler wants to go to Venice in July, but the city is 'Closing its Doors'.",
    "output": "You are the 'Conscious Consultant.' Venice is 'Full.' Advise them on the 'Shoulder Season' (May or September) or a 'Hidden Gem' nearby like Treviso. It’s about 'Better Travel, Not More Travel.' You’re helping them have a 'VIP Experience' without the 'Crowd Stress' and helping the city breathe at the same time. It’s a win for the traveler and the world!"
  },
  {
    "instruction": "Explain 'Saturation Alerts' in search algorithms to a data engineer.",
    "input": "We want to encourage 'Sustainable Tourism' by diverting traffic from crowded spots.",
    "output": "You need a 'Destination Saturity' index. Integrate data from 'Overtourism' trackers or local government 'Visitor Caps.' If a destination has a 'High Saturation' score for the selected dates, your search results should include a 'Pro-Tip' sidebar: 'Venice is very crowded right now. Have you considered Treviso? (20 mins away, -40% price)'. It’s 'Nudge Theory' in travel data to balance the global load."
  },
  {
    "instruction": "Suggest a 'Hidden Gem' to a traveler.",
    "input": "A traveler is discouraged because they heard their dream destination is 'too crowded to enjoy'.",
    "output": "I have an 'Inside Secret' for you! While the main spot is a bit busy right now, there is a 'Hidden Sister' city just 20 minutes away that has all the same magic but none of the crowds. Think of it as 'The Private Path'—you’ll get those perfect photos, the best local food without a wait, and a much more 'authentic' feeling. It’s like discovering the 'original' version before everyone else does!"
  },
  {
    "instruction": "Explain Travel Insurance Claims to a junior agent.",
    "input": "A traveler's flight was cancelled, and they want to know how to get their 'Hotel Money' back.",
    "output": "You are the 'Documentation Specialist.' The insurance company needs 'The Paper Trail.' You must provide an 'Insurance Letter' (proof the flight was cancelled) and tell the traveler to keep every single 'Physical Receipt' for food and taxis. It’s like a 'Legal Case'—without the evidence (the receipts), the 'Judge' (the insurance company) won't pay. Your job is to make sure the traveler has a 'Perfect File'!"
  },
  {
    "instruction": "Explain 'Automated Claim Letter' generation to a data engineer.",
    "input": "Our support team spends too much time writing 'Proof of Cancellation' letters.",
    "output": "You need an 'Insurance Document Generator.' Build a service that pulls the `Flight_ID`, `Original_Status`, and `Final_Status` from the OAG feed. If the flight was `CNCL` (Cancelled), generate a PDF with the airline's 'Official Reason' and a 'Unique Verification Code'. This should be automatically 'Pushed' to the traveler's app the moment the flight is cancelled, allowing them to start their insurance claim with one click."
  },
  {
    "instruction": "Help a traveler with an insurance claim.",
    "input": "A traveler is overwhelmed by the 'forms' they need to fill out for a $200 hotel claim.",
    "output": "I’ll make this 'Easy Mode' for you! I’ve already generated your 'Official Proof of Cancellation' and attached it to an email for you. All you need to do is take a quick photo of your hotel receipt and hit 'Send.' Think of me as your 'Claims Assistant'—I’ve done the heavy lifting so you can just get your money back and focus on your next trip!"
  },
  {
    "instruction": "Explain Hotel Loyalty Recognition to a junior agent.",
    "input": "A 'Diamond' member is checking in, and the hotel says 'No Upgrades Available'.",
    "output": "This is where 'Advocacy' happens. Call the hotel and ask for the 'Rooms Controller.' Remind them of the traveler's 'Loyalty Story'—they’ve stayed with the brand 50 times this year! Often, a room is 'held' for a VIP who hasn't arrived. Your job is to 'Nudge' that room toward your traveler. It’s about 'Loyalty as a Currency'—make sure the hotel 'pays' the traveler back for their trust!"
  },
  {
    "instruction": "Explain 'Loyalty Tier' prioritization in the booking API to a data engineer.",
    "input": "Our hotel search isn't highlighting 'Member Benefits' correctly.",
    "output": "You need to integrate the 'Loyalty API' from the hotel chains (e.g., Marriott Bonvoy). When a user is logged in, your API must query their 'Current Tier'. In the search results, you should dynamically 'Inject' a 'Member Perk' badge: 'Diamond Member: Free Breakfast & Priority Upgrade'. This requires 'Personalized Ranking'—boost hotels where the user has the highest tier to maximize their 'Recognition' value."
  },
  {
    "instruction": "Help a loyal traveler get their 'due' at a hotel.",
    "input": "A high-tier traveler is disappointed that they were given a 'standard' room.",
    "output": "I’m on it! You’ve been so loyal to this brand, and you deserve that 'Diamond' treatment. I’ve just called the manager on duty and reminded them of your 'VIP Status.' They’ve found a beautiful 'Corner Suite' that just opened up and they’re moving you there right now! It’s our way of saying 'thank you' for being such a legendary traveler—enjoy the view!"
  },
  {
    "instruction": "Explain Ground Transport Safety to a junior agent.",
    "input": "A traveler is arriving in a 'High-Risk' city at 2 AM and wants to take a 'local cab'.",
    "output": "You are the 'Safety Scout.' Advise against 'Bandit Cabs'—unmarked cars that aren't tracked. Suggest an 'Authorized Rideshare' or a 'Pre-booked Private Transfer.' It’s like choosing between a 'mystery door' and a 'locked gate.' The pre-booked car has a 'Name' and a 'Plate' that we can track. Your advice is the 'Final Shield' for their journey."
  },
  {
    "instruction": "Explain 'Transfer Tracking' integration to a data engineer.",
    "input": "We want to track if a traveler safely made it from the airport to their hotel.",
    "output": "You need to integrate with 'Ground Transport' APIs (like Uber for Business or Welcome Pickups). Your system should listen for the 'Drop-off Event.' If a traveler is not 'Checked-In' to their hotel within 60 minutes of the 'Drop-off,' trigger a 'Safety Pulse'—a simple app notification asking 'Are you okay?'. This 'Closed-Loop Safety' ensures that we aren't just tracking flights, but the 'Whole Human' until they reach their door."
  },
  {
    "instruction": "Give safety advice for ground transport to a traveler.",
    "input": "A traveler is nervous about getting from the airport to their hotel in a new country.",
    "output": "I totally understand—arriving in a new place can be a bit overwhelming! I’ve pre-booked a 'Trusted Transfer' for you. A professional driver will be waiting right at the 'Arrivals' gate with a sign with your name on it. It’s like having a 'Digital Bodyguard' waiting for you—they’ll take you straight to the lobby, and I’ll be tracking your journey the whole way to make sure you arrive safe and sound!"
  },
  {
    "instruction": "Explain Solo Female Traveler Safety to a junior agent.",
    "input": "A woman is traveling alone to a new city and wants to feel 'Secure' in her hotel.",
    "output": "You are the 'Security Architect.' Suggest 'Female-Friendly' hotels with 24/7 security and ask for a 'Room on a Higher Floor' (away from street access). Also, advise her to 'Never Say the Room Number Out Loud' at the desk. It’s about building a 'Circle of Privacy' around her. Your thoughtfulness is her 'Peace of Mind' for the whole trip."
  },
  {
    "instruction": "Explain 'Safety Attribute' filtering in the hotel engine to a data engineer.",
    "input": "We want to help solo travelers find 'Verified Secure' hotels.",
    "output": "You need to create a 'Safety Tag' based on user reviews and property attributes. Look for keywords like 'Safe neighborhood', '24h Security', and 'Well-lit'. Your search engine should allow a 'Solo Traveler' filter that boosts hotels with these verified 'Security Signals.' You can also implement a 'Room Preference' field in the booking API that automatically sends a 'High Floor / Near Elevator' request to the hotel's PMS for these travelers."
  },
  {
    "instruction": "Give safety tips to a solo female traveler.",
    "input": "A solo traveler is nervous about her first trip to a big city alone.",
    "output": "You are going to have an incredible time! To keep things completely stress-free, I’ve hand-picked a hotel with a '24/7 Concierge' and requested a beautiful room on a higher floor right near the elevator. Think of it as your 'Private Urban Oasis'—it’s super secure and very easy to find. I’m also sending you a 'Local Safety Map' with all the best, well-lit walking routes. You’re all set for a safe, empowered adventure!"
  },
  {
    "instruction": "Explain Digital Detox Travel to a junior agent.",
    "input": "A traveler is going to a remote lodge in the Amazon with 'Zero Connectivity'.",
    "output": "You are the 'Expectation Manager.' You must warn them that 'Off-Grid means Off-Grid.' No Wi-Fi, no signal, no 'Emergency Texts.' Advise them to 'Print Everything'—it’s like going back to the 'Analog Age.' Your job is to make sure they are 'Mentally Ready' to be disconnected so they can truly 'Reconnect' with nature. It’s a 'Digital Vacation' for their brain!"
  },
  {
    "instruction": "Explain 'Offline Mode' data persistence to a data engineer.",
    "input": "Our app is 'Useless' for travelers in remote areas with no signal.",
    "output": "You need a 'Robust Offline Sync'. Your app must detect when a traveler is heading to a 'Dead Zone' (based on destination metadata) and trigger a 'Full Cache' of their itinerary, maps, and 'Emergency Contacts'. Use 'Local Storage' (like SQLite) to ensure that even without a single bar of signal, the traveler can still see their 'Last Mile' instructions. The data must be 'Immutable' once offline to prevent sync conflicts when they return to a 5G zone."
  },
  {
    "instruction": "Prepare a traveler for a 'Digital Detox' trip.",
    "input": "A traveler is worried about being 'out of touch' with their family during a remote jungle stay.",
    "output": "This is going to be such a beautiful 'Brain Reset!' Because you’re heading to a true 'Hidden Sanctuary,' your phone is going to take a well-deserved nap. Think of it as a 'Digital Spa Day'—I’ve already provided your family with the lodge’s 'Satellite Emergency Number' just in case, so you can completely let go and relax. I’ve also printed your whole 'Adventure Guide' for you, so you have everything you need right in your hands. Enjoy the silence!"
  },
  {
    "instruction": "Explain The \"Last Mile\" Problem to a junior agent.",
    "input": "The traveler landed safely, but they can't figure out how to get to the 'Hidden' hotel entrance.",
    "output": "The 'Last Mile' is the most dangerous part of the trip for 'Customer Satisfaction.' The plane was fast, but the 'walk to the lobby' is where they get lost. You are the 'Final Guide.' Provide 'Visual Cues'—'Turn left at the blue fountain, go past the red door.' It’s about 'The Hand-off'—making sure the traveler is safely in the hotel’s arms before you 'close' the file. The trip isn't over until they are in the lobby!"
  },
  {
    "instruction": "Explain 'Last Mile' visual instructions in the mobile app to a data engineer.",
    "input": "Travelers are getting lost in the final 500 meters of their journey.",
    "output": "Standard 'Google Maps' pins are too imprecise for large resorts or hidden entrances. You need to implement 'Micro-Navigation' using 'Wayfinding Photos'. Your CMS should allow hotels to upload 'Last Mile Photos' (e.g., 'The Hidden Gate'). In the app, when the traveler’s GPS is within 500m of the destination, trigger a 'Visual Guide' carousel. This 'Ground-Truth' data bridges the gap between the 'Digital Pin' and the 'Physical Door'."
  },
  {
    "instruction": "Help a traveler find their 'Hidden' hotel.",
    "input": "A traveler is standing on a street corner in a foreign city and 'Can't see the hotel'.",
    "output": "I am looking at the 'Street View' right now with you! Don't worry—you are just 50 steps away. Look for the small 'Green Door' right next to the antique shop. I’ve just called the front desk, and a porter is walking out to meet you on the sidewalk right now! Think of it as a 'Secret Entrance' to your private getaway. You’re almost home!"
  }
]

# 2. Clean and Save to JSONL
with open("train.jsonl", "w") as f:
    for item in raw_data:
        # Strip invisible whitespace that breaks tokenizers
        item["instruction"] = " ".join(item.get("instruction", "").split())
        item["output"] = " ".join(item.get("output", "").split())
        f.write(json.dumps(item) + "\n")

# 3. Load using Hugging Face Datasets
dataset = load_dataset("json", data_files="train.jsonl")
train_ds = dataset["train"]

# 4. Format into the exact prompt the model expects
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    return {"text": prompt}

train_ds = train_ds.map(format_example)
print("Formatted example:\n", train_ds[0]["text"])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Formatted example:
 ### Instruction:
Explain PNR Structure to a junior agent.

### Response:
Think of a PNR as the 'Identity Document' of a trip. To make it official in the airline's eyes, you just need to remember the PRINT mnemonic: P for Phone, R for Received from, I for Itinerary, N for Name, and T for Ticketing. Without these five pillars, the central nervous system of the GDS won't allow the booking to live. It's like trying to mail a letter without an address; the system simply won't know where to send the data!


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch

model_name = "google/gemma-2-2b"

# 1. Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. Load the Brain (FP16 Precision)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16, 
)

# 3. Apply the LoRA Adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

2026-02-25 22:51:26.371778: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772059886.589507      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772059886.653618      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772059887.168623      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772059887.168670      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772059887.168673      55 computation_placer.cc:177] computation placer alr

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

trainable params: 20,766,720 || all params: 2,635,108,608 || trainable%: 0.7881


In [6]:
# 1. Convert Text to Numbers
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=False,
        truncation=True,
        max_length=1024,
    )

tokenized_ds = train_ds.map(
    tokenize,
    batched=True,
    remove_columns=train_ds.column_names,
)

# 2. Set the "Labels" (The answer key for the AI)
tokenized_ds = tokenized_ds.map(
    lambda x: {"labels": x["input_ids"]},
    batched=True
)

print("Data is tokenized and ready.")

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Data is tokenized and ready.


In [7]:
from transformers import TrainingArguments, Trainer

# 1. The Control Panel
training_args = TrainingArguments(
    output_dir="./finetune_out",
    per_device_train_batch_size=1,      # Keep this at 1 to prevent Memory Crashes
    gradient_accumulation_steps=4,      # Simulates a batch size of 4
    warmup_steps=10,
    max_steps=100,                      # 100 steps for a live demo (Use 500+ for real projects)
    learning_rate=2e-4,                 # The speed of learning
    fp16=True,                          # Use Fast Math
    optim="adamw_torch_fused",          # Efficient optimizer
    logging_steps=10,
    report_to="none",
)

# 2. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    processing_class=tokenizer,
)

# 3. Ignite the Engine
print("Starting training...")
trainer.train()

max_steps is given, it will override any value given in num_train_epochs


Starting training...


Step,Training Loss
10,2.976700
20,2.213100
30,2.015300
40,1.971300
50,1.916600
60,1.841400
70,1.862900
80,1.915700
90,1.930900
100,1.913700


TrainOutput(global_step=100, training_loss=2.0557659530639647, metrics={'train_runtime': 108.1725, 'train_samples_per_second': 3.698, 'train_steps_per_second': 0.924, 'total_flos': 569223159252480.0, 'train_loss': 2.0557659530639647, 'epoch': 0.8888888888888888})

In [8]:
import time

# 1. Save the adapters to Kaggle's local disk
model.save_pretrained("./my_custom_model")
tokenizer.save_pretrained("./my_custom_model")
print("Model saved!\n")

# 2. Testing the Complex Travel Scenario
# We use a multi-part instruction to trigger the different "Expertise" areas the model learned.
scenario_instruction = """Explain the situation below to a Junior Agent and a Data Engineer.

Context: A traveler with a motorized wheelchair (WCBD - dry cell) is flying from the US to Japan. The airline’s DCS is rejecting the PNR because the SSR code was entered as WCHR instead of the specific battery-type codes required for the cargo hold.

Please cover:
1. The Technical mismatch between WCHR and WCBD.
2. The Regulatory aspect citing DOT Part 382.
3. The Data challenge of why the "Profile-to-PNR" sync failed to map these attributes."""

# Format exactly like your training data (Instruction/Response format)
prompt = f"### Instruction:\n{scenario_instruction}\n\n### Response:\n"

# Tokenize and push to GPU
inp = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate - We add repetition_penalty and eos_token_id to stop the "looping" behavior
print("Thinking...")
out = model.generate(
    **inp, 
    max_new_tokens=512,        # 512 is the "sweet spot" for detailed travel explanations
    temperature=0.7,           # Adds a bit of creativity while staying professional
    do_sample=True,
    repetition_penalty=1.15,   # Higher penalty to stop the model from repeating "### Response:"
    eos_token_id=tokenizer.eos_token_id, # Tells the model to stop at the end of its answer
    pad_token_id=tokenizer.pad_token_id
)

# Decode and print
print("\n--- TRAVEL EXPERT OUTPUT ---")
# We split by "### Response:\n" and take the last part to avoid printing the prompt again
decoded_output = tokenizer.decode(out[0], skip_special_tokens=True)
response_only = decoded_output.split("### Response:\n")[-1].strip()
print(response_only)

Model saved!

Thinking...

--- TRAVEL EXPERT OUTPUT ---
You need a 'Data Mapping' bridge that converts `WCHR` into IATA compliant tags like `E09D`. Without this, your system will be flagging every ticket you create as potentially 'Illegal,' even if it includes all the relevant information in other fields. It’s not just about the number on paper—it’s ensuring we are feeding an engine designed to read human readable data rather than literal symbols or strings.


In [10]:
with open('./my_custom_model/adapter_config.json', 'r') as f:
    print(f.read())

{
  "alpha_pattern": {},
  "auto_mapping": null,
  "base_model_name_or_path": "google/gemma-2-2b",
  "bias": "none",
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
  "r": 16,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "down_proj",
    "v_proj",
    "up_proj",
    "k_proj",
    "q_proj",
    "gate_proj",
    "o_proj"
  ],
  "task_type": "CAUSAL_LM",
  "use_dora": false,
  "use_rslora": false
}


In [9]:
import kagglehub

# 1. Path to your adapter folder
local_path = "./my_custom_model"

# 2. Upload/Register the model 
# Handle is 1st, Local Path is 2nd. No keywords needed!
kagglehub.model_upload(
    "kush5618/gemma-2b-travel-expert/transformers/v1",
    local_path
)

Uploading Model https://api.kaggle.com/models/kush5618/gemma-2b-travel-expert/transformers/v1 ...
Model 'gemma-2b-travel-expert' does not exist or access is forbidden for user 'kush5618'. Creating or handling Model...
Model 'gemma-2b-travel-expert' Created.
Starting upload for file ./my_custom_model/README.md


Uploading: 100%|██████████| 5.09k/5.09k [00:00<00:00, 13.5kB/s]

Upload successful: ./my_custom_model/README.md (5KB)
Starting upload for file ./my_custom_model/adapter_config.json



Uploading: 100%|██████████| 720/720 [00:00<00:00, 2.08kB/s]

Upload successful: ./my_custom_model/adapter_config.json (720B)
Starting upload for file ./my_custom_model/tokenizer.json


...


Uploading: 100%|██████████| 83.1M/83.1M [00:01<00:00, 76.1MB/s]

Upload successful: ./my_custom_model/adapter_model.safetensors (79MB)
Starting upload for file ./my_custom_model/special_tokens_map.json



Uploading: 100%|██████████| 522/522 [00:00<00:00, 1.44kB/s]

Upload successful: ./my_custom_model/special_tokens_map.json (522B)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/kush5618/gemma-2b-travel-expert/transformers/v1


In [10]:
# Load the base model and adapters, then merge
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./merged_model_for_ollama")

In [11]:
# Clone the conversion tool
!git clone https://github.com/ggerganov/llama.cpp
# Install the necessary python dependencies
!pip install -r llama.cpp/requirements.txt

Cloning into 'llama.cpp'...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


remote: Enumerating objects: 81065, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 81065 (delta 35), reused 29 (delta 29), pack-reused 81010 (from 2)
Receiving objects: 100% (81065/81065), 305.09 MiB | 42.63 MiB/s, done.
Resolving deltas: 100% (58603/58603), done.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 109.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.3 MB/s eta 0:00:00:00:0100:01

In [17]:
from huggingface_hub import hf_hub_download
import shutil
import os

# Create the directory if it somehow doesn't exist
os.makedirs("./merged_model_for_ollama", exist_ok=True)

try:
    # This fetches the gated file using your HF login/token
    repo_id = "google/gemma-2-2b"
    print(f"Attempting to download tokenizer.model from {repo_id}...")
    
    local_file = hf_hub_download(
        repo_id=repo_id,
        filename="tokenizer.model",
        token=True # This automatically uses the token you used to load the model
    )
    
    # Copy it into your merged model folder
    shutil.copy(local_file, "./merged_model_for_ollama/tokenizer.model")
    print("Successfully moved tokenizer.model to ./merged_model_for_ollama/")
    
except Exception as e:
    print(f"Error downloading: {e}")
    print("If you get a 401 here, make sure you have a Hugging Face token added to Kaggle Secrets as 'HF_TOKEN'.")

Attempting to download tokenizer.model from google/gemma-2-2b...
Successfully moved tokenizer.model to ./merged_model_for_ollama/


In [18]:
# This takes about 2-5 minutes depending on model size
!python llama.cpp/convert_hf_to_gguf.py ./merged_model_for_ollama \
    --outfile travel_expert_fp16.gguf \
    --outtype f16

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO:hf-to-gguf:Loading model: merged_model_for_ollama
INFO:hf-to-gguf:Model architecture: Gemma2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.float16 --> F16, shape = {2304, 256000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.float16 --> F32, shape = {2304}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.float16 --> F16, shape = {9216, 2304}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.float16 --> F16, shape = {2304, 9216}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.float16 --> F16, shape = {2304, 9216}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.float16 --> F32, shape

In [21]:
# We use the new target name 'llama-quantize'
!cd llama.cpp/build && cmake --build . --config Release --target llama-quantize

# The tool is now named 'llama-quantize' and sits in the bin folder
!./llama.cpp/build/bin/llama-quantize travel_expert_fp16.gguf travel_expert_q4.gguf q4_k_m

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[  0%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  0%] Linking CXX static library libcpp-httplib.a
[  0%] Built target cpp-httplib
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  0%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  2%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  2%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-threading.cpp.o
[  2%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-quants.c.o
[  4%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
[  4%] Linking CXX shared library ../../bin/libggml-base.so
[  4%] Built target ggml-base
[  4%] Building C object ggml/src/CMakeFiles/ggml-cpu.dir/ggml-cpu/ggml-cpu.c.o
[  4%] Building CXX object ggml/src/CMakeFiles/ggml-cpu.dir/g

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


llama_model_loader: loaded meta data with 29 key-value pairs and 288 tensors from travel_expert_fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Gemma 2 2b
llama_model_loader: - kv   3:                       general.organization str              = Google
llama_model_loader: - kv   4:                           general.basename str              = gemma-2
llama_model_loader: - kv   5:                         general.size_label str              = 2B
llama_model_loader: - kv   6:                      gemma2.context_length u32              = 8192
llama_model_loader: - kv   7:                    gemma2.embedding_length u32  

In [22]:
import os
from IPython.display import FileLink

# Check if the file exists in the current directory
if os.path.exists("travel_expert_q4.gguf"):
    print("Found it! Click the link below to download your model:")
    display(FileLink("travel_expert_q4.gguf"))
else:
    # If it's not in the root, let's search for it
    print("Searching for the file...")
    !find /kaggle/working -name "travel_expert_q4.gguf"

Found it! Click the link below to download your model:


/kaggle/working/travel_expert_q4.gguf

In [23]:
# Remove the large unquantized file
!rm travel_expert_fp16.gguf
print("Deleted the large FP16 file. Your 1.6GB quantized model is now the star of the show!")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Deleted the large FP16 file. Your 1.6GB quantized model is now the star of the show!
